# Amazon ML Challenge 2026 — Business Entity Resolution
## FINAL_CLEAN.ipynb — production submission notebook

**What this notebook does, in order:**
1. Discovers train/test files under `/kaggle/input` automatically (no hard-coded dataset slug).
2. Reuses the verified checkpoint pipeline (blocking/indexing/features), embedded below so
   it doesn't depend on a specific Kaggle dataset attach path.
3. Trains a LightGBM matcher on a labeled sample of training candidate pairs
   (S1-entity-level train/validation split — leakage-safe).
4. Evaluates S1-level **Macro F0.5** on held-out validation, at both the baseline
   threshold (0.944) and an optional lightweight threshold search.
5. Runs a **memory-safe, chunked** test pipeline (the previous run OOM'd by accumulating
   ~58M candidate objects in RAM — this version never holds more than one S1 chunk's
   candidates in memory at a time, and streams output to disk incrementally).
6. Writes `matching_results.tsv` and `candidate_pairs.tsv`, validates them, and
   (best-effort) packages the final submission zip.

**Configure `FAST_MODE` / sample sizes / chunk sizes in the CONFIG cell below.**
The default configuration prioritizes getting a **valid first submission** quickly,
per the competition's own advice (16 attempts remaining — don't burn the first hour
chasing a maximal score before you have any submission at all).


In [ ]:
# ============================================================
# CONFIG — edit these, then Run All
# ============================================================
import os

# ---- Mode ----
# FAST_MODE prioritizes a valid first submission over maximal recall/score.
# It does NOT fake a runtime guarantee -- it just picks conservative sample
# and chunk sizes. Flip to False once you have a valid submission in hand
# and want to spend later attempts on a fuller run.
FAST_MODE = True

# ---- Training sample sizes (S1-entity-level) ----
# These mirror the sample sizes that produced the previously-verified
# baseline (Macro F0.5 ~0.81 @ threshold 0.944, ROC-AUC ~0.999). Raising
# them may help but costs time; this is the first knob to turn later.
TRAIN_S1_SAMPLE = 5000 if FAST_MODE else 15000
VAL_S1_SAMPLE = 2000 if FAST_MODE else 5000
RANDOM_SEED = 42

# ---- Baseline vs optimized threshold ----
BASELINE_THRESHOLD = 0.944          # INITIAL BASELINE THRESHOLD (verified prior baseline)
RUN_THRESHOLD_SEARCH = True         # lightweight sweep on the val set to look for a better one
THRESHOLD_SEARCH_GRID = [0.80, 0.85, 0.88, 0.90, 0.92, 0.94, 0.944, 0.95, 0.96, 0.97, 0.98]

# ---- Blocking caps ----
# The checkpoint's modules/blocking.py already ships with these as its
# DEFAULT_FREQ_CAPS (the confirmed-win loosened caps: AL2 80->160,
# A4W 10->20, AT2 20->40). Left as None here means "use the module
# default" -- only set this dict if you deliberately want to override it.
FREQ_CAPS_OVERRIDE = None

# ---- Test pipeline chunking (THE critical memory-safety knob) ----
# S1 rows are processed in chunks of this size. Candidates/features/scores
# for a chunk are written to disk and discarded before the next chunk
# starts -- never accumulated across the whole test set in RAM. Lower this
# if you see memory pressure warnings during the run.
S1_CHUNK_SIZE = 3000
MEMORY_SAFETY_FREE_GB = 1.5   # if free RAM drops below this, shrink behavior defensively

# ---- Paths ----
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/home/claude/work/kaggle_sim"
os.makedirs(WORK_DIR, exist_ok=True)
INDEX_TRAIN_DIR = os.path.join(WORK_DIR, "index_train")
INDEX_TEST_DIR = os.path.join(WORK_DIR, "index_test")
SHARDS_DIR = os.path.join(WORK_DIR, "test_shards")
OUTPUT_DIR = os.path.join(WORK_DIR, "output")
MODEL_DIR = os.path.join(WORK_DIR, "model")
for d in (INDEX_TRAIN_DIR, INDEX_TEST_DIR, SHARDS_DIR, OUTPUT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

TEAM_NAME = "team"   # used only for the optional final submission zip name

print("FAST_MODE:", FAST_MODE)
print("TRAIN_S1_SAMPLE:", TRAIN_S1_SAMPLE, " VAL_S1_SAMPLE:", VAL_S1_SAMPLE)
print("S1_CHUNK_SIZE:", S1_CHUNK_SIZE)
print("WORK_DIR:", WORK_DIR)


In [ ]:
import base64, os, sys

# ============================================================
# Embedded checkpoint modules (normalize/blocking/indexer/candidates/
# features/labels/checkpoint/similarity/data_loader). Embedding them
# as base64 means this notebook does NOT depend on a specific Kaggle
# dataset slug or attach path for the pipeline code itself -- only the
# raw train/test TSVs need to be discovered under /kaggle/input.
# ============================================================

MODULES_DIR = os.path.join(WORK_DIR, "er_modules")
os.makedirs(MODULES_DIR, exist_ok=True)

_MODULE_SOURCES_B64 = {
    "data_loader.py": "aW1wb3J0IGNzdg0KDQoNCmRlZiByZWFkX3RzdihwYXRoKToNCiAgICAiIiINCiAgICBSZWFkIGEgVFNWIGZpbGUgb25lIHJvdyBhdCBhIHRpbWUuDQogICAgIiIiDQoNCiAgICB3aXRoIG9wZW4oDQogICAgICAgIHBhdGgsDQogICAgICAgICJyIiwNCiAgICAgICAgZW5jb2Rpbmc9InV0Zi04IiwNCiAgICAgICAgZXJyb3JzPSJyZXBsYWNlIiwNCiAgICAgICAgbmV3bGluZT0iIg0KICAgICkgYXMgZjoNCg0KICAgICAgICByZWFkZXIgPSBjc3YuRGljdFJlYWRlcihmLCBkZWxpbWl0ZXI9Ilx0IikNCg0KICAgICAgICBmb3Igcm93IGluIHJlYWRlcjoNCiAgICAgICAgICAgIHlpZWxkIHJvdw0KDQoNCmRlZiBwcmV2aWV3X2ZpbGUocGF0aCwgbj01KToNCg0KICAgIHByaW50KCI9IiAqIDcwKQ0KICAgIHByaW50KCJGSUxFOiIsIHBhdGgpDQogICAgcHJpbnQoIj0iICogNzApDQoNCiAgICBmb3IgaSwgcm93IGluIGVudW1lcmF0ZShyZWFkX3RzdihwYXRoKSk6DQoNCiAgICAgICAgcHJpbnQocm93KQ0KDQogICAgICAgIGlmIGkgKyAxID49IG46DQogICAgICAgICAgICBicmVhaw0KDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQoNCiAgICBwcmV2aWV3X2ZpbGUoDQogICAgICAgIHIiZGF0YXNldFx0cmFpblx0cmFpbl9zb3VyY2UxLnRzdiIsDQogICAgICAgIG49NQ0KICAgICk=",
    "normalize.py": "IiIiCm5vcm1hbGl6ZS5weQo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KUFJFU0VSVkVEOiBub3JtYWxpemVfdGV4dCAvIG5vcm1hbGl6ZV9idXNpbmVzc19uYW1lIC8Kbm9ybWFsaXplX2FkZHJlc3MgYXJlIFVOQ0hBTkdFRCBmcm9tIHRoZSBvcmlnaW5hbCBpbXBsZW1lbnRhdGlvbgphbmQgYXJlIHN0aWxsIHdoYXQgYmxvY2tpbmcucHkncyBBU0NJSS1iYXNlZCBrZXlzIChOUDYsIE5QOCwgTlM2LApGTCwgRkwzLCBGMiwgRk1MLCBBTiwgQTIsIEFXMiwgQUwyLCBOTCwgQTRXLCBBVDIpIGFuZCB0aGUKZnJlcXVlbmN5LWNhcCB0dW5pbmcgaW4gYmxvY2tpbmcuREVGQVVMVF9GUkVRX0NBUFMgd2VyZSBidWlsdCBhbmQKbWVhc3VyZWQgYWdhaW5zdC4gRG8gbm90IGNoYW5nZSB0aGVpciBiZWhhdmlvciAtLSB0aGF0IHdvdWxkCnNpbGVudGx5IGludmFsaWRhdGUgZXZlcnkgcmVjYWxsIG51bWJlciBpbiBvdXRwdXQvKi50eHQuCgpBRERFRDogbm9ybWFsaXplX3VuaWNvZGVfcHJlc2VydmluZygpLCB1c2VkIGJ5IGZlYXR1cmVzLnB5IChub3QKYmxvY2tpbmcucHksIHdoaWNoIGFscmVhZHkgaGFkIGl0cyBvd24gZXF1aXZhbGVudCBsb2dpYyBpbmxpbmVkCmluIG1ha2VfdW5pY29kZV9uYW1lX2tleXMpLiBUaGlzIGNsb3NlcyBhIHJlYWwgZ2FwIGZvdW5kIGR1cmluZwp0aGUgUGhhc2UgMSBhdWRpdDoKCiAgbm9ybWFsaXplX2J1c2luZXNzX25hbWUoKSAvIG5vcm1hbGl6ZV9hZGRyZXNzKCkgc3RyaXAgZXZlcnkKICBjaGFyYWN0ZXIgb3V0c2lkZSBbYS16MC05XFxzXSBBRlRFUiBORktEIGRlY29tcG9zaXRpb24uIE5GS0QKICBkZWNvbXBvc2l0aW9uIG9ubHkgaGVscHMgc2NyaXB0cyB3aGVyZSB0aGUgYWNjZW50IGlzIGEgc2VwYXJhdGUKICBjb21iaW5pbmcgY2hhcmFjdGVyIG9uIHRvcCBvZiBhIExhdGluIGJhc2UgbGV0dGVyIChlLmcuICJlIiArCiAgY29tYmluaW5nIGFjdXRlKS4gSXQgZG9lcyBub3RoaW5nIGZvciBEZXZhbmFnYXJpLCBUZWx1Z3UsCiAgS2FubmFkYSwgQmVuZ2FsaSwgZXRjLiAtLSB0aG9zZSBjaGFyYWN0ZXJzIHNpbXBseSBoYXZlIG5vCiAgYS16MC05IGVxdWl2YWxlbnQsIHNvIHRoZXkgYXJlIGRlbGV0ZWQgb3V0cmlnaHQuCgogIEV4YW1wbGUgbWVhc3VyZWQgaW4gb3V0cHV0L3MyX21pc3NlZF9tYXRjaGVzX2luc3BlY3Rpb24udHN2OgogICAgIuCkj+CkuOCkj+CkuCDgpLjgpL/gpLjgpY3gpJ/gpK7gpY3gpLgg4KSy4KS/4KSu4KS/4KSf4KWH4KShIiAgLS0+ICBub3JtYWxpemVfYnVzaW5lc3NfbmFtZSgpICAtLT4gICIiCgogIGJsb2NraW5nLnB5J3MgbWFrZV91bmljb2RlX25hbWVfa2V5cygpIGFscmVhZHkgZ2VuZXJhdGVzCiAgY2FuZGlkYXRlLWJsb2NraW5nIGtleXMgc3RyYWlnaHQgZnJvbSB0aGUgdW50b3VjaGVkIFVuaWNvZGUKICBzdHJpbmcsIHdoaWNoIGlzIHdoeSB0aGVzZSBwYWlycyBETyBzaG93IHVwIGFzIGNhbmRpZGF0ZXMuCiAgQnV0IGZlYXR1cmVzLnB5IGN1cnJlbnRseSBjb21wdXRlcyBldmVyeSBuYW1lL2FkZHJlc3MKICBzaW1pbGFyaXR5IGZlYXR1cmUgZnJvbSBub3JtYWxpemVfYnVzaW5lc3NfbmFtZSgpIC8KICBub3JtYWxpemVfYWRkcmVzcygpIGFsb25lLiBPbmNlIGJvdGggc2lkZXMgbm9ybWFsaXplIHRvICIiLAogIGV2ZXJ5IHNpbWlsYXJpdHkgZmVhdHVyZSBmb3IgdGhhdCBwYWlyIGJlY29tZXMgMCAob3IgdGhlCiAgaGFyZC1jb2RlZCAibWlzc2luZyIgZGVmYXVsdCkgcmVnYXJkbGVzcyBvZiB3aGV0aGVyIHRoZSB0d28KICBVbmljb2RlIHN0cmluZ3MgYXJlIG5lYXItaWRlbnRpY2FsIG9yIGNvbXBsZXRlbHkgdW5yZWxhdGVkLgogIFRoZSBNTCBtb2RlbCBpbiBQaGFzZSAyIHdvdWxkIHRoZW4gaGF2ZSBubyBzaWduYWwgYXQgYWxsIHRvCiAgZGlzdGluZ3Vpc2ggYSB0cnVlIG1hdGNoIGZyb20gYSBmYWxzZSBwb3NpdGl2ZSBvbiB0aGlzCiAgc3Vic2V0IC0tIGkuZS4gYmxvY2tpbmcgcmVjYWxsIGlzIGZpbmUgZm9yIG5vbi1MYXRpbiBzY3JpcHRzLAogIGJ1dCBkb3duc3RyZWFtIHByZWNpc2lvbi9yZWNhbGwgb24gdGhvc2Ugcm93cyB3b3VsZCBjb2xsYXBzZS4KCiAgbm9ybWFsaXplX3VuaWNvZGVfcHJlc2VydmluZygpIGtlZXBzIGxldHRlcnMvZGlnaXRzL2NvbWJpbmluZwogIG1hcmtzIGluIEFOWSBzY3JpcHQgKHZpYSB1bmljb2RlZGF0YSBjYXRlZ29yeSwgbm90IGEgTGF0aW4KICByZWdleCkgc28gZmVhdHVyZXMucHkgY2FuIGNvbXB1dGUgYSBzZWNvbmQsIHNjcmlwdC1hZ25vc3RpYwogIHNldCBvZiBzaW1pbGFyaXR5IGZlYXR1cmVzIGFuZCBmYWxsIGJhY2sgdG8gaXQgd2hlbmV2ZXIgdGhlCiAgQVNDSUktZm9sZGVkIG5vcm1hbGl6YXRpb24gaXMgZW1wdHkgb24gZWl0aGVyIHNpZGUuCiIiIgoKaW1wb3J0IHJlCmltcG9ydCB1bmljb2RlZGF0YQoKCmRlZiBub3JtYWxpemVfdGV4dCh0ZXh0KToKICAgICIiIgogICAgQmFzaWMgbm9ybWFsaXphdGlvbiBmb3IgYnVzaW5lc3MgbmFtZXMgYW5kIGFkZHJlc3Nlcy4KICAgIFVOQ0hBTkdFRCAtLSBkbyBub3QgbW9kaWZ5LCBzZWUgbW9kdWxlIGRvY3N0cmluZy4KICAgICIiIgoKICAgIGlmIG5vdCB0ZXh0OgogICAgICAgIHJldHVybiAiIgoKICAgIHRleHQgPSBzdHIodGV4dCkKICAgIHRleHQgPSB0ZXh0Lmxvd2VyKCkKCiAgICB0ZXh0ID0gdW5pY29kZWRhdGEubm9ybWFsaXplKCJORktEIiwgdGV4dCkKICAgIHRleHQgPSAiIi5qb2luKGNoYXIgZm9yIGNoYXIgaW4gdGV4dCBpZiBub3QgdW5pY29kZWRhdGEuY29tYmluaW5nKGNoYXIpKQoKICAgIHRleHQgPSByZS5zdWIociJbXmEtejAtOVxzXSIsICIgIiwgdGV4dCkKICAgIHRleHQgPSByZS5zdWIociJccysiLCAiICIsIHRleHQpLnN0cmlwKCkKCiAgICByZXR1cm4gdGV4dAoKCmRlZiBub3JtYWxpemVfYnVzaW5lc3NfbmFtZShuYW1lKToKICAgICIiIgogICAgVU5DSEFOR0VEIC0tIGRvIG5vdCBtb2RpZnksIHNlZSBtb2R1bGUgZG9jc3RyaW5nLgogICAgIiIiCgogICAgbmFtZSA9IG5vcm1hbGl6ZV90ZXh0KG5hbWUpCgogICAgc3VmZml4ZXMgPSBbCiAgICAgICAgImluY29ycG9yYXRlZCIsCiAgICAgICAgImluYyIsCiAgICAgICAgImNvcnBvcmF0aW9uIiwKICAgICAgICAiY29ycCIsCiAgICAgICAgImxpbWl0ZWQiLAogICAgICAgICJsdGQiLAogICAgICAgICJsbGMiLAogICAgICAgICJjb21wYW55IiwKICAgICAgICAiY28iLAogICAgICAgICJwcml2YXRlIGxpbWl0ZWQiLAogICAgICAgICJwdnQgbHRkIiwKICAgIF0KCiAgICB3b3JkcyA9IG5hbWUuc3BsaXQoKQoKICAgIHdoaWxlIHdvcmRzIGFuZCAiICIuam9pbih3b3Jkc1stMjpdKSBpbiBzdWZmaXhlczoKICAgICAgICB3b3JkcyA9IHdvcmRzWzotMl0KCiAgICB3aGlsZSB3b3JkcyBhbmQgd29yZHNbLTFdIGluIHN1ZmZpeGVzOgogICAgICAgIHdvcmRzID0gd29yZHNbOi0xXQoKICAgIHJldHVybiAiICIuam9pbih3b3JkcykKCgpkZWYgbm9ybWFsaXplX2FkZHJlc3MoYWRkcmVzcyk6CiAgICAiIiIKICAgIFVOQ0hBTkdFRCAtLSBkbyBub3QgbW9kaWZ5LCBzZWUgbW9kdWxlIGRvY3N0cmluZy4KICAgICIiIgoKICAgIGFkZHJlc3MgPSBub3JtYWxpemVfdGV4dChhZGRyZXNzKQoKICAgIHJlcGxhY2VtZW50cyA9IHsKICAgICAgICAiIHN0cmVldCAiOiAiIHN0ICIsCiAgICAgICAgIiByb2FkICI6ICIgcmQgIiwKICAgICAgICAiIGF2ZW51ZSAiOiAiIGF2ZSAiLAogICAgICAgICIgYm91bGV2YXJkICI6ICIgYmx2ZCAiLAogICAgICAgICIgZHJpdmUgIjogIiBkciAiLAogICAgICAgICIgbGFuZSAiOiAiIGxuICIsCiAgICAgICAgIiBoaWdod2F5ICI6ICIgaHd5ICIsCiAgICAgICAgIiBhcGFydG1lbnQgIjogIiBhcHQgIiwKICAgIH0KCiAgICBhZGRyZXNzID0gIiAiICsgYWRkcmVzcyArICIgIgoKICAgIGZvciBvbGQsIG5ldyBpbiByZXBsYWNlbWVudHMuaXRlbXMoKToKICAgICAgICBhZGRyZXNzID0gYWRkcmVzcy5yZXBsYWNlKG9sZCwgbmV3KQoKICAgIHJldHVybiBhZGRyZXNzLnN0cmlwKCkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIE5FVzogc2NyaXB0LWFnbm9zdGljIG5vcm1hbGl6YXRpb24gZm9yIEZFQVRVUkUgY2FsY3VsYXRpb24KIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCiMgQSBzbWFsbCBzZXQgb2YgYnVzaW5lc3Mtc3VmZml4IFdPUkRTIHRoYXQgYXJlIG1lYW5pbmdmdWwgaW4KIyB0cmFuc2xpdGVyYXRlZC9FbmdsaXNoIGZvcm0gZXZlbiBpbnNpZGUgYW4gb3RoZXJ3aXNlIG5vbi1MYXRpbgojIHN0cmluZyAoZS5nLiAiUHZ0IEx0ZCIgdGFja2VkIG9udG8gYSBEZXZhbmFnYXJpIG5hbWUpLiBNYXRjaGluZwojIHRoZSBBU0NJSSBzdWZmaXggbGlzdCBrZWVwcyB0aGlzIGNvbnNpc3RlbnQgd2l0aAojIG5vcm1hbGl6ZV9idXNpbmVzc19uYW1lKCkgcmF0aGVyIHRoYW4gaW52ZW50aW5nIGEgc2Vjb25kIGxpc3QuCl9BU0NJSV9TVUZGSVhfV09SRFMgPSB7CiAgICAiaW5jb3Jwb3JhdGVkIiwgImluYyIsICJjb3Jwb3JhdGlvbiIsICJjb3JwIiwgImxpbWl0ZWQiLCAibHRkIiwKICAgICJsbGMiLCAiY29tcGFueSIsICJjbyIsICJwcml2YXRlIiwgInB2dCIsCn0KCgpkZWYgbm9ybWFsaXplX3VuaWNvZGVfcHJlc2VydmluZyh0ZXh0KToKICAgICIiIgogICAgTGlrZSBub3JtYWxpemVfdGV4dCgpLCBidXQga2VlcHMgbGV0dGVycy9kaWdpdHMvbWFya3MgZnJvbQogICAgQU5ZIHNjcmlwdCBpbnN0ZWFkIG9mIG9ubHkgW2EtejAtOV0uIFVzZWQgYnkgZmVhdHVyZXMucHkgYXMgYQogICAgZmFsbGJhY2sgd2hlbmV2ZXIgbm9ybWFsaXplX2J1c2luZXNzX25hbWUoKS9ub3JtYWxpemVfYWRkcmVzcygpCiAgICBjb2xsYXBzZSB0byAiIiBmb3IgYSBub24tTGF0aW4gc3RyaW5nLCBzbyBzaW1pbGFyaXR5IGZlYXR1cmVzCiAgICBzdGlsbCBjYXJyeSByZWFsIHNpZ25hbCBmb3IgSW5kaWEtbGFuZ3VhZ2UgYnVzaW5lc3MgbmFtZXMuCgogICAgTk9UIHVzZWQgYnkgYmxvY2tpbmcucHk6IGJsb2NraW5nIGFscmVhZHkgaGFzIGl0cyBvd24sCiAgICBpbmRlcGVuZGVudGx5LXR1bmVkIFVuaWNvZGUga2V5IGxvZ2ljIChtYWtlX3VuaWNvZGVfbmFtZV9rZXlzKQogICAgYW5kIGNoYW5naW5nIHRoYXQgd291bGQgbW92ZSB0aGUgbWVhc3VyZWQgcmVjYWxsIG51bWJlcnMuCiAgICAiIiIKCiAgICBpZiBub3QgdGV4dDoKICAgICAgICByZXR1cm4gIiIKCiAgICB0ZXh0ID0gc3RyKHRleHQpLnN0cmlwKCkKICAgIGlmIG5vdCB0ZXh0OgogICAgICAgIHJldHVybiAiIgoKICAgIHRleHQgPSB1bmljb2RlZGF0YS5ub3JtYWxpemUoIk5GS0MiLCB0ZXh0KS5jYXNlZm9sZCgpCgogICAgb3V0X2NoYXJzID0gW10KICAgIGZvciBjaCBpbiB0ZXh0OgogICAgICAgIGNhdGVnb3J5ID0gdW5pY29kZWRhdGEuY2F0ZWdvcnkoY2gpCiAgICAgICAgaWYgY2F0ZWdvcnkuc3RhcnRzd2l0aCgiTCIpIG9yIGNhdGVnb3J5LnN0YXJ0c3dpdGgoIk4iKSBvciBjYXRlZ29yeS5zdGFydHN3aXRoKCJNIik6CiAgICAgICAgICAgIG91dF9jaGFycy5hcHBlbmQoY2gpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgb3V0X2NoYXJzLmFwcGVuZCgiICIpCgogICAgdGV4dCA9IHJlLnN1YihyIlxzKyIsICIgIiwgIiIuam9pbihvdXRfY2hhcnMpKS5zdHJpcCgpCgogICAgcmV0dXJuIHRleHQKCgpkZWYgbm9ybWFsaXplX2J1c2luZXNzX25hbWVfc2NyaXB0X2Fnbm9zdGljKG5hbWUpOgogICAgIiIiCiAgICBub3JtYWxpemVfdW5pY29kZV9wcmVzZXJ2aW5nKCkgcGx1cyB0cmFpbGluZy1zdWZmaXggc3RyaXBwaW5nLAogICAgZm9yIGJ1c2luZXNzIG5hbWVzIHNwZWNpZmljYWxseS4gU3VmZml4IHN0cmlwcGluZyBvbmx5IHJlbW92ZXMKICAgIEFTQ0lJIHN1ZmZpeCB3b3JkcyAoc2VlIF9BU0NJSV9TVUZGSVhfV09SRFMpOyBub24tTGF0aW4gbGVnYWwKICAgIHN1ZmZpeGVzIGFyZSBsZWZ0IGluIHBsYWNlIHNpbmNlIGEgaGFyZC1jb2RlZCBBU0NJSSBsaXN0IGNhbid0CiAgICBpZGVudGlmeSB0aGVtLCBhbmQgbGVhdmluZyB0aGVtIGluIGRvZXMgbm90IGh1cnQgc2ltaWxhcml0eQogICAgc2NvcmluZyB0aGUgd2F5IGFuIGVtcHR5IHN0cmluZyBkb2VzLgogICAgIiIiCgogICAgdGV4dCA9IG5vcm1hbGl6ZV91bmljb2RlX3ByZXNlcnZpbmcobmFtZSkKICAgIGlmIG5vdCB0ZXh0OgogICAgICAgIHJldHVybiAiIgoKICAgIHdvcmRzID0gdGV4dC5zcGxpdCgpCiAgICB3aGlsZSB3b3JkcyBhbmQgd29yZHNbLTFdIGluIF9BU0NJSV9TVUZGSVhfV09SRFM6CiAgICAgICAgd29yZHMgPSB3b3Jkc1s6LTFdCgogICAgcmV0dXJuICIgIi5qb2luKHdvcmRzKQoKCmRlZiBpc19ub25fbGF0aW4odGV4dCk6CiAgICAiIiIKICAgIFRydWUgaWYgdGhlIEFTQ0lJLWZvbGRlZCBub3JtYWxpemF0aW9uIHdvdWxkIGxvc2UgcmVhbCBjb250ZW50CiAgICAtLSBpLmUuIG5vcm1hbGl6ZV9idXNpbmVzc19uYW1lKHRleHQpIHdvdWxkIGJlICIiIG9yIG11Y2gKICAgIHNob3J0ZXIgdGhhbiBub3JtYWxpemVfdW5pY29kZV9wcmVzZXJ2aW5nKHRleHQpLiBVc2VkIGJ5CiAgICBmZWF0dXJlcy5weSB0byBkZWNpZGUgd2hldGhlciB0byB0cnVzdCB0aGUgQVNDSUktZm9sZGVkCiAgICBmZWF0dXJlcyBvciBmYWxsIGJhY2sgdG8gdGhlIHNjcmlwdC1hZ25vc3RpYyBvbmVzLgogICAgIiIiCgogICAgYXNjaWlfZm9ybSA9IG5vcm1hbGl6ZV90ZXh0KHRleHQpCiAgICB1bmljb2RlX2Zvcm0gPSBub3JtYWxpemVfdW5pY29kZV9wcmVzZXJ2aW5nKHRleHQpCgogICAgaWYgbm90IHVuaWNvZGVfZm9ybToKICAgICAgICByZXR1cm4gRmFsc2UKCiAgICByZXR1cm4gbGVuKGFzY2lpX2Zvcm0ucmVwbGFjZSgiICIsICIiKSkgPCAwLjUgKiBsZW4odW5pY29kZV9mb3JtLnJlcGxhY2UoIiAiLCAiIikpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKCiAgICBleGFtcGxlcyA9IFsKICAgICAgICAiUGF5bmUgRW50ZXJwcmlzZXMiLAogICAgICAgICLgpI/gpLjgpI/gpLgg4KS44KS/4KS44KWN4KSf4KSu4KWN4KS4IOCksuCkv+CkruCkv+Ckn+Clh+CkoSIsCiAgICAgICAgIuCwtuCwv+CwtSDgsK7gsYfgsKjgsYfgsJzgsY3igIzgsK7gsYbgsILgsJ/gsY0g4LCq4LGN4LCw4LGI4LC14LGH4LCf4LGNIOCwsuCwv+CwruCwv+Cwn+CxhuCwoeCxjSIsCiAgICAgICAgIuCyrOCyvuCyguCyrOCzhiDgsqrgsrXgsrDgs40g4LKq4LON4LKw4LOI4LK14LOH4LKf4LONIOCysuCyv+CyruCyv+Cyn+CzhuCyoeCzjSIsCiAgICBdCgogICAgZm9yIG5hbWUgaW4gZXhhbXBsZXM6CiAgICAgICAgcHJpbnQoZiJ7bmFtZSFyfSIpCiAgICAgICAgcHJpbnQoZiIgIGFzY2lpLWZvbGRlZCA6IHtub3JtYWxpemVfYnVzaW5lc3NfbmFtZShuYW1lKSFyfSIpCiAgICAgICAgcHJpbnQoZiIgIHNjcmlwdC1hZ25vc3RpYzoge25vcm1hbGl6ZV9idXNpbmVzc19uYW1lX3NjcmlwdF9hZ25vc3RpYyhuYW1lKSFyfSIpCiAgICAgICAgcHJpbnQoZiIgIGlzX25vbl9sYXRpbiA6IHtpc19ub25fbGF0aW4obmFtZSl9IikK",
    "similarity.py": "ZnJvbSBkaWZmbGliIGltcG9ydCBTZXF1ZW5jZU1hdGNoZXINCmltcG9ydCByZQ0KDQpmcm9tIG5vcm1hbGl6ZSBpbXBvcnQgKA0KICAgIG5vcm1hbGl6ZV9idXNpbmVzc19uYW1lLA0KICAgIG5vcm1hbGl6ZV9hZGRyZXNzDQopDQoNCg0KZGVmIHRleHRfc2ltaWxhcml0eSh0ZXh0MSwgdGV4dDIpOg0KICAgICIiIg0KICAgIENoYXJhY3Rlci1sZXZlbCBzaW1pbGFyaXR5Lg0KICAgICIiIg0KDQogICAgaWYgbm90IHRleHQxIG9yIG5vdCB0ZXh0MjoNCiAgICAgICAgcmV0dXJuIDAuMA0KDQogICAgcmV0dXJuIFNlcXVlbmNlTWF0Y2hlcigNCiAgICAgICAgTm9uZSwNCiAgICAgICAgdGV4dDEsDQogICAgICAgIHRleHQyDQogICAgKS5yYXRpbygpDQoNCg0KZGVmIHRva2VuX3NpbWlsYXJpdHkodGV4dDEsIHRleHQyKToNCiAgICAiIiINCiAgICBDb21wYXJlIHNldHMgb2Ygd29yZHMuDQoNCiAgICBFeGFtcGxlOg0KDQogICAgInBheW5lIGVudGVycHJpc2VzIg0KICAgICJwYXluZSBlbnRlcnByaXNlcyBsbGMiDQoNCiAgICBzaGFyZSB0aGUgaW1wb3J0YW50IHRva2Vucy4NCiAgICAiIiINCg0KICAgIGlmIG5vdCB0ZXh0MSBvciBub3QgdGV4dDI6DQogICAgICAgIHJldHVybiAwLjANCg0KICAgIHRva2VuczEgPSBzZXQodGV4dDEuc3BsaXQoKSkNCiAgICB0b2tlbnMyID0gc2V0KHRleHQyLnNwbGl0KCkpDQoNCiAgICBpZiBub3QgdG9rZW5zMSBvciBub3QgdG9rZW5zMjoNCiAgICAgICAgcmV0dXJuIDAuMA0KDQogICAgaW50ZXJzZWN0aW9uID0gdG9rZW5zMSAmIHRva2VuczINCiAgICB1bmlvbiA9IHRva2VuczEgfCB0b2tlbnMyDQoNCiAgICByZXR1cm4gbGVuKGludGVyc2VjdGlvbikgLyBsZW4odW5pb24pDQoNCg0KZGVmIG5hbWVfc2ltaWxhcml0eShuYW1lMSwgbmFtZTIpOg0KICAgICIiIg0KICAgIENvbWJpbmUgY2hhcmFjdGVyIGFuZCB0b2tlbiBzaW1pbGFyaXR5Lg0KICAgICIiIg0KDQogICAgbmFtZTEgPSBub3JtYWxpemVfYnVzaW5lc3NfbmFtZShuYW1lMSkNCiAgICBuYW1lMiA9IG5vcm1hbGl6ZV9idXNpbmVzc19uYW1lKG5hbWUyKQ0KDQogICAgY2hhcl9zY29yZSA9IHRleHRfc2ltaWxhcml0eSgNCiAgICAgICAgbmFtZTEsDQogICAgICAgIG5hbWUyDQogICAgKQ0KDQogICAgdG9rZW5fc2NvcmUgPSB0b2tlbl9zaW1pbGFyaXR5KA0KICAgICAgICBuYW1lMSwNCiAgICAgICAgbmFtZTINCiAgICApDQoNCiAgICAjIEdpdmUgbW9yZSBpbXBvcnRhbmNlIHRvIGNoYXJhY3RlciBzaW1pbGFyaXR5LA0KICAgICMgd2hpbGUgYWxzbyBjb25zaWRlcmluZyBzaGFyZWQgd29yZHMuDQogICAgcmV0dXJuICgNCiAgICAgICAgMC42ICogY2hhcl9zY29yZQ0KICAgICAgICArDQogICAgICAgIDAuNCAqIHRva2VuX3Njb3JlDQogICAgKQ0KDQoNCmRlZiBhZGRyZXNzX3NpbWlsYXJpdHkoYWRkcmVzczEsIGFkZHJlc3MyKToNCiAgICAiIiINCiAgICBDb21wYXJlIG5vcm1hbGl6ZWQgYWRkcmVzc2VzLg0KICAgICIiIg0KDQogICAgYWRkcmVzczEgPSBub3JtYWxpemVfYWRkcmVzcyhhZGRyZXNzMSkNCiAgICBhZGRyZXNzMiA9IG5vcm1hbGl6ZV9hZGRyZXNzKGFkZHJlc3MyKQ0KDQogICAgcmV0dXJuIHRleHRfc2ltaWxhcml0eSgNCiAgICAgICAgYWRkcmVzczEsDQogICAgICAgIGFkZHJlc3MyDQogICAgKQ0KDQoNCmRlZiBjb3VudHJ5X21hdGNoKGNvdW50cnkxLCBjb3VudHJ5Mik6DQogICAgIiIiDQogICAgQ29tcGFyZSBjb3VudHJpZXMuDQoNCiAgICAxLjAgPSBzYW1lIGNvdW50cnkNCiAgICAwLjAgPSBkaWZmZXJlbnQgY291bnRyeQ0KICAgICIiIg0KDQogICAgaWYgbm90IGNvdW50cnkxIG9yIG5vdCBjb3VudHJ5MjoNCiAgICAgICAgcmV0dXJuIDAuMA0KDQogICAgcmV0dXJuIDEuMCBpZiAoDQogICAgICAgIGNvdW50cnkxLnN0cmlwKCkubG93ZXIoKQ0KICAgICAgICA9PQ0KICAgICAgICBjb3VudHJ5Mi5zdHJpcCgpLmxvd2VyKCkNCiAgICApIGVsc2UgMC4wDQoNCg0KZGVmIGV4dHJhY3RfYWRkcmVzc19udW1iZXJzKGFkZHJlc3MpOg0KICAgICIiIg0KICAgIEV4dHJhY3QgbnVtZXJpYyBwYXJ0cyBmcm9tIGFuIGFkZHJlc3MuDQogICAgIiIiDQoNCiAgICBpZiBub3QgYWRkcmVzczoNCiAgICAgICAgcmV0dXJuIFtdDQoNCiAgICBhZGRyZXNzID0gbm9ybWFsaXplX2FkZHJlc3MoYWRkcmVzcykNCg0KICAgIHJldHVybiByZS5maW5kYWxsKA0KICAgICAgICByIlxkKyIsDQogICAgICAgIGFkZHJlc3MNCiAgICApDQoNCg0KZGVmIGFkZHJlc3NfbnVtYmVyX21hdGNoKGFkZHJlc3MxLCBhZGRyZXNzMik6DQogICAgIiIiDQogICAgQ29tcGFyZSBudW1lcmljIHBhcnRzIG9mIGFkZHJlc3Nlcy4NCg0KICAgIDEuMCA9IG51bWJlcnMgbWF0Y2gNCiAgICAwLjAgPSBudW1iZXJzIGRpZmZlcg0KICAgIDAuNSA9IG51bWJlciB1bmF2YWlsYWJsZSBpbiBvbmUvYm90aA0KICAgICIiIg0KDQogICAgbnVtYmVyczEgPSBleHRyYWN0X2FkZHJlc3NfbnVtYmVycygNCiAgICAgICAgYWRkcmVzczENCiAgICApDQoNCiAgICBudW1iZXJzMiA9IGV4dHJhY3RfYWRkcmVzc19udW1iZXJzKA0KICAgICAgICBhZGRyZXNzMg0KICAgICkNCg0KICAgIGlmIG5vdCBudW1iZXJzMSBvciBub3QgbnVtYmVyczI6DQogICAgICAgIHJldHVybiAwLjUNCg0KICAgIGlmIG51bWJlcnMxID09IG51bWJlcnMyOg0KICAgICAgICByZXR1cm4gMS4wDQoNCiAgICByZXR1cm4gMC4wDQoNCg0KZGVmIGNhbGN1bGF0ZV9zaW1pbGFyaXR5KHJvdzEsIHJvdzIpOg0KICAgICIiIg0KICAgIENhbGN1bGF0ZSBzaW1pbGFyaXR5IGZlYXR1cmVzLg0KICAgICIiIg0KDQogICAgbmFtZV9zY29yZSA9IG5hbWVfc2ltaWxhcml0eSgNCiAgICAgICAgcm93MVsiYnVzaW5lc3NfbmFtZSJdLA0KICAgICAgICByb3cyWyJidXNpbmVzc19uYW1lIl0NCiAgICApDQoNCiAgICBhZGRyZXNzX3Njb3JlID0gYWRkcmVzc19zaW1pbGFyaXR5KA0KICAgICAgICByb3cxWyJidXNpbmVzc19hZGRyZXNzIl0sDQogICAgICAgIHJvdzJbImJ1c2luZXNzX2FkZHJlc3MiXQ0KICAgICkNCg0KICAgIGFkZHJlc3NfbnVtYmVyX3Njb3JlID0gYWRkcmVzc19udW1iZXJfbWF0Y2goDQogICAgICAgIHJvdzFbImJ1c2luZXNzX2FkZHJlc3MiXSwNCiAgICAgICAgcm93MlsiYnVzaW5lc3NfYWRkcmVzcyJdDQogICAgKQ0KDQogICAgY291bnRyeV9zY29yZSA9IGNvdW50cnlfbWF0Y2goDQogICAgICAgIHJvdzFbImNvdW50cnkiXSwNCiAgICAgICAgcm93MlsiY291bnRyeSJdDQogICAgKQ0KDQogICAgcmV0dXJuIHsNCiAgICAgICAgIm5hbWVfc2ltaWxhcml0eSI6IG5hbWVfc2NvcmUsDQogICAgICAgICJhZGRyZXNzX3NpbWlsYXJpdHkiOiBhZGRyZXNzX3Njb3JlLA0KICAgICAgICAiYWRkcmVzc19udW1iZXJfbWF0Y2giOiBhZGRyZXNzX251bWJlcl9zY29yZSwNCiAgICAgICAgImNvdW50cnlfbWF0Y2giOiBjb3VudHJ5X3Njb3JlDQogICAgfQ0KDQoNCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6DQoNCiAgICBzMSA9IHsNCiAgICAgICAgImJ1c2luZXNzX25hbWUiOiAiUGF5bmUgRW50ZXJwcmlzZXMiLA0KICAgICAgICAiYnVzaW5lc3NfYWRkcmVzcyI6ICIzMzE1IEZyZW1vbnQgU3RyZWV0LCBQZW9yaWEsIElMIiwNCiAgICAgICAgImNvdW50cnkiOiAiVVMiDQogICAgfQ0KDQogICAgczIgPSB7DQogICAgICAgICJidXNpbmVzc19uYW1lIjogIlBheW5lIMOJbnRlcnByaXNlcyBMTEMiLA0KICAgICAgICAiYnVzaW5lc3NfYWRkcmVzcyI6ICIzMzE1IEZSRU1PTlQgU1QsIFBFT1JJQSwgSUwiLA0KICAgICAgICAiY291bnRyeSI6ICJVUyINCiAgICB9DQoNCiAgICBmYWxzZV9tYXRjaCA9IHsNCiAgICAgICAgImJ1c2luZXNzX25hbWUiOiAiSW50ZXJuYXRpb25hbCBBdXRvbWF0aW9uIENvbnN1bHRhbnRzIENvcnAiLA0KICAgICAgICAiYnVzaW5lc3NfYWRkcmVzcyI6ICIzMzggUmV2IFdhbHRvbiBEciwgTG9ja3BvcnQsIElsbGlub2lzIiwNCiAgICAgICAgImNvdW50cnkiOiAiVVMiDQogICAgfQ0KDQogICAgdHJ1ZV9tYXRjaCA9IHsNCiAgICAgICAgImJ1c2luZXNzX25hbWUiOiAiSW50ZXJuYXRpb25hbCBBdXRvbWF0aW9uIENvbnN1bHRhbnRzIEluYyIsDQogICAgICAgICJidXNpbmVzc19hZGRyZXNzIjogIjMyOSBSZXYgV2FsdG9uIERyaXZlLCBMb2NrcG9ydCwgSUwiLA0KICAgICAgICAiY291bnRyeSI6ICJVUyINCiAgICB9DQoNCiAgICBwcmludCgiPSIgKiA3MCkNCiAgICBwcmludCgiUEFZTkUgRU5URVJQUklTRVMiKQ0KICAgIHByaW50KCI9IiAqIDcwKQ0KDQogICAgcHJpbnQoDQogICAgICAgIGNhbGN1bGF0ZV9zaW1pbGFyaXR5KA0KICAgICAgICAgICAgczEsDQogICAgICAgICAgICBzMg0KICAgICAgICApDQogICAgKQ0KDQogICAgcHJpbnQoIlxuIiArICI9IiAqIDcwKQ0KICAgIHByaW50KCJJTlRFUk5BVElPTkFMIEFVVE9NQVRJT04iKQ0KICAgIHByaW50KCI9IiAqIDcwKQ0KDQogICAgcHJpbnQoDQogICAgICAgIGNhbGN1bGF0ZV9zaW1pbGFyaXR5KA0KICAgICAgICAgICAgdHJ1ZV9tYXRjaCwNCiAgICAgICAgICAgIGZhbHNlX21hdGNoDQogICAgICAgICkNCiAgICAp",
    "blocking.py": "IiIiDQpibG9ja2luZy5weQ0KPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09DQpCbG9ja2luZy1rZXkgZ2VuZXJhdGlvbiBmb3IgQnVzaW5lc3MgRW50aXR5IFJlc29sdXRpb24uDQoNClRoaXMgbW9kdWxlIGlzIFNPVVJDRS1JTkRFUEVOREVOVDogdGhlIHNhbWUgZnVuY3Rpb25zIGFyZSB1c2VkDQpmb3IgU291cmNlIDEsIFNvdXJjZSAyIGFuZCBTb3VyY2UgMywgYW5kIGZvciBib3RoIHRoZSAxMEsNCmRldmVsb3BtZW50IGRhdGEgYW5kIHRoZSBmdWxsIDVNKyByZWNvcmQgcHJvZHVjdGlvbiBkYXRhLg0KDQpOb3RoaW5nIGluIHRoaXMgZmlsZSBpcyBoYXJkLWNvZGVkIHRvIGFueSBkYXRhc2V0IHNpemUuIEZyZXF1ZW5jeQ0KdGhyZXNob2xkcyBhcmUgTk9UIGNvbXB1dGVkIGhlcmUgLS0gdGhleSBhcmUgY29tcHV0ZWQgYnkgaW5kZXhlci5weQ0KYWdhaW5zdCB3aGF0ZXZlciBkYXRhIHdhcyBhY3R1YWxseSBpbmRleGVkICgxMEsgb3IgZnVsbCksIHdoaWNoIGlzDQp3aGF0IG1ha2VzIHRoaXMgcGlwZWxpbmUgc2FmZSB0byBtb3ZlIGZyb20gbGFiIC0+IHByb2R1Y3Rpb24uDQoNCktleSB0eXBlcyBwcm9kdWNlZDoNCg0KICBOYW1lIGtleXMgICA6IE5QNiwgTlA4LCBOUzYsIEZMLCBGTDMsIEYyLCBGTUwNCiAgQWRkcmVzcyBrZXlzOiBBTiwgQTIsIEFXMiwgQUwyLCBOTA0KICBBNCBrZXlzICAgICA6IEE0VyAobnVtZXJpYyBhZGRyZXNzIGNvbXBvbmVudCArIG5lYXJieSBhZGRyZXNzIHdvcmQsDQogICAgICAgICAgICAgICAgd2luZG93ZWQgc28gY29zdCBpcyBPKG4pIGluIHRoZSBudW1iZXIgb2YgYWRkcmVzcw0KICAgICAgICAgICAgICAgIHRva2Vucywgbm90IE8obl4yKSkNCiIiIg0KDQppbXBvcnQgcmUNCg0KZnJvbSBub3JtYWxpemUgaW1wb3J0IG5vcm1hbGl6ZV9idXNpbmVzc19uYW1lLCBub3JtYWxpemVfYWRkcmVzcw0KDQoNCg0KIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0NCiMgVU5JQ09ERSBOQU1FIEJMT0NLSU5HDQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQ0KDQpkZWYgbWFrZV91bmljb2RlX25hbWVfa2V5cyhuYW1lKToNCiAgICAiIiINCiAgICBVbmljb2RlLXByZXNlcnZpbmcgbmFtZSBrZXlzLg0KDQogICAgS2VlcHMgSGluZGksIEthbm5hZGEsIEd1amFyYXRpLCBldGMuIGluc3RlYWQgb2YgZGVsZXRpbmcNCiAgICBub24tQVNDSUkgY2hhcmFjdGVycy4NCiAgICAiIiINCiAgICBpZiBub3QgbmFtZToNCiAgICAgICAgcmV0dXJuIFtdDQoNCiAgICBpbXBvcnQgdW5pY29kZWRhdGENCg0KICAgIG4gPSB1bmljb2RlZGF0YS5ub3JtYWxpemUoIk5GS0MiLCBzdHIobmFtZSkpLmNhc2Vmb2xkKCkNCiAgICBuID0gIiAiLmpvaW4obi5zcGxpdCgpKQ0KDQogICAgaWYgbm90IG46DQogICAgICAgIHJldHVybiBbXQ0KDQogICAga2V5cyA9IFtdDQoNCiAgICBjb21wYWN0ID0gIiIuam9pbihjaCBmb3IgY2ggaW4gbiBpZiBub3QgY2guaXNzcGFjZSgpKQ0KDQogICAgaWYgbGVuKGNvbXBhY3QpID49IDQ6DQogICAgICAgIGtleXMuYXBwZW5kKCgiVU5QNCIsIGNvbXBhY3RbOjRdKSkNCg0KICAgIGlmIGxlbihjb21wYWN0KSA+PSA2Og0KICAgICAgICBrZXlzLmFwcGVuZCgoIlVOUDYiLCBjb21wYWN0Wzo2XSkpDQoNCiAgICB3b3JkcyA9IG4uc3BsaXQoKQ0KDQogICAgaWYgbGVuKHdvcmRzKSA+PSAyOg0KICAgICAgICBmaXJzdCA9IHdvcmRzWzBdDQogICAgICAgIGxhc3QgPSB3b3Jkc1stMV0NCg0KICAgICAgICBpZiBsZW4oZmlyc3QpID49IDIgYW5kIGxlbihsYXN0KSA+PSAyOg0KICAgICAgICAgICAga2V5cy5hcHBlbmQoKCJVRkwiLCBmaXJzdFs6M10gKyAifCIgKyBsYXN0WzozXSkpDQoNCiAgICByZXR1cm4ga2V5cw0KDQoNCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09DQojIE5BTUUgQkxPQ0tJTkcNCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09DQoNCmRlZiBtYWtlX25hbWVfa2V5cyhuYW1lKToNCiAgICAiIiINCiAgICBHZW5lcmF0ZSBzZWxlY3RpdmUgbmFtZS1ibG9ja2luZyBrZXlzLg0KDQogICAgV2UgYXZvaWQgZ2VuZXJpYyBzaW5nbGUtd29yZCBrZXlzIGJlY2F1c2UgdGhleSBjcmVhdGUNCiAgICBlbm9ybW91cyBjYW5kaWRhdGUgYnVja2V0cy4gTG9naWMgcHJlc2VydmVkIGZyb20gdGhlDQogICAgb3JpZ2luYWwgY2FuZGlkYXRlX2dlbmVyYXRvci5weSAoYWxyZWFkeSBtZWFzdXJlZCBhdA0KICAgIH45MiUgUzIgcmVjYWxsIG9uIHRoZSAxLDAwMC1yb3cgZGV2IHNhbXBsZSkuDQogICAgIiIiDQoNCiAgICBuID0gbm9ybWFsaXplX2J1c2luZXNzX25hbWUobmFtZSkNCg0KICAgIGlmIG5vdCBuOg0KICAgICAgICByZXR1cm4gW10NCg0KICAgIGtleXMgPSBbXQ0KICAgIGNvbXBhY3QgPSBuLnJlcGxhY2UoIiAiLCAiIikNCg0KICAgIGlmIGxlbihjb21wYWN0KSA+PSA2Og0KICAgICAgICBrZXlzLmFwcGVuZCgoIk5QNiIsIGNvbXBhY3RbOjZdKSkNCg0KICAgIGlmIGxlbihjb21wYWN0KSA+PSA4Og0KICAgICAgICBrZXlzLmFwcGVuZCgoIk5QOCIsIGNvbXBhY3RbOjhdKSkNCg0KICAgIGlmIGxlbihjb21wYWN0KSA+PSA2Og0KICAgICAgICBrZXlzLmFwcGVuZCgoIk5TNiIsIGNvbXBhY3RbLTY6XSkpDQoNCiAgICB3b3JkcyA9IG4uc3BsaXQoKQ0KDQogICAgaWYgbGVuKHdvcmRzKSA+PSAyOg0KICAgICAgICBmaXJzdCwgbGFzdCA9IHdvcmRzWzBdLCB3b3Jkc1stMV0NCg0KICAgICAgICBpZiBsZW4oZmlyc3QpID49IDMgYW5kIGxlbihsYXN0KSA+PSAzOg0KICAgICAgICAgICAga2V5cy5hcHBlbmQoKCJGTCIsIGZpcnN0Wzo0XSArICJ8IiArIGxhc3RbOjRdKSkNCiAgICAgICAgICAgIGtleXMuYXBwZW5kKCgiRkwzIiwgZmlyc3RbOjNdICsgInwiICsgbGFzdFs6NV0pKQ0KDQogICAgICAgIHcxLCB3MiA9IHdvcmRzWzBdLCB3b3Jkc1sxXQ0KDQogICAgICAgIGlmIGxlbih3MSkgPj0gMyBhbmQgbGVuKHcyKSA+PSAzOg0KICAgICAgICAgICAga2V5cy5hcHBlbmQoKCJGMiIsIHcxWzo1XSArICJ8IiArIHcyWzo1XSkpDQoNCiAgICBpZiBsZW4od29yZHMpID49IDM6DQogICAgICAgIGZpcnN0LCBtaWRkbGUsIGxhc3QgPSB3b3Jkc1swXSwgd29yZHNbLTJdLCB3b3Jkc1stMV0NCg0KICAgICAgICBpZiBsZW4oZmlyc3QpID49IDMgYW5kIGxlbihsYXN0KSA+PSAzOg0KICAgICAgICAgICAga2V5cy5hcHBlbmQoDQogICAgICAgICAgICAgICAgKCJGTUwiLCBmaXJzdFs6M10gKyAifCIgKyBtaWRkbGVbOjNdICsgInwiICsgbGFzdFs6M10pDQogICAgICAgICAgICApDQoNCiAgICByZXR1cm4ga2V5cw0KDQoNCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09DQojIEFERFJFU1MgQkxPQ0tJTkcgKG9yaWdpbmFsIGtleSBzZXQpDQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQ0KDQpkZWYgbWFrZV9hZGRyZXNzX2tleXMoYWRkcmVzcyk6DQogICAgIiIiDQogICAgR2VuZXJhdGUgc2VsZWN0aXZlIGFkZHJlc3MtYmxvY2tpbmcga2V5cy4NCg0KICAgIFdlIGRlbGliZXJhdGVseSBkbyBOT1QgdXNlIGEgYmFyZSBudW1iZXIgYXMgYSBrZXkgLS0gbG9naWMNCiAgICBwcmVzZXJ2ZWQgZnJvbSB0aGUgb3JpZ2luYWwgY2FuZGlkYXRlX2dlbmVyYXRvci5weS4NCiAgICAiIiINCg0KICAgIGEgPSBub3JtYWxpemVfYWRkcmVzcyhhZGRyZXNzKQ0KDQogICAgaWYgbm90IGE6DQogICAgICAgIHJldHVybiBbXQ0KDQogICAga2V5cyA9IFtdDQogICAgd29yZHMgPSBhLnNwbGl0KCkNCg0KICAgIGlmIG5vdCB3b3JkczoNCiAgICAgICAgcmV0dXJuIFtdDQoNCiAgICBudW1iZXJzID0gW3cgZm9yIHcgaW4gd29yZHMgaWYgdy5pc2RpZ2l0KCldDQogICAgbm9uX251bWVyaWMgPSBbdyBmb3IgdyBpbiB3b3JkcyBpZiBub3Qgdy5pc2RpZ2l0KCldDQoNCiAgICBpZiBudW1iZXJzOg0KICAgICAgICBudW1iZXIgPSBudW1iZXJzWzBdDQoNCiAgICAgICAgZm9yIHdvcmQgaW4gd29yZHM6DQogICAgICAgICAgICBpZiBub3Qgd29yZC5pc2RpZ2l0KCkgYW5kIGxlbih3b3JkKSA+PSAzOg0KICAgICAgICAgICAgICAgIGtleXMuYXBwZW5kKCgiQU4iLCBudW1iZXIgKyAifCIgKyB3b3JkWzo2XSkpDQogICAgICAgICAgICAgICAgYnJlYWsNCg0KICAgICAgICBpZiBsZW4obm9uX251bWVyaWMpID49IDI6DQogICAgICAgICAgICBrZXlzLmFwcGVuZCgNCiAgICAgICAgICAgICAgICAoDQogICAgICAgICAgICAgICAgICAgICJBMiIsDQogICAgICAgICAgICAgICAgICAgIG51bWJlciArICJ8IiArIG5vbl9udW1lcmljWzBdWzo0XSArICJ8IiArIG5vbl9udW1lcmljWzFdWzo0XSwNCiAgICAgICAgICAgICAgICApDQogICAgICAgICAgICApDQoNCiAgICBpZiBsZW4obm9uX251bWVyaWMpID49IDI6DQogICAgICAgIHcxLCB3MiA9IG5vbl9udW1lcmljWzBdLCBub25fbnVtZXJpY1sxXQ0KDQogICAgICAgIGlmIGxlbih3MSkgPj0gMyBhbmQgbGVuKHcyKSA+PSAzOg0KICAgICAgICAgICAga2V5cy5hcHBlbmQoKCJBVzIiLCB3MVs6NV0gKyAifCIgKyB3Mls6NV0pKQ0KDQogICAgICAgIHcxLCB3MiA9IG5vbl9udW1lcmljWy0yXSwgbm9uX251bWVyaWNbLTFdDQoNCiAgICAgICAgaWYgbGVuKHcxKSA+PSAzIGFuZCBsZW4odzIpID49IDM6DQogICAgICAgICAgICBrZXlzLmFwcGVuZCgoIkFMMiIsIHcxWzo1XSArICJ8IiArIHcyWzo1XSkpDQoNCiAgICBpZiBudW1iZXJzIGFuZCBub25fbnVtZXJpYzoNCiAgICAgICAgbnVtYmVyID0gbnVtYmVyc1swXQ0KICAgICAgICBsYXN0X3dvcmQgPSBub25fbnVtZXJpY1stMV0NCg0KICAgICAgICBpZiBsZW4obGFzdF93b3JkKSA+PSAzOg0KICAgICAgICAgICAga2V5cy5hcHBlbmQoKCJOTCIsIG51bWJlciArICJ8IiArIGxhc3Rfd29yZFs6Nl0pKQ0KDQogICAgcmV0dXJuIGtleXMNCg0KDQoNCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09DQojIEFUMiBCTE9DS0lORyAoaW5mb3JtYXRpdmUgYWRkcmVzcy10b2tlbiBwYWlycykNCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09DQoNCiMgVmVyeSBjb21tb24gYWRkcmVzcyB3b3JkcyBhcmUgcG9vciBibG9ja2luZyBzaWduYWxzLg0KIyBLZWVwIHRoaXMgbGlzdCBkZWxpYmVyYXRlbHkgc21hbGwuDQpHRU5FUklDX0FERFJFU1NfV09SRFMgPSB7DQogICAgInN0cmVldCIsICJzdCIsICJyb2FkIiwgInJkIiwgImF2ZW51ZSIsICJhdmUiLA0KICAgICJsYW5lIiwgImxuIiwgImRyaXZlIiwgImRyIiwgImJvdWxldmFyZCIsICJibHZkIiwNCiAgICAiaGlnaHdheSIsICJod3kiLCAibmVhciIsICJvcHBvc2l0ZSIsICJvcHAiLA0KICAgICJidWlsZGluZyIsICJibGRnIiwgImZsb29yIiwgImZsIiwgImJsb2NrIiwNCiAgICAiZGlzdHJpY3QiLCAiY2l0eSIsICJzdGF0ZSIsICJpbmRpYSIsDQogICAgInByaXZhdGUiLCAibGltaXRlZCIsICJsdGQiLCAicHZ0IiwNCn0NCg0KZGVmIG1ha2VfYXQyX2tleXMoYWRkcmVzcyk6DQogICAgIiIiDQogICAgR2VuZXJhdGUgYSBzbWFsbCBudW1iZXIgb2YgaW5mb3JtYXRpdmUgYWRkcmVzcy10b2tlbi1wYWlyIGtleXMuDQoNCiAgICBVbmxpa2UgQVcyL0FMMiwgdGhpcyBkb2VzIG5vdCBkZXBlbmQgb24gdG9rZW4gcG9zaXRpb24uDQogICAgSXQgc2VsZWN0cyBpbmZvcm1hdGl2ZSB0b2tlbnMgYW5kIGNyZWF0ZXMgb25seSBhZGphY2VudCBwYWlycw0KICAgIGFmdGVyIHNvcnRpbmcgYnkgdGhlaXIgb3JpZ2luYWwgcG9zaXRpb24uDQoNCiAgICBUaGUgZ29hbCBpcyB0byByZWNvdmVyIG1hdGNoZXMgd2hlcmUgdGhlIGJ1c2luZXNzIG5hbWUgaXMgaW4NCiAgICBhbm90aGVyIHNjcmlwdCBidXQgdGhlIGFkZHJlc3MgaXMgc2hhcmVkLg0KICAgICIiIg0KICAgIGEgPSBub3JtYWxpemVfYWRkcmVzcyhhZGRyZXNzKQ0KDQogICAgaWYgbm90IGE6DQogICAgICAgIHJldHVybiBbXQ0KDQogICAgdG9rZW5zID0gYS5zcGxpdCgpDQoNCiAgICBpbmZvcm1hdGl2ZSA9IFtdDQoNCiAgICBmb3IgdG9rIGluIHRva2VuczoNCiAgICAgICAgaWYgdG9rLmlzZGlnaXQoKToNCiAgICAgICAgICAgIGNvbnRpbnVlDQoNCiAgICAgICAgaWYgbGVuKHRvaykgPCA0Og0KICAgICAgICAgICAgY29udGludWUNCg0KICAgICAgICBpZiB0b2sgaW4gR0VORVJJQ19BRERSRVNTX1dPUkRTOg0KICAgICAgICAgICAgY29udGludWUNCg0KICAgICAgICBpbmZvcm1hdGl2ZS5hcHBlbmQodG9rKQ0KDQogICAgaWYgbGVuKGluZm9ybWF0aXZlKSA8IDI6DQogICAgICAgIHJldHVybiBbXQ0KDQogICAgIyBLZWVwIGF0IG1vc3QgNiBpbmZvcm1hdGl2ZSB0b2tlbnMgcGVyIGFkZHJlc3MuDQogICAgIyBQcmVmZXIgbG9uZ2VyIHRva2VucyBiZWNhdXNlIHRoZXkgYXJlIGdlbmVyYWxseSBtb3JlIGRpc3RpbmN0aXZlLg0KICAgIGluZm9ybWF0aXZlID0gc29ydGVkKA0KICAgICAgICBpbmZvcm1hdGl2ZSwNCiAgICAgICAga2V5PWxhbWJkYSB4OiAoLWxlbih4KSwgeCkNCiAgICApWzo2XQ0KDQogICAga2V5cyA9IFtdDQogICAgc2VlbiA9IHNldCgpDQoNCiAgICAjIE9ubHkgNS02IHBhaXJzLCByYXRoZXIgdGhhbiBPKG5eMikgb3ZlciB0aGUgd2hvbGUgYWRkcmVzcy4NCiAgICBmb3IgaSBpbiByYW5nZShsZW4oaW5mb3JtYXRpdmUpIC0gMSk6DQogICAgICAgIHcxID0gaW5mb3JtYXRpdmVbaV1bOjhdDQogICAgICAgIHcyID0gaW5mb3JtYXRpdmVbaSArIDFdWzo4XQ0KDQogICAgICAgIGlmIHcxID09IHcyOg0KICAgICAgICAgICAgY29udGludWUNCg0KICAgICAgICBhMSwgYTIgPSBzb3J0ZWQoKHcxLCB3MikpDQogICAgICAgIGtleV92YWx1ZSA9IGExICsgInwiICsgYTINCg0KICAgICAgICBpZiBrZXlfdmFsdWUgbm90IGluIHNlZW46DQogICAgICAgICAgICBzZWVuLmFkZChrZXlfdmFsdWUpDQogICAgICAgICAgICBrZXlzLmFwcGVuZCgoIkFUMiIsIGtleV92YWx1ZSkpDQoNCiAgICByZXR1cm4ga2V5cw0KDQoNCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09DQojIEE0IEJMT0NLSU5HIChudW1lcmljIGFkZHJlc3MgY29tcG9uZW50ICsgTkVBUkJZIHdvcmRzKQ0KIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0NCiMNCiMgVGhlIG9yaWdpbmFsIEE0IGV4cGVyaW1lbnQgKHRlc3RfYTRfb3ZlcmhlYWRfZmFzdC5weSkgcGFpcmVkIGV2ZXJ5DQojIG51bWJlciBpbiB0aGUgYWRkcmVzcyB3aXRoIEVWRVJZIHBhaXIgb2Ygd29yZHMgKE8obl4yKSBwZXIgcmVjb3JkKS4NCiMgVGhhdCBpcyBmaW5lIGZvciBhIG9uZS1vZmYgMSwwMDAtcm93IGV4cGVyaW1lbnQgYnV0IGlzIG5vdCBzYWZlIHRvDQojIHJ1biB1bm1vZGlmaWVkIGFnYWluc3QgNU0rIHJlY29yZHM6IGFkZHJlc3NlcyB3aXRoIG1hbnkgdG9rZW5zDQojIHdvdWxkIGdlbmVyYXRlIGEgbGFyZ2UgbnVtYmVyIG9mIGtleXMgcGVyIHJlY29yZC4NCiMNCiMgVGhpcyBwcm9kdWN0aW9uIHZlcnNpb24ga2VlcHMgdGhlIHNhbWUgaWRlYSAtLSAibnVtZXJpYyBhZGRyZXNzDQojIGNvbXBvbmVudCArIG5lYXJieSBub3JtYWxpemVkIGFkZHJlc3Mgd29yZHMiIC0tIGJ1dCBvbmx5IGxvb2tzIGF0DQojIGEgc21hbGwgd2luZG93IG9mIHdvcmRzIGFyb3VuZCBlYWNoIG51bWJlciAoV0lORE9XIG9uIGVhY2ggc2lkZSksDQojIG1ha2luZyBrZXkgZ2VuZXJhdGlvbiBPKG4pIHBlciByZWNvcmQgaW5zdGVhZCBvZiBPKG5eMikuIFJlY2FsbA0KIyBvbiB0aGUgZGV2IHNldCBzaG91bGQgc3RheSBjbG9zZSB0byB0aGUgb3JpZ2luYWwgZXhwZXJpbWVudA0KIyBiZWNhdXNlIHRydWUgbWF0Y2hlcyBhbG1vc3QgYWx3YXlzIGtlZXAgbnVtYmVycyBhbmQgc3RyZWV0IHdvcmRzDQojIGFkamFjZW50IHRvIG9uZSBhbm90aGVyLg0KDQpBNF9XSU5ET1cgPSAzICAjIHdvcmRzIGV4YW1pbmVkIG9uIGVhY2ggc2lkZSBvZiBhIG51bWJlcg0KDQoNCmRlZiBtYWtlX2E0X2tleXMoYWRkcmVzcywgd2luZG93PUE0X1dJTkRPVyk6DQoNCiAgICBhID0gbm9ybWFsaXplX2FkZHJlc3MoYWRkcmVzcykNCg0KICAgIGlmIG5vdCBhOg0KICAgICAgICByZXR1cm4gW10NCg0KICAgIHRva2VucyA9IGEuc3BsaXQoKQ0KDQogICAgbnVtYmVyX3Bvc2l0aW9ucyA9IFtpIGZvciBpLCB0IGluIGVudW1lcmF0ZSh0b2tlbnMpIGlmIHQuaXNkaWdpdCgpXQ0KDQogICAgaWYgbm90IG51bWJlcl9wb3NpdGlvbnM6DQogICAgICAgIHJldHVybiBbXQ0KDQogICAga2V5cyA9IFtdDQogICAgc2VlbiA9IHNldCgpDQoNCiAgICBmb3IgcG9zIGluIG51bWJlcl9wb3NpdGlvbnM6DQoNCiAgICAgICAgbnVtYmVyID0gdG9rZW5zW3Bvc10NCg0KICAgICAgICBuZWFyYnkgPSBbXQ0KDQogICAgICAgIGZvciBvZmZzZXQgaW4gcmFuZ2UoMSwgd2luZG93ICsgMSk6DQoNCiAgICAgICAgICAgIGZvciBpZHggaW4gKHBvcyAtIG9mZnNldCwgcG9zICsgb2Zmc2V0KToNCg0KICAgICAgICAgICAgICAgIGlmIDAgPD0gaWR4IDwgbGVuKHRva2Vucyk6DQoNCiAgICAgICAgICAgICAgICAgICAgdG9rID0gdG9rZW5zW2lkeF0NCg0KICAgICAgICAgICAgICAgICAgICBpZiBub3QgdG9rLmlzZGlnaXQoKSBhbmQgbGVuKHRvaykgPj0gMjoNCiAgICAgICAgICAgICAgICAgICAgICAgIG5lYXJieS5hcHBlbmQodG9rKQ0KDQogICAgICAgICMgZGUtZHVwIHdoaWxlIHByZXNlcnZpbmcgb3JkZXIsIGNhcCB0byB0aGUgMyBjbG9zZXN0IHdvcmRzDQogICAgICAgIG5lYXJieSA9IGxpc3QoZGljdC5mcm9ta2V5cyhuZWFyYnkpKVs6M10NCg0KICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4obmVhcmJ5KSk6DQogICAgICAgICAgICBmb3IgaiBpbiByYW5nZShpICsgMSwgbGVuKG5lYXJieSkpOg0KDQogICAgICAgICAgICAgICAgdzEsIHcyID0gc29ydGVkKChuZWFyYnlbaV1bOjZdLCBuZWFyYnlbal1bOjZdKSkNCiAgICAgICAgICAgICAgICBrZXlfdmFsdWUgPSBudW1iZXIgKyAifCIgKyB3MSArICJ8IiArIHcyDQoNCiAgICAgICAgICAgICAgICBpZiBrZXlfdmFsdWUgbm90IGluIHNlZW46DQogICAgICAgICAgICAgICAgICAgIHNlZW4uYWRkKGtleV92YWx1ZSkNCiAgICAgICAgICAgICAgICAgICAga2V5cy5hcHBlbmQoKCJBNFciLCBrZXlfdmFsdWUpKQ0KDQogICAgcmV0dXJuIGtleXMNCg0KDQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQ0KIyBDT01CSU5FRA0KIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0NCg0KZGVmIG1ha2VfYWxsX2tleXMobmFtZSwgYWRkcmVzcyk6DQogICAgIiIiDQogICAgQ29udmVuaWVuY2UgaGVscGVyIHJldHVybmluZyBldmVyeSAoa2V5X3R5cGUsIGtleV92YWx1ZSkgcGFpcg0KICAgIGZvciBhIHJlY29yZCdzIG5hbWUgKyBhZGRyZXNzLCB1c2VkIGJ5IGluZGV4ZXIucHkuDQogICAgIiIiDQoNCiAgICByZXR1cm4gbWFrZV9uYW1lX2tleXMobmFtZSkgKyBtYWtlX3VuaWNvZGVfbmFtZV9rZXlzKG5hbWUpICsgbWFrZV9hZGRyZXNzX2tleXMoYWRkcmVzcykgKyBtYWtlX2E0X2tleXMoYWRkcmVzcykNCg0KDQpOQU1FX0tFWV9UWVBFUyA9IHsiTlA2IiwgIk5QOCIsICJOUzYiLCAiRkwiLCAiRkwzIiwgIkYyIiwgIkZNTCIsICJVTlA0IiwgIlVOUDYiLCAiVUZMIn0NCkFERFJFU1NfS0VZX1RZUEVTID0geyJBTiIsICJBMiIsICJBVzIiLCAiQUwyIiwgIk5MIiwgIkE0VyIsICJBVDIifQ0KQUxMX0tFWV9UWVBFUyA9IE5BTUVfS0VZX1RZUEVTIHwgQUREUkVTU19LRVlfVFlQRVMNCg0KDQojIERlZmF1bHQgcGVyLWtleS10eXBlIG1heGltdW0gYnVja2V0IHNpemUgKCJmcmVxdWVuY3kgY2FwIikuDQojIElmIGEgc3BlY2lmaWMga2V5J3MgYnVja2V0IChmb3IgdGhlIGRhdGFzZXQgYWN0dWFsbHkgaW5kZXhlZCAtLQ0KIyAxMEsgZGV2IE9SIGZ1bGwgcHJvZHVjdGlvbikgZ3Jvd3MgcGFzdCB0aGlzIHNpemUsIHRoYXQga2V5IGlzDQojIHRyZWF0ZWQgYXMgdG9vIGJyb2FkIGFuZCBza2lwcGVkIGZvciB0aGF0IHJlY29yZC4gVGhpcyBpcyB3aGF0DQojIGtlZXBzIEFMMiAvIE5TNiB1c2FibGUgd2l0aG91dCBkZWxldGluZyB0aGVtIG91dHJpZ2h0LCBhbmQgaXMNCiMgcmVjb21wdXRlZCBmcmVzaCBldmVyeSB0aW1lIGJ1aWxkX2luZGV4KCkgcnVucywgc28gaXQgaXMgYWx3YXlzDQojIGV2YWx1YXRlZCBhZ2FpbnN0IHRoZSBDVVJSRU5UIGluZGV4LCBuZXZlciBhZ2FpbnN0IHN0YWxlIDEwSw0KIyBzdGF0aXN0aWNzLg0KREVGQVVMVF9GUkVRX0NBUFMgPSB7DQogICAgIk5QNiI6IDMwMCwNCiAgICAiTlA4IjogMzAwLA0KICAgICJOUzYiOiAzMDAsDQogICAgIkZMIjogNDAwLA0KICAgICJGTDMiOiA0MDAsDQogICAgIkYyIjogNDAwLA0KICAgICJGTUwiOiA0MDAsDQogICAgIkFOIjogNDAwLA0KICAgICJBMiI6IDQwMCwNCiAgICAiQVcyIjogNDAwLA0KICAgICJBTDIiOiAxNjAsDQogICAgIk5MIjogNDAwLA0KICAgICJBNFciOiAyMCwNCiAgICAiQVQyIjogNDAsDQogICAgIlVOUDQiOiAzMDAsDQogICAgIlVOUDYiOiAzMDAsDQogICAgIlVGTCI6IDMwMCwNCn0NCg==",
    "indexer.py": "IiIiCmluZGV4ZXIucHkKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkNIQU5HRVMgZnJvbSB0aGUgb3JpZ2luYWwgKHNlZSBQSEFTRTFfQVJDSElURUNUVVJFX0FVRElULm1kLAoiRXhpc3RpbmcgU1FMaXRlIGFyY2hpdGVjdHVyZSIgYW5kICJDaGVja3BvaW50L3Jlc3VtZSBwbGFuIik6CgoxLiBidWNrZXQoKSBub3cgYWNjZXB0cyBhbiBvcHRpb25hbCBgbGltaXRgLiBjYW5kaWRhdGVzLnB5IHBhc3NlcwogICBsaW1pdCA9IGNhcCArIDEuIFByZXZpb3VzbHkgU3FsaXRlSW5kZXguYnVja2V0KCkgYWx3YXlzIHJhbgogICBgU0VMRUNUIHN1ZmZpeCBGUk9NIHBvc3RpbmdzIFdIRVJFIGtleV90eXBlPT8gQU5EIGtleV92YWx1ZT0/YAogICB3aXRoIG5vIExJTUlUIGFuZCBjYWxsZWQgZmV0Y2hhbGwoKSAtLSBmb3IgYSBob3Qga2V5IChhIGNvbW1vbgogICA2LWNoYXJhY3RlciBuYW1lIHByZWZpeCwgc2F5KSB3aG9zZSBidWNrZXQgaXMgZmFyIGxhcmdlciB0aGFuCiAgIGl0cyBmcmVxdWVuY3kgY2FwLCB0aGlzIGZldGNoZWQgYW5kIG1hdGVyaWFsaXplZCB0aGUgRU5USVJFCiAgIG92ZXJzaXplZCBidWNrZXQganVzdCB0byBpbW1lZGlhdGVseSBkaXNjYXJkIGl0IGluCiAgIGNhbmRpZGF0ZXMucHkncyBgaWYgYnVja2V0X3NpemUgPiBjYXA6IGNvbnRpbnVlYC4gQXQgZnVsbAogICBwcm9kdWN0aW9uIHNjYWxlIChTMi9TMyB+NU0gcm93cyBlYWNoLCB2cy4gdGhlIDEwMEstcm93IGluZGV4CiAgIHRoZSA5OC40NSUvOTcuOTYlIG51bWJlcnMgd2VyZSBtZWFzdXJlZCBhZ2FpbnN0KSBob3QgYnVja2V0cwogICBhcmUgcHJvcG9ydGlvbmFsbHkgbGFyZ2VyLCBzbyB0aGlzIHdhcyBvbiB0cmFjayB0byBiZWNvbWUgb25lCiAgIG9mIHRoZSBiaWdnZXN0IGF2b2lkYWJsZSBjb3N0cyBpbiBjYW5kaWRhdGUgZ2VuZXJhdGlvbi4gV2l0aCBhCiAgIExJTUlUIG9mIGNhcCsxLCBhbiBvdmVyc2l6ZWQgYnVja2V0IG5vdyBjb3N0cyBPKGNhcCkgdG8gZGV0ZWN0CiAgIGFzIG92ZXJzaXplZCwgbm90IE8oYnVja2V0IHNpemUpLgoKMi4gU3FsaXRlSW5kZXggbm93IGNvbW1pdHMgcGVyaW9kaWNhbGx5IChldmVyeSBmbHVzaCgpLCBub3Qgb25seQogICBhdCBmaW5hbGl6ZSgpKSBhbmQgd3JpdGVzIGEgc21hbGwgSlNPTiBtYW5pZmVzdCByZWNvcmRpbmcgaG93CiAgIG1hbnkgc291cmNlIHJvd3MgaGF2ZSBiZWVuIGluZGV4ZWQuIE9uIHJlc3RhcnQsIGJ1aWxkX2luZGV4KCkKICAgY2hlY2tzIHRoZSBtYW5pZmVzdDogaWYgYSBzb3VyY2UncyBpbmRleCBwcmV2aW91c2x5IGZpbmlzaGVkLAogICBpdCBpcyByZXVzZWQgYXMtaXMgKG5vIHJlYnVpbGQpOyBpZiBpdCB3YXMgbGVmdCBwYXJ0d2F5CiAgIHRocm91Z2gsIGluZGV4aW5nIHJlc3VtZXMgZnJvbSB0aGUgbGFzdCBjb21wbGV0ZWQgcm93IGluc3RlYWQKICAgb2Ygc3RhcnRpbmcgb3Zlci4gVGhpcyBpcyB3aGF0IG1ha2VzIGEgS2FnZ2xlIHNlc3Npb24KICAgaW50ZXJydXB0aW9uIERVUklORyBpbmRleC1idWlsZGluZyByZWNvdmVyYWJsZSAtLSBwcmV2aW91c2x5CiAgIHRoZSB3aG9sZSBidWlsZCBzYXQgaW5zaWRlIG9uZSBvcGVuIFNRTGl0ZSB0cmFuc2FjdGlvbiB3aXRoIG5vCiAgIHJlY29yZGVkIHByb2dyZXNzLCBzbyBhIGtpbGwgYXQgcm93IDQsOTAwLDAwMCBvZiA1LDAwMCwwMDAgbG9zdAogICB0aGUgZW50aXJlIGJ1aWxkLgoKMy4gS2FnZ2xlLXBhdGggZGlzY292ZXJ5OiBgZGlzY292ZXJfZGF0YXNldF9yb290KClgIHdhbGtzCiAgIC9rYWdnbGUvaW5wdXQgZm9yIGEgZGlyZWN0b3J5IGNvbnRhaW5pbmcgYSB0cmFpbl9zb3VyY2UxLnRzdiwKICAgaW5zdGVhZCBvZiBhIGhhcmQtY29kZWQgbG9jYWwgcGF0aC4gRmFsbHMgYmFjayB0byBhIHBhdGggdGhlCiAgIGNhbGxlciBzdXBwbGllcywgc28gdGhpcyByZW1haW5zIHVzYWJsZSBvdXRzaWRlIEthZ2dsZSB0b28uCgpFdmVyeXRoaW5nIGVsc2UgKE1lbW9yeUluZGV4LCBzcGxpdF9lbnRpdHlfaWQsIGpvaW5fZW50aXR5X2lkLApjb21wb3NlX2tleSwgdGhlIGtleS1nZW5lcmF0aW9uIGNhbGwgc2l0ZXMpIGlzIFVOQ0hBTkdFRCAtLQp0aGVzZSBhcmUgd2hhdCB0aGUgbWVhc3VyZWQgcmVjYWxsIG51bWJlcnMgZGVwZW5kIG9uLgoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3FsaXRlMwpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gYmxvY2tpbmcgaW1wb3J0ICgKICAgIG1ha2VfbmFtZV9rZXlzLAogICAgbWFrZV91bmljb2RlX25hbWVfa2V5cywKICAgIG1ha2VfYWRkcmVzc19rZXlzLAogICAgbWFrZV9hNF9rZXlzLAogICAgbWFrZV9hdDJfa2V5cywKKQpmcm9tIGRhdGFfbG9hZGVyIGltcG9ydCByZWFkX3RzdgoKCmRlZiBzcGxpdF9lbnRpdHlfaWQoZW50aXR5X2lkKToKICAgIGlmICItIiBub3QgaW4gZW50aXR5X2lkOgogICAgICAgIHJldHVybiAiIiwgZW50aXR5X2lkCiAgICBwcmVmaXgsIHN1ZmZpeCA9IGVudGl0eV9pZC5zcGxpdCgiLSIsIDEpCiAgICBpZiBzdWZmaXguaXNkaWdpdCgpOgogICAgICAgIHJldHVybiBwcmVmaXgsIGludChzdWZmaXgpCiAgICByZXR1cm4gcHJlZml4LCBzdWZmaXgKCgpkZWYgam9pbl9lbnRpdHlfaWQocHJlZml4LCBzdWZmaXgpOgogICAgcmV0dXJuIGYie3ByZWZpeH0te3N1ZmZpeH0iCgoKZGVmIGNvbXBvc2Vfa2V5KGNvdW50cnksIGtleV92YWx1ZSk6CiAgICByZXR1cm4gKGNvdW50cnkgb3IgIiIpLnN0cmlwKCkudXBwZXIoKSArICJ8fCIgKyBrZXlfdmFsdWUKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIEtBR0dMRSBQQVRIIERJU0NPVkVSWQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIGRpc2NvdmVyX2RhdGFzZXRfcm9vdChyZXF1aXJlZF9maWxlcz0oInRyYWluX3NvdXJjZTEudHN2IiwpLCBzZWFyY2hfcm9vdD0iL2thZ2dsZS9pbnB1dCIpOgogICAgIiIiCiAgICBXYWxrIHNlYXJjaF9yb290IGxvb2tpbmcgZm9yIGEgZGlyZWN0b3J5IHRoYXQgY29udGFpbnMgZXZlcnkKICAgIGZpbGUgaW4gcmVxdWlyZWRfZmlsZXMuIFJldHVybnMgdGhlIGZpcnN0IG1hdGNoLCBvciBOb25lLgoKICAgIEF2b2lkcyBoYXJkLWNvZGluZyBhIHNwZWNpZmljIEthZ2dsZSBkYXRhc2V0IHNsdWcgKHRoZQogICAgbm90ZWJvb2sgc2tlbGV0b24gaW4gbm90ZWJvb2tjNjQ3ZDVkZWU1LmlweW5iIGhhcmQtY29kZXMKICAgICIuLi4vdGVqYXN3ZWVyYWpwdXQvYW1hem9uLW1sLWNoYWxsZW5nZS0yMDI2LXN0dWRlbnQtcmVzb3VyY2UvCiAgICBzdHVkZW50X3Jlc291cmNlIiAtLSBpZiB0aGUgZGF0YXNldCBpcyByZS1hdHRhY2hlZCB1bmRlciBhCiAgICBkaWZmZXJlbnQgc2x1Zywgb3IgYSBuZXcgdmVyc2lvbiwgb3IgdGhlIHBlcnNvbiBmb3JrcyB0aGUKICAgIG5vdGVib29rLCBldmVyeSBwYXRoIGJyZWFrcyBzaWxlbnRseSkuIFRoaXMgd2Fsa3MgdGhlIHRyZWUKICAgIG9uY2UgYXQgc3RhcnR1cCBpbnN0ZWFkLgogICAgIiIiCiAgICByb290ID0gUGF0aChzZWFyY2hfcm9vdCkKICAgIGlmIG5vdCByb290LmV4aXN0cygpOgogICAgICAgIHJldHVybiBOb25lCgogICAgZm9yIGRpcnBhdGgsIF9kaXJuYW1lcywgZmlsZW5hbWVzIGluIG9zLndhbGsocm9vdCk6CiAgICAgICAgZmlsZW5hbWVzID0gc2V0KGZpbGVuYW1lcykKICAgICAgICBpZiBhbGwoZiBpbiBmaWxlbmFtZXMgZm9yIGYgaW4gcmVxdWlyZWRfZmlsZXMpOgogICAgICAgICAgICByZXR1cm4gZGlycGF0aAoKICAgIHJldHVybiBOb25lCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBJTi1NRU1PUlkgQkFDS0VORCAodW5jaGFuZ2VkKQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKY2xhc3MgTWVtb3J5SW5kZXg6CgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNvdXJjZV9wcmVmaXgpOgogICAgICAgIHNlbGYuc291cmNlX3ByZWZpeCA9IHNvdXJjZV9wcmVmaXgKICAgICAgICBzZWxmLnBvc3RpbmdzID0gZGVmYXVsdGRpY3QobGFtYmRhOiBkZWZhdWx0ZGljdChsaXN0KSkKICAgICAgICBzZWxmLmF0dHJzID0ge30KICAgICAgICBzZWxmLmNvdW50ID0gMAoKICAgIGRlZiBhZGRfcmVjb3JkKHNlbGYsIGVudGl0eV9pZCwgbmFtZSwgYWRkcmVzcywgY291bnRyeSk6CgogICAgICAgIF8sIHN1ZmZpeCA9IHNwbGl0X2VudGl0eV9pZChlbnRpdHlfaWQpCiAgICAgICAgc2VsZi5hdHRyc1tzdWZmaXhdID0gKG5hbWUsIGFkZHJlc3MsIGNvdW50cnkpCgogICAgICAgIGZvciBrZXlfdHlwZSwga2V5X3ZhbHVlIGluIG1ha2VfbmFtZV9rZXlzKG5hbWUpICsgbWFrZV91bmljb2RlX25hbWVfa2V5cyhuYW1lKToKICAgICAgICAgICAgc2VsZi5wb3N0aW5nc1trZXlfdHlwZV1bY29tcG9zZV9rZXkoY291bnRyeSwga2V5X3ZhbHVlKV0uYXBwZW5kKHN1ZmZpeCkKCiAgICAgICAgZm9yIGtleV90eXBlLCBrZXlfdmFsdWUgaW4gbWFrZV9hZGRyZXNzX2tleXMoYWRkcmVzcyk6CiAgICAgICAgICAgIHNlbGYucG9zdGluZ3Nba2V5X3R5cGVdW2NvbXBvc2Vfa2V5KGNvdW50cnksIGtleV92YWx1ZSldLmFwcGVuZChzdWZmaXgpCgogICAgICAgIGZvciBrZXlfdHlwZSwga2V5X3ZhbHVlIGluIG1ha2VfYTRfa2V5cyhhZGRyZXNzKSArIG1ha2VfYXQyX2tleXMoYWRkcmVzcyk6CiAgICAgICAgICAgIHNlbGYucG9zdGluZ3Nba2V5X3R5cGVdW2NvbXBvc2Vfa2V5KGNvdW50cnksIGtleV92YWx1ZSldLmFwcGVuZChzdWZmaXgpCgogICAgICAgIHNlbGYuY291bnQgKz0gMQoKICAgIGRlZiBidWNrZXQoc2VsZiwga2V5X3R5cGUsIGtleV92YWx1ZSwgbGltaXQ9Tm9uZSk6CiAgICAgICAgcmVzdWx0ID0gc2VsZi5wb3N0aW5ncy5nZXQoa2V5X3R5cGUsIHt9KS5nZXQoa2V5X3ZhbHVlLCBbXSkKICAgICAgICBpZiBsaW1pdCBpcyBub3QgTm9uZToKICAgICAgICAgICAgIyBBbHJlYWR5IHJlc2lkZW50IGluIFJBTSBlaXRoZXIgd2F5OyBzbGljaW5nIGp1c3QKICAgICAgICAgICAgIyBhdm9pZHMgaGFuZGluZyBhbiBvdmVyc2l6ZWQgbGlzdCB0byB0aGUgY2FsbGVyLgogICAgICAgICAgICByZXR1cm4gcmVzdWx0WzpsaW1pdF0KICAgICAgICByZXR1cm4gcmVzdWx0CgogICAgZGVmIGdldF9hdHRycyhzZWxmLCBzdWZmaXgpOgogICAgICAgIHJldHVybiBzZWxmLmF0dHJzLmdldChzdWZmaXgpCgogICAgZGVmIG51bV9rZXlzKHNlbGYpOgogICAgICAgIHJldHVybiBzdW0obGVuKHYpIGZvciB2IGluIHNlbGYucG9zdGluZ3MudmFsdWVzKCkpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBTUUxJVEUgKERJU0stQkFDS0VEKSBCQUNLRU5EIC0tIHdpdGggY2FwcGVkIGZldGNoICsgY2hlY2twb2ludAojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKY2xhc3MgU3FsaXRlSW5kZXg6CgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNvdXJjZV9wcmVmaXgsIGRiX3BhdGgsIG1hbmlmZXN0X3BhdGg9Tm9uZSk6CgogICAgICAgIHNlbGYuc291cmNlX3ByZWZpeCA9IHNvdXJjZV9wcmVmaXgKICAgICAgICBzZWxmLmRiX3BhdGggPSBkYl9wYXRoCiAgICAgICAgc2VsZi5tYW5pZmVzdF9wYXRoID0gbWFuaWZlc3RfcGF0aCBvciAoc3RyKGRiX3BhdGgpICsgIi5tYW5pZmVzdC5qc29uIikKICAgICAgICBzZWxmLmNvdW50ID0gMAoKICAgICAgICBzZWxmLmNvbm4gPSBzcWxpdGUzLmNvbm5lY3QoZGJfcGF0aCkKICAgICAgICBzZWxmLmNvbm4uZXhlY3V0ZSgiUFJBR01BIHN5bmNocm9ub3VzID0gT0ZGIikKICAgICAgICBzZWxmLmNvbm4uZXhlY3V0ZSgiUFJBR01BIGpvdXJuYWxfbW9kZSA9IFdBTCIpICAjIHN1cnZpdmVzIGEgY3Jhc2ggbWlkLXdyaXRlCgogICAgICAgIHNlbGYuY29ubi5leGVjdXRlKAogICAgICAgICAgICAiQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgcG9zdGluZ3MgIgogICAgICAgICAgICAiKGtleV90eXBlIFRFWFQsIGtleV92YWx1ZSBURVhULCBzdWZmaXggVEVYVCkiCiAgICAgICAgKQogICAgICAgIHNlbGYuY29ubi5leGVjdXRlKAogICAgICAgICAgICAiQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgYXR0cnMgIgogICAgICAgICAgICAiKHN1ZmZpeCBURVhUIFBSSU1BUlkgS0VZLCBuYW1lIFRFWFQsIGFkZHJlc3MgVEVYVCwgY291bnRyeSBURVhUKSIKICAgICAgICApCiAgICAgICAgc2VsZi5jb25uLmNvbW1pdCgpCgogICAgICAgIHNlbGYuX2J1ZmZlciA9IFtdCiAgICAgICAgc2VsZi5fYXR0cl9idWZmZXIgPSBbXQogICAgICAgIHNlbGYuX2luZGV4X2J1aWx0ID0gRmFsc2UKCiAgICAjIC0tLS0tLS0tIG1hbmlmZXN0IC8gcmVzdW1lIC0tLS0tLS0tCgogICAgZGVmIF9yZWFkX21hbmlmZXN0KHNlbGYpOgogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKHNlbGYubWFuaWZlc3RfcGF0aCk6CiAgICAgICAgICAgIHdpdGggb3BlbihzZWxmLm1hbmlmZXN0X3BhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgIHJldHVybiBqc29uLmxvYWQoZikKICAgICAgICByZXR1cm4gTm9uZQoKICAgIGRlZiBfd3JpdGVfbWFuaWZlc3Qoc2VsZiwgcm93c19pbmRleGVkLCBjb21wbGV0ZSk6CiAgICAgICAgbWFuaWZlc3QgPSB7CiAgICAgICAgICAgICJzb3VyY2VfcHJlZml4Ijogc2VsZi5zb3VyY2VfcHJlZml4LAogICAgICAgICAgICAicm93c19pbmRleGVkIjogcm93c19pbmRleGVkLAogICAgICAgICAgICAiY29tcGxldGUiOiBjb21wbGV0ZSwKICAgICAgICB9CiAgICAgICAgdG1wX3BhdGggPSBzZWxmLm1hbmlmZXN0X3BhdGggKyAiLnRtcCIKICAgICAgICB3aXRoIG9wZW4odG1wX3BhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAganNvbi5kdW1wKG1hbmlmZXN0LCBmKQogICAgICAgIG9zLnJlcGxhY2UodG1wX3BhdGgsIHNlbGYubWFuaWZlc3RfcGF0aCkKCiAgICBkZWYgcmVzdW1lX3BvaW50KHNlbGYpOgogICAgICAgICIiIgogICAgICAgIFJldHVybnMgKHJvd3NfYWxyZWFkeV9pbmRleGVkLCBhbHJlYWR5X2NvbXBsZXRlKS4gVGhlCiAgICAgICAgY2FsbGVyIChidWlsZF9pbmRleCkgdXNlcyB0aGlzIHRvIGRlY2lkZSBob3cgbWFueSBpbnB1dAogICAgICAgIHJvd3MgdG8gc2tpcCwgYW5kIHdoZXRoZXIgdG8gc2tpcCBidWlsZGluZyB0aGlzIHNvdXJjZSdzCiAgICAgICAgaW5kZXggYXQgYWxsLgogICAgICAgICIiIgogICAgICAgIG1hbmlmZXN0ID0gc2VsZi5fcmVhZF9tYW5pZmVzdCgpCiAgICAgICAgaWYgbWFuaWZlc3QgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIDAsIEZhbHNlCiAgICAgICAgcmV0dXJuIG1hbmlmZXN0LmdldCgicm93c19pbmRleGVkIiwgMCksIG1hbmlmZXN0LmdldCgiY29tcGxldGUiLCBGYWxzZSkKCiAgICAjIC0tLS0tLS0tIGluZ2VzdCAtLS0tLS0tLQoKICAgIGRlZiBhZGRfcmVjb3JkKHNlbGYsIGVudGl0eV9pZCwgbmFtZSwgYWRkcmVzcywgY291bnRyeSk6CgogICAgICAgIF8sIHN1ZmZpeCA9IHNwbGl0X2VudGl0eV9pZChlbnRpdHlfaWQpCiAgICAgICAgc3VmZml4ID0gc3RyKHN1ZmZpeCkKCiAgICAgICAgc2VsZi5fYXR0cl9idWZmZXIuYXBwZW5kKChzdWZmaXgsIG5hbWUsIGFkZHJlc3MsIGNvdW50cnkpKQoKICAgICAgICBmb3Iga2V5X3R5cGUsIGtleV92YWx1ZSBpbiBtYWtlX25hbWVfa2V5cyhuYW1lKSArIG1ha2VfdW5pY29kZV9uYW1lX2tleXMobmFtZSk6CiAgICAgICAgICAgIHNlbGYuX2J1ZmZlci5hcHBlbmQoKGtleV90eXBlLCBjb21wb3NlX2tleShjb3VudHJ5LCBrZXlfdmFsdWUpLCBzdWZmaXgpKQoKICAgICAgICBmb3Iga2V5X3R5cGUsIGtleV92YWx1ZSBpbiBtYWtlX2FkZHJlc3Nfa2V5cyhhZGRyZXNzKToKICAgICAgICAgICAgc2VsZi5fYnVmZmVyLmFwcGVuZCgoa2V5X3R5cGUsIGNvbXBvc2Vfa2V5KGNvdW50cnksIGtleV92YWx1ZSksIHN1ZmZpeCkpCgogICAgICAgIGZvciBrZXlfdHlwZSwga2V5X3ZhbHVlIGluIG1ha2VfYTRfa2V5cyhhZGRyZXNzKSArIG1ha2VfYXQyX2tleXMoYWRkcmVzcyk6CiAgICAgICAgICAgIHNlbGYuX2J1ZmZlci5hcHBlbmQoKGtleV90eXBlLCBjb21wb3NlX2tleShjb3VudHJ5LCBrZXlfdmFsdWUpLCBzdWZmaXgpKQoKICAgICAgICBzZWxmLmNvdW50ICs9IDEKCiAgICAgICAgaWYgbGVuKHNlbGYuX2J1ZmZlcikgPj0gMjAwXzAwMDoKICAgICAgICAgICAgc2VsZi5fZmx1c2goKQoKICAgIGRlZiBfZmx1c2goc2VsZik6CgogICAgICAgIGlmIHNlbGYuX2J1ZmZlcjoKICAgICAgICAgICAgc2VsZi5jb25uLmV4ZWN1dGVtYW55KCJJTlNFUlQgSU5UTyBwb3N0aW5ncyBWQUxVRVMgKD8sPyw/KSIsIHNlbGYuX2J1ZmZlcikKICAgICAgICAgICAgc2VsZi5fYnVmZmVyID0gW10KCiAgICAgICAgaWYgc2VsZi5fYXR0cl9idWZmZXI6CiAgICAgICAgICAgIHNlbGYuY29ubi5leGVjdXRlbWFueSgKICAgICAgICAgICAgICAgICJJTlNFUlQgT1IgUkVQTEFDRSBJTlRPIGF0dHJzIFZBTFVFUyAoPyw/LD8sPykiLCBzZWxmLl9hdHRyX2J1ZmZlcgogICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGYuX2F0dHJfYnVmZmVyID0gW10KCiAgICAgICAgIyBDb21taXQgb24gZXZlcnkgZmx1c2ggKG5vdCBqdXN0IGF0IGZpbmFsaXplKCkpIHNvIGEga2lsbAogICAgICAgICMgYmV0d2VlbiBmbHVzaGVzIGxvc2VzIGF0IG1vc3Qgb25lIGJ1ZmZlcidzIHdvcnRoIG9mIHJvd3MsCiAgICAgICAgIyBhbmQgcmVjb3JkIGhvdyBmYXIgd2UgZ290IHNvIGEgcmVzdGFydCBrbm93cyB3aGVyZSB0bwogICAgICAgICMgcmVzdW1lIGZyb20uCiAgICAgICAgc2VsZi5jb25uLmNvbW1pdCgpCiAgICAgICAgc2VsZi5fd3JpdGVfbWFuaWZlc3Qocm93c19pbmRleGVkPXNlbGYuY291bnQsIGNvbXBsZXRlPUZhbHNlKQoKICAgIGRlZiBmaW5hbGl6ZShzZWxmKToKCiAgICAgICAgc2VsZi5fZmx1c2goKQoKICAgICAgICBpZiBub3Qgc2VsZi5faW5kZXhfYnVpbHQ6CiAgICAgICAgICAgIHNlbGYuY29ubi5leGVjdXRlKAogICAgICAgICAgICAgICAgIkNSRUFURSBJTkRFWCBJRiBOT1QgRVhJU1RTIGlkeF9wb3N0aW5ncyAiCiAgICAgICAgICAgICAgICAiT04gcG9zdGluZ3Moa2V5X3R5cGUsIGtleV92YWx1ZSkiCiAgICAgICAgICAgICkKICAgICAgICAgICAgc2VsZi5jb25uLmNvbW1pdCgpCiAgICAgICAgICAgIHNlbGYuX2luZGV4X2J1aWx0ID0gVHJ1ZQoKICAgICAgICBzZWxmLl93cml0ZV9tYW5pZmVzdChyb3dzX2luZGV4ZWQ9c2VsZi5jb3VudCwgY29tcGxldGU9VHJ1ZSkKCiAgICAjIC0tLS0tLS0tIHF1ZXJ5IC0tLS0tLS0tCgogICAgZGVmIGJ1Y2tldChzZWxmLCBrZXlfdHlwZSwga2V5X3ZhbHVlLCBsaW1pdD1Ob25lKToKICAgICAgICBpZiBsaW1pdCBpcyBub3QgTm9uZToKICAgICAgICAgICAgY3VyID0gc2VsZi5jb25uLmV4ZWN1dGUoCiAgICAgICAgICAgICAgICAiU0VMRUNUIHN1ZmZpeCBGUk9NIHBvc3RpbmdzIFdIRVJFIGtleV90eXBlPT8gQU5EIGtleV92YWx1ZT0/IExJTUlUID8iLAogICAgICAgICAgICAgICAgKGtleV90eXBlLCBrZXlfdmFsdWUsIGxpbWl0KSwKICAgICAgICAgICAgKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGN1ciA9IHNlbGYuY29ubi5leGVjdXRlKAogICAgICAgICAgICAgICAgIlNFTEVDVCBzdWZmaXggRlJPTSBwb3N0aW5ncyBXSEVSRSBrZXlfdHlwZT0/IEFORCBrZXlfdmFsdWU9PyIsCiAgICAgICAgICAgICAgICAoa2V5X3R5cGUsIGtleV92YWx1ZSksCiAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gW3Jvd1swXSBmb3Igcm93IGluIGN1ci5mZXRjaGFsbCgpXQoKICAgIGRlZiBnZXRfYXR0cnMoc2VsZiwgc3VmZml4KToKICAgICAgICBjdXIgPSBzZWxmLmNvbm4uZXhlY3V0ZSgKICAgICAgICAgICAgIlNFTEVDVCBuYW1lLCBhZGRyZXNzLCBjb3VudHJ5IEZST00gYXR0cnMgV0hFUkUgc3VmZml4PT8iLAogICAgICAgICAgICAoc3RyKHN1ZmZpeCksKSwKICAgICAgICApCiAgICAgICAgcmV0dXJuIGN1ci5mZXRjaG9uZSgpCgogICAgZGVmIG51bV9rZXlzKHNlbGYpOgogICAgICAgIGN1ciA9IHNlbGYuY29ubi5leGVjdXRlKAogICAgICAgICAgICAiU0VMRUNUIENPVU5UKCopIEZST00gKFNFTEVDVCBESVNUSU5DVCBrZXlfdHlwZSwga2V5X3ZhbHVlIEZST00gcG9zdGluZ3MpIgogICAgICAgICkKICAgICAgICByZXR1cm4gY3VyLmZldGNob25lKClbMF0KCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIEJVSUxEICh3aXRoIHJlc3VtZSkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBidWlsZF9pbmRleChwYXRoLCBzb3VyY2VfcHJlZml4LCBsaW1pdD1Ob25lLCBiYWNrZW5kPSJtZW1vcnkiLCBkYl9wYXRoPU5vbmUsCiAgICAgICAgICAgICAgICAgbWFuaWZlc3RfcGF0aD1Ob25lKToKICAgICIiIgogICAgU2FtZSBzaWduYXR1cmUgYW5kIGJlaGF2aW9yIGFzIGJlZm9yZSBmb3IgYmFja2VuZD0ibWVtb3J5IgogICAgKG5vIHBlcnNpc3RlbmNlIGlzIHBvc3NpYmxlIGZvciBhbiBpbi1SQU0gc3RydWN0dXJlIGFjcm9zcyBhCiAgICBrZXJuZWwgcmVzdGFydCAtLSBpZiB0aGUgcnVuIGRpZXMsIGEgbWVtb3J5LWJhY2tlbmQgaW5kZXggbXVzdAogICAgYmUgcmVidWlsdCBmcm9tIHNjcmF0Y2g7IHVzZSBiYWNrZW5kPSJzcWxpdGUiIGZvciBhbnl0aGluZwogICAgdGhhdCBuZWVkcyB0byBzdXJ2aXZlIGEgS2FnZ2xlIGludGVycnVwdGlvbiwgcGVyIHRoZQogICAgY2hlY2twb2ludC9yZXN1bWUgcGxhbikuCgogICAgYmFja2VuZD0ic3FsaXRlIiBub3cgcmVzdW1lczogaWYgZGJfcGF0aCArIGl0cyBtYW5pZmVzdCBzaG93IGEKICAgIHByZXZpb3VzbHkgQ09NUExFVEUgYnVpbGQgZm9yIHRoaXMgc291cmNlLCB0aGUgZXhpc3RpbmcKICAgIGRhdGFiYXNlIGlzIHJlb3BlbmVkIGFuZCByZXR1cm5lZCBpbW1lZGlhdGVseSAobm8gcmUtcmVhZCBvZgogICAgYHBhdGhgIGF0IGFsbCkuIElmIHRoZSBtYW5pZmVzdCBzaG93cyBhIHBhcnRpYWwgYnVpbGQsIG9ubHkKICAgIHRoZSByZW1haW5pbmcgcm93cyBvZiBgcGF0aGAgYXJlIHJlYWQgYW5kIGFwcGVuZGVkLgogICAgIiIiCgogICAgaWYgYmFja2VuZCA9PSAic3FsaXRlIjoKICAgICAgICBpZiBkYl9wYXRoIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImRiX3BhdGggaXMgcmVxdWlyZWQgZm9yIHRoZSBzcWxpdGUgYmFja2VuZCIpCgogICAgICAgIGluZGV4ID0gU3FsaXRlSW5kZXgoc291cmNlX3ByZWZpeCwgZGJfcGF0aCwgbWFuaWZlc3RfcGF0aD1tYW5pZmVzdF9wYXRoKQogICAgICAgIHJvd3NfZG9uZSwgY29tcGxldGUgPSBpbmRleC5yZXN1bWVfcG9pbnQoKQoKICAgICAgICBpZiBjb21wbGV0ZToKICAgICAgICAgICAgcHJpbnQoCiAgICAgICAgICAgICAgICBmIltpbmRleGVyXSB7c291cmNlX3ByZWZpeH0gaW5kZXggYXQge2RiX3BhdGh9IGlzIGFscmVhZHkgIgogICAgICAgICAgICAgICAgZiJjb21wbGV0ZSAoe3Jvd3NfZG9uZTosfSByb3dzKSAtLSByZXVzaW5nLCBubyByZWJ1aWxkLiIKICAgICAgICAgICAgKQogICAgICAgICAgICBpbmRleC5jb3VudCA9IHJvd3NfZG9uZQogICAgICAgICAgICByZXR1cm4gaW5kZXgKCiAgICAgICAgaWYgcm93c19kb25lOgogICAgICAgICAgICBwcmludCgKICAgICAgICAgICAgICAgIGYiW2luZGV4ZXJdIFJlc3VtaW5nIHtzb3VyY2VfcHJlZml4fSBpbmRleCBhdCB7ZGJfcGF0aH06ICIKICAgICAgICAgICAgICAgIGYie3Jvd3NfZG9uZTosfSByb3dzIGFscmVhZHkgaW5kZXhlZCwgY29udGludWluZyBmcm9tIHRoZXJlLiIKICAgICAgICAgICAgKQogICAgICAgIGluZGV4LmNvdW50ID0gcm93c19kb25lCgogICAgZWxzZToKICAgICAgICBpbmRleCA9IE1lbW9yeUluZGV4KHNvdXJjZV9wcmVmaXgpCiAgICAgICAgcm93c19kb25lID0gMAoKICAgIHByaW50KGYiW2luZGV4ZXJdIEJ1aWxkaW5nIHtiYWNrZW5kfSBpbmRleCBmb3Ige3NvdXJjZV9wcmVmaXh9IGZyb206IHtwYXRofSIpCgogICAgZm9yIGksIHJvdyBpbiBlbnVtZXJhdGUocmVhZF90c3YocGF0aCkpOgoKICAgICAgICBpZiBpIDwgcm93c19kb25lOgogICAgICAgICAgICBjb250aW51ZSAgIyBhbHJlYWR5IGluZGV4ZWQgaW4gYSBwcmV2aW91cywgaW50ZXJydXB0ZWQgcnVuCiAgICAgICAgICAgICMgTk9URTogdGhpcyBzdGlsbCBwYXJzZXMgZXZlcnkgc2tpcHBlZCByb3cgKHJlYWRfdHN2IGlzIGEKICAgICAgICAgICAgIyBwbGFpbiBjc3YuRGljdFJlYWRlciBnZW5lcmF0b3IpIC0tIHJlc3VtZSBhdm9pZHMgcmVkb2luZwogICAgICAgICAgICAjIGtleSBnZW5lcmF0aW9uICsgaW5zZXJ0cywgd2hpY2ggZG9taW5hdGUgY29zdCwgYnV0IG5vdAogICAgICAgICAgICAjIHRoZSBjb21wYXJhdGl2ZWx5IGNoZWFwIFRTViBwYXJzZS4gQWNjZXB0YWJsZSB0cmFkZS1vZmYKICAgICAgICAgICAgIyBmb3IgY29ycmVjdG5lc3Mvc2ltcGxpY2l0eTsgcmV2aXNpdCBvbmx5IGlmIHByb2ZpbGluZwogICAgICAgICAgICAjIHNob3dzIHRoZSBwYXJzZSBpdHNlbGYgaXMgc2lnbmlmaWNhbnQuCgogICAgICAgIGlmIGxpbWl0IGlzIG5vdCBOb25lIGFuZCBpID49IGxpbWl0OgogICAgICAgICAgICBicmVhawoKICAgICAgICBpbmRleC5hZGRfcmVjb3JkKAogICAgICAgICAgICBlbnRpdHlfaWQ9cm93LmdldCgiZW50aXR5X2lkIiwgIiIpLAogICAgICAgICAgICBuYW1lPXJvdy5nZXQoImJ1c2luZXNzX25hbWUiLCAiIiksCiAgICAgICAgICAgIGFkZHJlc3M9cm93LmdldCgiYnVzaW5lc3NfYWRkcmVzcyIsICIiKSwKICAgICAgICAgICAgY291bnRyeT1yb3cuZ2V0KCJjb3VudHJ5IiwgIiIpLnN0cmlwKCksCiAgICAgICAgKQoKICAgICAgICBpZiBpbmRleC5jb3VudCAlIDUwMF8wMDAgPT0gMDoKICAgICAgICAgICAgcHJpbnQoZiJbaW5kZXhlcl0gICBpbmRleGVkIHtpbmRleC5jb3VudDosfSB7c291cmNlX3ByZWZpeH0gcmVjb3Jkcy4uLiIpCgogICAgaWYgYmFja2VuZCA9PSAic3FsaXRlIjoKICAgICAgICBpbmRleC5maW5hbGl6ZSgpCgogICAgcHJpbnQoCiAgICAgICAgZiJbaW5kZXhlcl0gRmluaXNoZWQge3NvdXJjZV9wcmVmaXh9OiB7aW5kZXguY291bnQ6LH0gcmVjb3JkcywgIgogICAgICAgIGYie2luZGV4Lm51bV9rZXlzKCk6LH0gZGlzdGluY3QgYmxvY2tpbmcga2V5cy4iCiAgICApCgogICAgcmV0dXJuIGluZGV4Cg==",
    "candidates.py": "IiIiCmNhbmRpZGF0ZXMucHkKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkNIQU5HRSBmcm9tIHRoZSBvcmlnaW5hbDogaW5kZXguYnVja2V0KGtleV90eXBlLCBjb21wb3NlZCkgaXMgbm93CmNhbGxlZCBhcyBpbmRleC5idWNrZXQoa2V5X3R5cGUsIGNvbXBvc2VkLCBsaW1pdD1jYXAgKyAxKS4KCldoeSBjYXAgKyAxIGFuZCBub3QgY2FwOiB3ZSBzdGlsbCBuZWVkIHRvIGRpc3Rpbmd1aXNoICJidWNrZXQgc2l6ZQo9PSBjYXAgZXhhY3RseSIgKGtlZXAgaXQpIGZyb20gImJ1Y2tldCBzaXplID4gY2FwIiAoZHJvcCBpdCksIHNvIHdlCmFzayBmb3Igb25lIG1vcmUgcm93IHRoYW4gdGhlIGNhcCBhbGxvd3MuIEZvciBNZW1vcnlJbmRleCB0aGlzIGlzCmZyZWUgKGFscmVhZHkgaW4gUkFNKS4gRm9yIFNxbGl0ZUluZGV4IHRoaXMgdHVybnMgYSBwb3RlbnRpYWxseQpodWdlLCB3YXN0ZWQgYFNFTEVDVCAuLi4gYCArIGZldGNoYWxsKCkgb24gYSBob3Qga2V5IGludG8gYSBib3VuZGVkCmBTRUxFQ1QgLi4uIExJTUlUIGNhcCsxYCAtLSBzZWVTZWUgaW5kZXhlci5weSdzIG1vZHVsZSBkb2NzdHJpbmcKZm9yIHRoZSBmdWxsIHJlYXNvbmluZyBhbmQgd2h5IHRoaXMgbWF0dGVycyBzcGVjaWZpY2FsbHkgYXQgZnVsbAooNU0rIHJvdykgcHJvZHVjdGlvbiBzY2FsZSwgbm90IGF0IHRoZSAxMDBLLXJvdyBkZXYgc2NhbGUgdGhlCm9yaWdpbmFsIG51bWJlcnMgd2VyZSBtZWFzdXJlZCBvbi4KCkV2ZXJ5dGhpbmcgZWxzZSBpbiB0aGlzIGZpbGUgKHRoZSBjYW5kaWRhdGUgZGljdCBzaGFwZSwga2V5X3R5cGVzLwptaW5fZnJlcSBib29ra2VlcGluZywgZ2VuZXJhdGVfY2FuZGlkYXRlX3BhaXJzX2Zvcl9yb3cpIGlzClVOQ0hBTkdFRC4KIiIiCgpmcm9tIGJsb2NraW5nIGltcG9ydCAoCiAgICBtYWtlX25hbWVfa2V5cywKICAgIG1ha2VfdW5pY29kZV9uYW1lX2tleXMsCiAgICBtYWtlX2FkZHJlc3Nfa2V5cywKICAgIG1ha2VfYTRfa2V5cywKICAgIG1ha2VfYXQyX2tleXMsCiAgICBERUZBVUxUX0ZSRVFfQ0FQUywKKQpmcm9tIGluZGV4ZXIgaW1wb3J0IGNvbXBvc2Vfa2V5CgoKZGVmIGZpbmRfY2FuZGlkYXRlcyhzMV9yb3csIGluZGV4LCBmcmVxX2NhcHM9Tm9uZSk6CgogICAgZnJlcV9jYXBzID0gZnJlcV9jYXBzIG9yIERFRkFVTFRfRlJFUV9DQVBTCgogICAgY291bnRyeSA9IHMxX3Jvdy5nZXQoImNvdW50cnkiLCAiIikuc3RyaXAoKQogICAgbmFtZSA9IHMxX3Jvdy5nZXQoImJ1c2luZXNzX25hbWUiLCAiIikKICAgIGFkZHJlc3MgPSBzMV9yb3cuZ2V0KCJidXNpbmVzc19hZGRyZXNzIiwgIiIpCgogICAgYWxsX2tleXMgPSAoCiAgICAgICAgbWFrZV9uYW1lX2tleXMobmFtZSkKICAgICAgICArIG1ha2VfdW5pY29kZV9uYW1lX2tleXMobmFtZSkKICAgICAgICArIG1ha2VfYWRkcmVzc19rZXlzKGFkZHJlc3MpCiAgICAgICAgKyBtYWtlX2E0X2tleXMoYWRkcmVzcykKICAgICAgICArIG1ha2VfYXQyX2tleXMoYWRkcmVzcykKICAgICkKCiAgICBjYW5kaWRhdGVzID0ge30KCiAgICBmb3Iga2V5X3R5cGUsIGtleV92YWx1ZSBpbiBhbGxfa2V5czoKCiAgICAgICAgY2FwID0gZnJlcV9jYXBzLmdldChrZXlfdHlwZSwgREVGQVVMVF9GUkVRX0NBUFMuZ2V0KGtleV90eXBlLCA1MDApKQoKICAgICAgICBjb21wb3NlZCA9IGNvbXBvc2Vfa2V5KGNvdW50cnksIGtleV92YWx1ZSkKCiAgICAgICAgIyBBc2sgZm9yIGF0IG1vc3QgY2FwICsgMSByb3dzLiBJZiB3ZSBnZXQgYmFjayBjYXAgKyAxLCB0aGUKICAgICAgICAjIHRydWUgYnVja2V0IGlzID4gY2FwIHNvbWV3aGVyZSBhdCBvciBiZXlvbmQgdGhhdCByb3csIHNvCiAgICAgICAgIyBpdCdzIHN0aWxsIGNvcnJlY3RseSB0cmVhdGVkIGFzICJ0b28gYnJvYWQiIGJlbG93IC0tIHdlCiAgICAgICAgIyBqdXN0IG5ldmVyIGhhZCB0byBtYXRlcmlhbGl6ZSB0aGUgcmVzdCBvZiBpdC4KICAgICAgICBidWNrZXQgPSBpbmRleC5idWNrZXQoa2V5X3R5cGUsIGNvbXBvc2VkLCBsaW1pdD1jYXAgKyAxKQogICAgICAgIGJ1Y2tldF9zaXplID0gbGVuKGJ1Y2tldCkKCiAgICAgICAgaWYgYnVja2V0X3NpemUgPT0gMDoKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgaWYgYnVja2V0X3NpemUgPiBjYXA6CiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGZvciBzdWZmaXggaW4gYnVja2V0OgoKICAgICAgICAgICAgZW50cnkgPSBjYW5kaWRhdGVzLmdldChzdWZmaXgpCgogICAgICAgICAgICBpZiBlbnRyeSBpcyBOb25lOgogICAgICAgICAgICAgICAgY2FuZGlkYXRlc1tzdWZmaXhdID0gewogICAgICAgICAgICAgICAgICAgICJrZXlfdHlwZXMiOiB7a2V5X3R5cGV9LAogICAgICAgICAgICAgICAgICAgICJtaW5fZnJlcSI6IGJ1Y2tldF9zaXplLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgZW50cnlbImtleV90eXBlcyJdLmFkZChrZXlfdHlwZSkKICAgICAgICAgICAgICAgIGVudHJ5WyJtaW5fZnJlcSJdID0gbWluKGVudHJ5WyJtaW5fZnJlcSJdLCBidWNrZXRfc2l6ZSkKCiAgICByZXR1cm4gY2FuZGlkYXRlcwoKCmRlZiBnZW5lcmF0ZV9jYW5kaWRhdGVfcGFpcnNfZm9yX3JvdyhzMV9yb3csIGluZGV4ZXNfYnlfc291cmNlLCBmcmVxX2NhcHM9Tm9uZSk6CgogICAgczFfaWQgPSBzMV9yb3cuZ2V0KCJlbnRpdHlfaWQiLCAiIikKICAgIHJvd3MgPSBbXQoKICAgIGZvciBzb3VyY2UsIGluZGV4IGluIGluZGV4ZXNfYnlfc291cmNlLml0ZW1zKCk6CgogICAgICAgIGNhbmRpZGF0ZXMgPSBmaW5kX2NhbmRpZGF0ZXMoczFfcm93LCBpbmRleCwgZnJlcV9jYXBzPWZyZXFfY2FwcykKCiAgICAgICAgZm9yIHN1ZmZpeCwgaW5mbyBpbiBjYW5kaWRhdGVzLml0ZW1zKCk6CgogICAgICAgICAgICBjYW5kaWRhdGVfaWQgPSBmIntzb3VyY2V9LXtzdWZmaXh9IgoKICAgICAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgInNvdXJjZTFfZW50aXR5X2lkIjogczFfaWQsCiAgICAgICAgICAgICAgICAgICAgImNhbmRpZGF0ZV9lbnRpdHlfaWQiOiBjYW5kaWRhdGVfaWQsCiAgICAgICAgICAgICAgICAgICAgInNvdXJjZSI6IHNvdXJjZSwKICAgICAgICAgICAgICAgICAgICAiYmxvY2tpbmdfa2V5X3R5cGVzIjogInwiLmpvaW4oc29ydGVkKGluZm9bImtleV90eXBlcyJdKSksCiAgICAgICAgICAgICAgICAgICAgImJsb2NraW5nX251bV9rZXlzIjogbGVuKGluZm9bImtleV90eXBlcyJdKSwKICAgICAgICAgICAgICAgICAgICAiYmxvY2tpbmdfbWluX2ZyZXEiOiBpbmZvWyJtaW5fZnJlcSJdLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCgogICAgcmV0dXJuIHJvd3MK",
    "features.py": "IiIiCmZlYXR1cmVzLnB5Cj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpDSEFOR0VTIGZyb20gdGhlIG9yaWdpbmFsIChzZWUgUEhBU0UxX0FSQ0hJVEVDVFVSRV9BVURJVC5tZCwKc2VjdGlvbnMgIkV4aXN0aW5nIGZlYXR1cmUgc2V0IiBhbmQgIlJ1bnRpbWUgb3B0aW1pemF0aW9uIHBsYW4iCmZvciB0aGUgbWVhc3VyZW1lbnRzIGJlaGluZCB0aGVzZSBjaGFuZ2VzKToKCjEuIE5PTi1MQVRJTiBGSVggKGNvcnJlY3RuZXNzLCBub3QganVzdCBzcGVlZCk6CiAgIG5hbWVfY2hhcl9zaW1pbGFyaXR5IC8gbmFtZV9sZXZlbnNodGVpbl9zaW1pbGFyaXR5IC8gZXRjLiB1c2VkCiAgIHRvIGJlIGNvbXB1dGVkIE9OTFkgb24gbm9ybWFsaXplX2J1c2luZXNzX25hbWUoKS9ub3JtYWxpemVfYWRkcmVzcygpLAogICB3aGljaCBzdHJpcCBub24tTGF0aW4gc2NyaXB0cyB0byAiIi4gRm9yIGEgdHJ1ZS1tYXRjaCBwYWlyIHdoZXJlCiAgIGJvdGggc2lkZXMgYXJlIGluIERldmFuYWdhcmkvVGVsdWd1L0thbm5hZGEvQmVuZ2FsaSwgZXZlcnkKICAgc2ltaWxhcml0eSBmZWF0dXJlIHNpbGVudGx5IGJlY2FtZSAwIG9yIHRoZSAibWlzc2luZyIgZGVmYXVsdCwKICAgZXZlbiB0aG91Z2ggdGhlIHN0cmluZ3MgYXJlIGdlbnVpbmVseSB2ZXJ5IHNpbWlsYXIuIFRoaXMgbW9kdWxlCiAgIG5vdyBhbHNvIGNvbXB1dGVzIHRoZSBzYW1lIGZlYXR1cmUgc3VpdGUgb24gYSBzY3JpcHQtYWdub3N0aWMKICAgbm9ybWFsaXphdGlvbiAobm9ybWFsaXplLm5vcm1hbGl6ZV91bmljb2RlX3ByZXNlcnZpbmcpIGFuZCB1c2VzCiAgIHdoaWNoZXZlciBub3JtYWxpemF0aW9uIGFjdHVhbGx5IGhhcyBjb250ZW50LCBzbyB0aGUgTUwgbW9kZWwKICAgZ2V0cyByZWFsIHNpZ25hbCBvbiBub24tTGF0aW4gcm93cyBpbnN0ZWFkIG9mIGEgY29uc3RhbnQuCgoyLiBTUEVFRDogdGhlIG9yaWdpbmFsIGhhZCBUV08gaW5kZXBlbmRlbnQgaW1wbGVtZW50YXRpb25zIG9mCiAgICJlZGl0LWRpc3RhbmNlLWJhc2VkIHJhdGlvIiBwZXIgZmllbGQgKHRleHRfc2ltaWxhcml0eSgpIHZpYQogICBkaWZmbGliLlNlcXVlbmNlTWF0Y2hlciwgYW5kIGxldmVuc2h0ZWluX3NpbWlsYXJpdHkoKSB2aWEKICAgcmFwaWRmdXp6KS4gZGlmZmxpYidzIFNlcXVlbmNlTWF0Y2hlciBpcyBwdXJlIFB5dGhvbiBhbmQgaXMgdGhlCiAgIHNpbmdsZSBtb3N0IGV4cGVuc2l2ZSBjYWxsIGluIHRoZSBvbGQgZmVhdHVyZSBzZXQgLS0gYXQgdGhlCiAgIH40NDBNLXBhaXIgc2NhbGUgZXN0aW1hdGVkIGZvciB0aGUgZnVsbCB0ZXN0IHNldCwgcmVtb3Zpbmcgb25lCiAgIG9mIHRoZSB0d28gcmVkdW5kYW50IHBhc3NlcyBwZXIgZmllbGQgaXMgYSBtZWFuaW5nZnVsIHdpbi4KICAgcmFwaWRmdXp6LmZ1enoucmF0aW8oKSBpcyBhIGNvbXBhdGlibGUtYnktZGVzaWduLCBDLWFjY2VsZXJhdGVkCiAgIHJlaW1wbGVtZW50YXRpb24gb2YgdGhlIHNhbWUgImRpZmZsaWItc3R5bGUiIHJhdGlvLCBzbwogICBuYW1lX2NoYXJfc2ltaWxhcml0eSAvIGFkZHJlc3NfY2hhcl9zaW1pbGFyaXR5IGFyZSBub3cgY29tcHV0ZWQKICAgd2l0aCBpdCB0b28uIElmIHJhcGlkZnV6eiBpcyB1bmF2YWlsYWJsZSwgdGhpcyBmYWxscyBiYWNrIHRvIHRoZQogICBvcmlnaW5hbCBkaWZmbGliIGltcGxlbWVudGF0aW9uIGV4YWN0bHkgYXMgYmVmb3JlIC0tIG5vIGZlYXR1cmUKICAgY2hhbmdlcyBzaGFwZSwgb25seSB3aGljaCBiYWNrZW5kIGNvbXB1dGVzIGl0LgoKMy4gRmVhdHVyZSBsaXN0IGlzIEFQUEVOREVEIHRvLCBub3QgcmVvcmRlcmVkIG9yIHJlbW92ZWQsIHNvCiAgIGV4aXN0aW5nIHRyYWluX2ZlYXR1cmVzXyoudHN2IGZpbGVzIHN0YXkgYSBzdHJpY3QgcHJlZml4LQogICBjb21wYXRpYmxlIHN1YnNldCBvZiB0aGUgbmV3IHNjaGVtYSAob2xkIGNvbHVtbnMga2VlcCB0aGUgc2FtZQogICBuYW1lIGFuZCBtZWFuaW5nKS4KIiIiCgpmcm9tIGZ1bmN0b29scyBpbXBvcnQgbHJ1X2NhY2hlCgpmcm9tIG5vcm1hbGl6ZSBpbXBvcnQgKAogICAgbm9ybWFsaXplX2J1c2luZXNzX25hbWUsCiAgICBub3JtYWxpemVfYWRkcmVzcywKICAgIG5vcm1hbGl6ZV9idXNpbmVzc19uYW1lX3NjcmlwdF9hZ25vc3RpYywKICAgIG5vcm1hbGl6ZV91bmljb2RlX3ByZXNlcnZpbmcsCiAgICBpc19ub25fbGF0aW4sCikKCmZyb20gc2ltaWxhcml0eSBpbXBvcnQgdG9rZW5fc2ltaWxhcml0eSwgYWRkcmVzc19udW1iZXJfbWF0Y2gsIGNvdW50cnlfbWF0Y2gKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgUmF0aW8gYmFja2VuZDogcmFwaWRmdXp6IHByZWZlcnJlZCAoQy1hY2NlbGVyYXRlZCwgZGlmZmxpYi0KIyBjb21wYXRpYmxlIHJhdGlvKSwgcHVyZS1QeXRob24gZGlmZmxpYiBhcyBhIGd1YXJhbnRlZWQgZmFsbGJhY2suCiMgT25lIGltcGxlbWVudGF0aW9uIG5vdyBzZXJ2ZXMgQk9USCAiY2hhcl9zaW1pbGFyaXR5IiBhbmQKIyAibGV2ZW5zaHRlaW5fc2ltaWxhcml0eSIgLS0gc2VlIGNoYW5nZSAjMiBhYm92ZS4KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKdHJ5OgogICAgZnJvbSByYXBpZGZ1enouZnV6eiBpbXBvcnQgcmF0aW8gYXMgX3JhcGlkZnV6el9yYXRpbwoKICAgIGRlZiBfcmF0aW8oYSwgYik6CiAgICAgICAgaWYgbm90IGEgb3Igbm90IGI6CiAgICAgICAgICAgIHJldHVybiAwLjAKICAgICAgICByZXR1cm4gX3JhcGlkZnV6el9yYXRpbyhhLCBiKSAvIDEwMC4wCgpleGNlcHQgSW1wb3J0RXJyb3I6CgogICAgZnJvbSBkaWZmbGliIGltcG9ydCBTZXF1ZW5jZU1hdGNoZXIKCiAgICBkZWYgX3JhdGlvKGEsIGIpOgogICAgICAgIGlmIG5vdCBhIG9yIG5vdCBiOgogICAgICAgICAgICByZXR1cm4gMC4wCiAgICAgICAgcmV0dXJuIFNlcXVlbmNlTWF0Y2hlcihOb25lLCBhLCBiKS5yYXRpbygpCgoKbGV2ZW5zaHRlaW5fc2ltaWxhcml0eSA9IF9yYXRpbyAgIyBrZXB0IGFzIGEgcHVibGljIG5hbWUgZm9yIGNvbXBhdGliaWxpdHkKCgpAbHJ1X2NhY2hlKG1heHNpemU9Ml8wMDBfMDAwKQpkZWYgY2FjaGVkX25vcm1hbGl6ZV9idXNpbmVzc19uYW1lKHZhbHVlKToKICAgIHJldHVybiBub3JtYWxpemVfYnVzaW5lc3NfbmFtZSh2YWx1ZSkKCgpAbHJ1X2NhY2hlKG1heHNpemU9Ml8wMDBfMDAwKQpkZWYgY2FjaGVkX25vcm1hbGl6ZV9hZGRyZXNzKHZhbHVlKToKICAgIHJldHVybiBub3JtYWxpemVfYWRkcmVzcyh2YWx1ZSkKCgpAbHJ1X2NhY2hlKG1heHNpemU9Ml8wMDBfMDAwKQpkZWYgY2FjaGVkX25vcm1hbGl6ZV9idXNpbmVzc19uYW1lX3VuaWNvZGUodmFsdWUpOgogICAgcmV0dXJuIG5vcm1hbGl6ZV9idXNpbmVzc19uYW1lX3NjcmlwdF9hZ25vc3RpYyh2YWx1ZSkKCgpAbHJ1X2NhY2hlKG1heHNpemU9Ml8wMDBfMDAwKQpkZWYgY2FjaGVkX25vcm1hbGl6ZV9hZGRyZXNzX3VuaWNvZGUodmFsdWUpOgogICAgcmV0dXJuIG5vcm1hbGl6ZV91bmljb2RlX3ByZXNlcnZpbmcodmFsdWUpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIENoYXJhY3RlciBuLWdyYW0gSmFjY2FyZAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpkZWYgY2hhcl9uZ3JhbV9qYWNjYXJkKGEsIGIsIG49Mik6CgogICAgaWYgbm90IGEgb3Igbm90IGI6CiAgICAgICAgcmV0dXJuIDAuMAoKICAgIGlmIGxlbihhKSA8IG4gb3IgbGVuKGIpIDwgbjoKICAgICAgICByZXR1cm4gMC4wCgogICAgZ3JhbXNfYSA9IHthW2k6aSArIG5dIGZvciBpIGluIHJhbmdlKGxlbihhKSAtIG4gKyAxKX0KICAgIGdyYW1zX2IgPSB7YltpOmkgKyBuXSBmb3IgaSBpbiByYW5nZShsZW4oYikgLSBuICsgMSl9CgogICAgaWYgbm90IGdyYW1zX2Egb3Igbm90IGdyYW1zX2I6CiAgICAgICAgcmV0dXJuIDAuMAoKICAgIHJldHVybiBsZW4oZ3JhbXNfYSAmIGdyYW1zX2IpIC8gbGVuKGdyYW1zX2EgfCBncmFtc19iKQoKCmRlZiBwcmVmaXhfc2ltaWxhcml0eShhLCBiLCBrPTQpOgoKICAgIGlmIG5vdCBhIG9yIG5vdCBiOgogICAgICAgIHJldHVybiAwLjAKCiAgICBhX3AsIGJfcCA9IGFbOmtdLCBiWzprXQogICAgbWF4X2xlbiA9IG1pbihsZW4oYV9wKSwgbGVuKGJfcCkpCgogICAgaWYgbWF4X2xlbiA9PSAwOgogICAgICAgIHJldHVybiAwLjAKCiAgICBtYXRjaGVkID0gc3VtKDEgZm9yIHgsIHkgaW4gemlwKGFfcCwgYl9wKSBpZiB4ID09IHkpCiAgICByZXR1cm4gbWF0Y2hlZCAvIG1heF9sZW4KCgpkZWYgc3VmZml4X3NpbWlsYXJpdHkoYSwgYiwgaz00KToKCiAgICBpZiBub3QgYSBvciBub3QgYjoKICAgICAgICByZXR1cm4gMC4wCgogICAgYV9zLCBiX3MgPSBhWy1rOl0sIGJbLWs6XQogICAgbWF4X2xlbiA9IG1pbihsZW4oYV9zKSwgbGVuKGJfcykpCgogICAgaWYgbWF4X2xlbiA9PSAwOgogICAgICAgIHJldHVybiAwLjAKCiAgICBtYXRjaGVkID0gc3VtKDEgZm9yIHgsIHkgaW4gemlwKHJldmVyc2VkKGFfcyksIHJldmVyc2VkKGJfcykpIGlmIHggPT0geSkKICAgIHJldHVybiBtYXRjaGVkIC8gbWF4X2xlbgoKCmRlZiB0b2tlbl9vdmVybGFwX2NvdW50KGEsIGIpOgoKICAgIGlmIG5vdCBhIG9yIG5vdCBiOgogICAgICAgIHJldHVybiAwCgogICAgcmV0dXJuIGxlbihzZXQoYS5zcGxpdCgpKSAmIHNldChiLnNwbGl0KCkpKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBGZWF0dXJlIGNvbHVtbnMKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIE9yaWdpbmFsIGNvbHVtbnMgYXJlIFVOQ0hBTkdFRCBpbiBuYW1lLCBtZWFuaW5nIGFuZCBwb3NpdGlvbi4KIyBOZXcgY29sdW1ucyBhcmUgYXBwZW5kZWQgYXQgdGhlIGVuZC4KCkZFQVRVUkVfQ09MVU1OUyA9IFsKCiAgICAjIE5hbWUgZmVhdHVyZXMgKG9yaWdpbmFsKQogICAgIm5hbWVfZXhhY3RfbWF0Y2giLAogICAgIm5hbWVfY2hhcl9zaW1pbGFyaXR5IiwKICAgICJuYW1lX2xldmVuc2h0ZWluX3NpbWlsYXJpdHkiLAogICAgIm5hbWVfdG9rZW5famFjY2FyZCIsCiAgICAibmFtZV90b2tlbl9vdmVybGFwIiwKICAgICJuYW1lX2NvbXBhY3Rfc2ltaWxhcml0eSIsCiAgICAibmFtZV9wcmVmaXhfc2ltaWxhcml0eSIsCiAgICAibmFtZV9zdWZmaXhfc2ltaWxhcml0eSIsCiAgICAibmFtZV9jaGFyX25ncmFtX2phY2NhcmQiLAoKICAgICMgQWRkcmVzcyBmZWF0dXJlcyAob3JpZ2luYWwpCiAgICAiYWRkcmVzc19leGFjdF9tYXRjaCIsCiAgICAiYWRkcmVzc19jaGFyX3NpbWlsYXJpdHkiLAogICAgImFkZHJlc3NfbGV2ZW5zaHRlaW5fc2ltaWxhcml0eSIsCiAgICAiYWRkcmVzc190b2tlbl9qYWNjYXJkIiwKICAgICJhZGRyZXNzX3Rva2VuX292ZXJsYXAiLAogICAgImFkZHJlc3NfbnVtYmVyX21hdGNoIiwKICAgICJhZGRyZXNzX2NoYXJfbmdyYW1famFjY2FyZCIsCgogICAgIyBPdGhlciAob3JpZ2luYWwpCiAgICAiY291bnRyeV9tYXRjaCIsCiAgICAibmFtZV9taXNzaW5nIiwKICAgICJhZGRyZXNzX21pc3NpbmciLAogICAgImNvbWJpbmVkX25hbWVfYWRkcmVzc19zY29yZSIsCgogICAgIyAtLS0tIE5FVzogbm9uLUxhdGluIGZhbGxiYWNrIHNpZ25hbCAtLS0tCiAgICAibmFtZV9pc19ub25fbGF0aW4iLAogICAgImFkZHJlc3NfaXNfbm9uX2xhdGluIiwKICAgICJuYW1lX2NoYXJfc2ltaWxhcml0eV91bmljb2RlIiwKICAgICJuYW1lX3Rva2VuX2phY2NhcmRfdW5pY29kZSIsCl0KCgpkZWYgY2FsY3VsYXRlX2ZlYXR1cmVzKG5hbWUxLCBhZGRyZXNzMSwgY291bnRyeTEsIG5hbWUyLCBhZGRyZXNzMiwgY291bnRyeTIpOgoKICAgICMgLS0tLS0tLS0gQVNDSUktZm9sZGVkIG5vcm1hbGl6YXRpb24gKG9yaWdpbmFsIHBhdGgpIC0tLS0tLS0tCiAgICBuMSA9IGNhY2hlZF9ub3JtYWxpemVfYnVzaW5lc3NfbmFtZShuYW1lMSkKICAgIG4yID0gY2FjaGVkX25vcm1hbGl6ZV9idXNpbmVzc19uYW1lKG5hbWUyKQogICAgYTEgPSBjYWNoZWRfbm9ybWFsaXplX2FkZHJlc3MoYWRkcmVzczEpCiAgICBhMiA9IGNhY2hlZF9ub3JtYWxpemVfYWRkcmVzcyhhZGRyZXNzMikKCiAgICBuYW1lX21pc3NpbmcgPSBpbnQobm90IG4xIG9yIG5vdCBuMikKICAgIGFkZHJlc3NfbWlzc2luZyA9IGludChub3QgYTEgb3Igbm90IGEyKQoKICAgIG5hbWVfY2hhcl9zaW0gPSBfcmF0aW8objEsIG4yKQogICAgbmFtZV9sZXZfc2ltID0gX3JhdGlvKG4xLCBuMikgICMgc2FtZSBiYWNrZW5kIG5vdywgc2VlIG1vZHVsZSBkb2NzdHJpbmcgIzIKICAgIG5hbWVfdG9rX2phY2NhcmQgPSB0b2tlbl9zaW1pbGFyaXR5KG4xLCBuMikKICAgIG5hbWVfY29tcGFjdF9zaW0gPSBfcmF0aW8objEucmVwbGFjZSgiICIsICIiKSwgbjIucmVwbGFjZSgiICIsICIiKSkKCiAgICBhZGRyZXNzX2NoYXJfc2ltID0gX3JhdGlvKGExLCBhMikKICAgIGFkZHJlc3NfbGV2X3NpbSA9IF9yYXRpbyhhMSwgYTIpCiAgICBhZGRyZXNzX3Rva19qYWNjYXJkID0gdG9rZW5fc2ltaWxhcml0eShhMSwgYTIpCgogICAgY29tYmluZWQgPSAwLjYgKiBuYW1lX2NoYXJfc2ltICsgMC40ICogYWRkcmVzc19jaGFyX3NpbQoKICAgICMgLS0tLS0tLS0gTkVXOiBzY3JpcHQtYWdub3N0aWMgZmFsbGJhY2sgZm9yIG5vbi1MYXRpbiBuYW1lcyAtLS0tLS0tLQogICAgIyBPbmx5IHBheSBmb3IgdGhlIGV4dHJhIG5vcm1hbGl6YXRpb24vc2ltaWxhcml0eSB3b3JrIHdoZW4gdGhlCiAgICAjIEFTQ0lJLWZvbGRlZCBmb3JtIGFjdHVhbGx5IGxvc3QgY29udGVudCAtLSBjaGVhcCBmb3IgdGhlCiAgICAjIChtYWpvcml0eSkgTGF0aW4tc2NyaXB0IHJvd3MsIHNpbmNlIGlzX25vbl9sYXRpbigpIHNob3J0LQogICAgIyBjaXJjdWl0cyBvbiB0aGUgYWxyZWFkeS1jYWNoZWQgYXNjaWlfZm9ybS4KICAgIG5hbWVfbm9uX2xhdGluID0gaXNfbm9uX2xhdGluKG5hbWUxKSBvciBpc19ub25fbGF0aW4obmFtZTIpCgogICAgaWYgbmFtZV9ub25fbGF0aW46CiAgICAgICAgdW4xID0gY2FjaGVkX25vcm1hbGl6ZV9idXNpbmVzc19uYW1lX3VuaWNvZGUobmFtZTEpCiAgICAgICAgdW4yID0gY2FjaGVkX25vcm1hbGl6ZV9idXNpbmVzc19uYW1lX3VuaWNvZGUobmFtZTIpCiAgICAgICAgbmFtZV9jaGFyX3NpbV91bmljb2RlID0gX3JhdGlvKHVuMSwgdW4yKQogICAgICAgIG5hbWVfdG9rX2phY2NhcmRfdW5pY29kZSA9IHRva2VuX3NpbWlsYXJpdHkodW4xLCB1bjIpCiAgICBlbHNlOgogICAgICAgIG5hbWVfY2hhcl9zaW1fdW5pY29kZSA9IG5hbWVfY2hhcl9zaW0KICAgICAgICBuYW1lX3Rva19qYWNjYXJkX3VuaWNvZGUgPSBuYW1lX3Rva19qYWNjYXJkCgogICAgYWRkcmVzc19ub25fbGF0aW4gPSBpc19ub25fbGF0aW4oYWRkcmVzczEpIG9yIGlzX25vbl9sYXRpbihhZGRyZXNzMikKCiAgICByZXR1cm4gewoKICAgICAgICAibmFtZV9leGFjdF9tYXRjaCI6IGludChib29sKG4xKSBhbmQgbjEgPT0gbjIpLAogICAgICAgICJuYW1lX2NoYXJfc2ltaWxhcml0eSI6IG5hbWVfY2hhcl9zaW0sCiAgICAgICAgIm5hbWVfbGV2ZW5zaHRlaW5fc2ltaWxhcml0eSI6IG5hbWVfbGV2X3NpbSwKICAgICAgICAibmFtZV90b2tlbl9qYWNjYXJkIjogbmFtZV90b2tfamFjY2FyZCwKICAgICAgICAibmFtZV90b2tlbl9vdmVybGFwIjogdG9rZW5fb3ZlcmxhcF9jb3VudChuMSwgbjIpLAogICAgICAgICJuYW1lX2NvbXBhY3Rfc2ltaWxhcml0eSI6IG5hbWVfY29tcGFjdF9zaW0sCiAgICAgICAgIm5hbWVfcHJlZml4X3NpbWlsYXJpdHkiOiBwcmVmaXhfc2ltaWxhcml0eShuMSwgbjIpLAogICAgICAgICJuYW1lX3N1ZmZpeF9zaW1pbGFyaXR5Ijogc3VmZml4X3NpbWlsYXJpdHkobjEsIG4yKSwKICAgICAgICAibmFtZV9jaGFyX25ncmFtX2phY2NhcmQiOiBjaGFyX25ncmFtX2phY2NhcmQobjEsIG4yKSwKCiAgICAgICAgImFkZHJlc3NfZXhhY3RfbWF0Y2giOiBpbnQoYm9vbChhMSkgYW5kIGExID09IGEyKSwKICAgICAgICAiYWRkcmVzc19jaGFyX3NpbWlsYXJpdHkiOiBhZGRyZXNzX2NoYXJfc2ltLAogICAgICAgICJhZGRyZXNzX2xldmVuc2h0ZWluX3NpbWlsYXJpdHkiOiBhZGRyZXNzX2xldl9zaW0sCiAgICAgICAgImFkZHJlc3NfdG9rZW5famFjY2FyZCI6IGFkZHJlc3NfdG9rX2phY2NhcmQsCiAgICAgICAgImFkZHJlc3NfdG9rZW5fb3ZlcmxhcCI6IHRva2VuX292ZXJsYXBfY291bnQoYTEsIGEyKSwKICAgICAgICAiYWRkcmVzc19udW1iZXJfbWF0Y2giOiBhZGRyZXNzX251bWJlcl9tYXRjaChhZGRyZXNzMSwgYWRkcmVzczIpLAogICAgICAgICJhZGRyZXNzX2NoYXJfbmdyYW1famFjY2FyZCI6IGNoYXJfbmdyYW1famFjY2FyZChhMSwgYTIpLAoKICAgICAgICAiY291bnRyeV9tYXRjaCI6IGNvdW50cnlfbWF0Y2goY291bnRyeTEsIGNvdW50cnkyKSwKICAgICAgICAibmFtZV9taXNzaW5nIjogbmFtZV9taXNzaW5nLAogICAgICAgICJhZGRyZXNzX21pc3NpbmciOiBhZGRyZXNzX21pc3NpbmcsCiAgICAgICAgImNvbWJpbmVkX25hbWVfYWRkcmVzc19zY29yZSI6IGNvbWJpbmVkLAoKICAgICAgICAibmFtZV9pc19ub25fbGF0aW4iOiBpbnQobmFtZV9ub25fbGF0aW4pLAogICAgICAgICJhZGRyZXNzX2lzX25vbl9sYXRpbiI6IGludChhZGRyZXNzX25vbl9sYXRpbiksCiAgICAgICAgIm5hbWVfY2hhcl9zaW1pbGFyaXR5X3VuaWNvZGUiOiBuYW1lX2NoYXJfc2ltX3VuaWNvZGUsCiAgICAgICAgIm5hbWVfdG9rZW5famFjY2FyZF91bmljb2RlIjogbmFtZV90b2tfamFjY2FyZF91bmljb2RlLAogICAgfQo=",
    "labels.py": "IiIiCmxhYmVscy5weQo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KUFJFU0VSVkVEOiBsb2FkX2dyb3VuZF90cnV0aCgpIC8gbGFiZWxfZm9yKCkgYXJlIFVOQ0hBTkdFRCBhbmQKc3RpbGwgY29ycmVjdCAtLSBrZWVwIHVzaW5nIHRoZW0gZm9yIHRyYWluXzEway1zY2FsZSB3b3JrIHdoZXJlCnRoZWlyIGV4dHJhIGNvbnZlbmllbmNlIChwcmUtc3BsaXQgIlMyIi8iUzMiLyJhbGwiIHNldHMpIGlzIGNoZWFwLgoKQURERUQ6IGxvYWRfZ3JvdW5kX3RydXRoX2NvbXBhY3QoKSwgYSBtZW1vcnktbGVhbiBhbHRlcm5hdGl2ZSBmb3IKdGhlIGZ1bGwgMi4yTS1yb3cgdHJhaW5fZ3JvdW5kX3RydXRoLnRzdi4KCk1FQVNVUkVEICh0aGlzIHNhbmRib3gsIDMuOSBHQiB0b3RhbCBSQU0sIFB5dGhvbiAzLjEyKToKICBsb2FkX2dyb3VuZF90cnV0aCgpIG9uIHRoZSBmdWxsIGZpbGUgLT4gMiwyMDYsODIxIGVudHJpZXMsCiAgcGVhayBSU1MgfjIuODQgR0IsIH4yNXMuCgpUaGF0IGlzIHRoZSBFTlRJUkUgcGVyLVMxLXJvdyBncm91bmQgdHJ1dGggc3RydWN0dXJlIHNpdHRpbmcgaW4KUkFNIGJlZm9yZSBjYW5kaWRhdGUgZ2VuZXJhdGlvbiwgZmVhdHVyZSBjYWxjdWxhdGlvbiwgaW5kZXhlcywgb3IKYW55dGhpbmcgZWxzZSBoYXMgYmVlbiBsb2FkZWQgLS0gb24gYSBLYWdnbGUgaW5zdGFuY2Ugd2l0aCwgc2F5LAoxMy0xNiBHQiB0b3RhbCBSQU0sIHRoaXMgaXMgYSBtZWFuaW5nZnVsIGZpeGVkIGNvc3QsIGFuZCBvbiBhCnNtYWxsZXIgaW5zdGFuY2UgaXQgY2FuIGJlIHRoZSBkaWZmZXJlbmNlIGJldHdlZW4gZml0dGluZyBpbgptZW1vcnkgYW5kIGFuIE9PTSBraWxsIG1pZC1ydW4gKHdoaWNoIGlzIGV4YWN0bHkgd2hhdCB0cnVuY2F0ZWQgYQpsYXJnZXIgdmFsaWRhdGlvbiBydW4gYXR0ZW1wdGVkIGR1cmluZyB0aGlzIGF1ZGl0KS4KClRoZSBjb3N0IGNvbWVzIGZyb20gc3RvcmluZyBUSFJFRSBQeXRob24gc2V0cyBwZXIgUzEgcm93CigiUzIiLCAiUzMiLCAiYWxsIikgaW5zdGVhZCBvZiBvbmUgLS0gZWFjaCBlbXB0eS9zbWFsbCBzZXQgaGFzCnJlYWwgZml4ZWQgb3ZlcmhlYWQgKGhhc2ggdGFibGUgYWxsb2NhdGlvbikgaW5kZXBlbmRlbnQgb2YgaG93Cm1hbnkgaWRzIGl0IGhvbGRzLCBzbyBidWlsZGluZyB0aHJlZSBwZXIgcm93IHJvdWdobHkgdHJpcGxlcyB0aGUKcGVyLXJvdyBjb250YWluZXIgb3ZlcmhlYWQgdmVyc3VzIHN0b3JpbmcgdGhlIHJhdyBpZCBzZXQgb25jZSBhbmQKZmlsdGVyaW5nIGJ5IHByZWZpeCBvbiBkZW1hbmQgKHdoaWNoIGlzIE8obGVuKGVudHJ5KSksIGFuZCBlbnRyaWVzCmhhdmUgb24gYXZlcmFnZSBvbmx5IGEgaGFuZGZ1bCBvZiBtYXRjaGVzKS4KCmxvYWRfZ3JvdW5kX3RydXRoX2NvbXBhY3QoKSBzdG9yZXMgZXhhY3RseSBvbmUgZnJvemVuc2V0IHBlciByb3cuCm1hdGNoZXNfZm9yX3NvdXJjZSgpL2xhYmVsX2Zvcl9jb21wYWN0KCkgYmVsb3cgcmVjb25zdHJ1Y3QgdGhlCiJTMiBvbmx5IiAvICJTMyBvbmx5IiB2aWV3IG9uIGRlbWFuZCwgYXQgZWZmZWN0aXZlbHkgbm8gZXh0cmEgY29zdApzaW5jZSBwZXItcm93IG1hdGNoIGNvdW50cyBhcmUgc21hbGwgKG1lZGlhbiB3ZWxsIHVuZGVyIDEwLCBwZXIKYW5hbHl6ZV9ncm91bmRfdHJ1dGgucHkncyBvd24gZGlzdHJpYnV0aW9uIGNoZWNrKS4KIiIiCgppbXBvcnQgY3N2CgoKZGVmIGxvYWRfZ3JvdW5kX3RydXRoKHBhdGgpOgogICAgIiIiCiAgICBVTkNIQU5HRUQuIFJldHVybnMgZGljdDogc291cmNlMV9lbnRpdHlfaWQgLT4gewogICAgICAgICJTMiI6IHNldCBvZiBtYXRjaGVkIFMyIGlkcywKICAgICAgICAiUzMiOiBzZXQgb2YgbWF0Y2hlZCBTMyBpZHMsCiAgICAgICAgImFsbCI6IHNldCBvZiBldmVyeSBtYXRjaGVkIGlkIChhbnkgc291cmNlKSwKICAgIH0KICAgICIiIgoKICAgIGdyb3VuZF90cnV0aCA9IHt9CgogICAgd2l0aCBvcGVuKHBhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIiwgbmV3bGluZT0iIikgYXMgZjoKCiAgICAgICAgcmVhZGVyID0gY3N2LkRpY3RSZWFkZXIoZiwgZGVsaW1pdGVyPSJcdCIpCgogICAgICAgIGZvciByb3cgaW4gcmVhZGVyOgoKICAgICAgICAgICAgczFfaWQgPSByb3dbInNvdXJjZTFfZW50aXR5X2lkIl0KICAgICAgICAgICAgcmF3ID0gKHJvdy5nZXQoIm1hdGNoZWRfZW50aXR5X2lkcyIpIG9yICIiKS5zdHJpcCgpCgogICAgICAgICAgICBpZiByYXc6CiAgICAgICAgICAgICAgICBhbGxfaWRzID0ge3guc3RyaXAoKSBmb3IgeCBpbiByYXcuc3BsaXQoIiwiKSBpZiB4LnN0cmlwKCl9CiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBhbGxfaWRzID0gc2V0KCkKCiAgICAgICAgICAgIGdyb3VuZF90cnV0aFtzMV9pZF0gPSB7CiAgICAgICAgICAgICAgICAiUzIiOiB7eCBmb3IgeCBpbiBhbGxfaWRzIGlmIHguc3RhcnRzd2l0aCgiUzItIil9LAogICAgICAgICAgICAgICAgIlMzIjoge3ggZm9yIHggaW4gYWxsX2lkcyBpZiB4LnN0YXJ0c3dpdGgoIlMzLSIpfSwKICAgICAgICAgICAgICAgICJhbGwiOiBhbGxfaWRzLAogICAgICAgICAgICB9CgogICAgcmV0dXJuIGdyb3VuZF90cnV0aAoKCmRlZiBsYWJlbF9mb3IoczFfaWQsIGNhbmRpZGF0ZV9pZCwgZ3JvdW5kX3RydXRoKToKICAgIGVudHJ5ID0gZ3JvdW5kX3RydXRoLmdldChzMV9pZCkKICAgIGlmIG5vdCBlbnRyeToKICAgICAgICByZXR1cm4gMAogICAgcmV0dXJuIGludChjYW5kaWRhdGVfaWQgaW4gZW50cnlbImFsbCJdKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgTkVXOiBtZW1vcnktbGVhbiBsb2FkZXIgZm9yIGZ1bGwtc2NhbGUgKDIuMk0rIHJvdykgZ3JvdW5kIHRydXRoCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpfRU1QVFlfRlJPWkVOU0VUID0gZnJvemVuc2V0KCkKCgpkZWYgbG9hZF9ncm91bmRfdHJ1dGhfY29tcGFjdChwYXRoKToKICAgICIiIgogICAgUmV0dXJucyBkaWN0OiBzb3VyY2UxX2VudGl0eV9pZCAtPiBmcm96ZW5zZXQgb2YgbWF0Y2hlZCBpZHMKICAgIChtaXhlZCBTMi0qL1MzLSopLiBPbmUgY29udGFpbmVyIHBlciByb3cgaW5zdGVhZCBvZiB0aHJlZS4KCiAgICBmcm96ZW5zZXQgKHZzIHNldCkgaXMgdXNlZCBiZWNhdXNlIHRoZXNlIGFyZSBuZXZlciBtdXRhdGVkCiAgICBhZnRlciBsb2FkaW5nLCBhbmQgQ1B5dGhvbidzIGZyb3plbnNldCBoYXMgc2xpZ2h0bHkgbGVzcwogICAgcGVyLW9iamVjdCBvdmVyaGVhZCB0aGFuIGEgbXV0YWJsZSBzZXQgZm9yIHJlYWQtb25seSB1c2UuCiAgICAiIiIKCiAgICBncm91bmRfdHJ1dGggPSB7fQoKICAgIHdpdGggb3BlbihwYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIsIG5ld2xpbmU9IiIpIGFzIGY6CgogICAgICAgIHJlYWRlciA9IGNzdi5EaWN0UmVhZGVyKGYsIGRlbGltaXRlcj0iXHQiKQoKICAgICAgICBmb3Igcm93IGluIHJlYWRlcjoKCiAgICAgICAgICAgIHMxX2lkID0gcm93WyJzb3VyY2UxX2VudGl0eV9pZCJdCiAgICAgICAgICAgIHJhdyA9IChyb3cuZ2V0KCJtYXRjaGVkX2VudGl0eV9pZHMiKSBvciAiIikuc3RyaXAoKQoKICAgICAgICAgICAgaWYgcmF3OgogICAgICAgICAgICAgICAgZ3JvdW5kX3RydXRoW3MxX2lkXSA9IGZyb3plbnNldCgKICAgICAgICAgICAgICAgICAgICB4LnN0cmlwKCkgZm9yIHggaW4gcmF3LnNwbGl0KCIsIikgaWYgeC5zdHJpcCgpCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBncm91bmRfdHJ1dGhbczFfaWRdID0gX0VNUFRZX0ZST1pFTlNFVAoKICAgIHJldHVybiBncm91bmRfdHJ1dGgKCgpkZWYgbWF0Y2hlc19mb3Jfc291cmNlKGVudHJ5LCBzb3VyY2VfcHJlZml4KToKICAgICIiIgogICAgZW50cnk6IGEgZnJvemVuc2V0IGZyb20gbG9hZF9ncm91bmRfdHJ1dGhfY29tcGFjdCgpLiBGaWx0ZXJzCiAgICBieSBwcmVmaXggb24gZGVtYW5kIC0tIGNoZWFwLCBzaW5jZSBlbnRyeSBpcyB0eXBpY2FsbHkgb25seSBhCiAgICBoYW5kZnVsIG9mIGlkcyAoc2VlIGFuYWx5emVfZ3JvdW5kX3RydXRoLnB5J3MgbWF0Y2hfY291bnQKICAgIGRpc3RyaWJ1dGlvbjogbWVkaWFuIG1hdGNoIGNvdW50IGlzIHNtYWxsKS4KICAgICIiIgogICAgaWYgbm90IGVudHJ5OgogICAgICAgIHJldHVybiBfRU1QVFlfRlJPWkVOU0VUCiAgICByZXR1cm4gZnJvemVuc2V0KHggZm9yIHggaW4gZW50cnkgaWYgeC5zdGFydHN3aXRoKHNvdXJjZV9wcmVmaXgpKQoKCmRlZiBsYWJlbF9mb3JfY29tcGFjdChzMV9pZCwgY2FuZGlkYXRlX2lkLCBncm91bmRfdHJ1dGgpOgogICAgZW50cnkgPSBncm91bmRfdHJ1dGguZ2V0KHMxX2lkKQogICAgaWYgbm90IGVudHJ5OgogICAgICAgIHJldHVybiAwCiAgICByZXR1cm4gaW50KGNhbmRpZGF0ZV9pZCBpbiBlbnRyeSkK",
    "checkpoint.py": "IiIiCmNoZWNrcG9pbnQucHkKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkdlbmVyaWMgY2hlY2twb2ludC9yZXN1bWUgc3VwcG9ydCBmb3IgbG9uZy1ydW5uaW5nLCBzaGFyZC1iYXNlZApzdGFnZXMgb2YgdGhlIHBpcGVsaW5lIChpbmRleCBidWlsZGluZywgY2FuZGlkYXRlIGdlbmVyYXRpb24gKwpmZWF0dXJlIGNhbGN1bGF0aW9uLCBzY29yaW5nKS4KCkRlc2lnbiBnb2FsczoKICAtIEEgS2FnZ2xlIHNlc3Npb24gaW50ZXJydXB0aW9uIGF0IEFOWSBwb2ludCBtdXN0IGxvc2UgYXQgbW9zdAogICAgb25lIHNoYXJkJ3Mgd29ydGggb2Ygd29yaywgbmV2ZXIgdGhlIHdob2xlIHJ1bi4KICAtIFJlc3VtaW5nIG11c3QgYmUgYSBuby1vcCBkZWNpc2lvbiAoInNraXAsIGFscmVhZHkgZG9uZSIpIGZvcgogICAgZXZlcnkgc2hhcmQgdGhhdCBhbHJlYWR5IGZpbmlzaGVkLCBzbyByZXN0YXJ0aW5nIHRoZSBzYW1lCiAgICBub3RlYm9vayBjZWxsIGlzIGFsd2F5cyBzYWZlIHRvIHJlLXJ1bi4KICAtIE5vIGV4dGVybmFsIHN0YXRlIGJleW9uZCBwbGFpbiBmaWxlcyBhbHJlYWR5IGluCiAgICAva2FnZ2xlL3dvcmtpbmcsIHNvIGl0IHN1cnZpdmVzIGEga2VybmVsIHJlc3RhcnQgb24gdGhlIHNhbWUKICAgIHBlcnNpc3RlZCB3b3JraW5nIGRpcmVjdG9yeS4KClRoaXMgbW9kdWxlIGtub3dzIG5vdGhpbmcgYWJvdXQgYnVzaW5lc3MgbG9naWMgKGJsb2NraW5nLApmZWF0dXJlcywgZXRjLikgLS0gaXQgb25seSB0cmFja3MgIndoaWNoIHNoYXJkcyBvZiB3aGljaCBzdGFnZQphcmUgZG9uZSIsIHNvIGl0IGlzIHNoYXJlZCBieSBldmVyeSBzdGFnZSBvZiB0aGUgcGlwZWxpbmUuCgpNYW5pZmVzdCBsYXlvdXQgKG9uZSBKU09OIGZpbGUgcGVyIHN0YWdlKToKCiAgICB7CiAgICAgICJzdGFnZSI6ICJjYW5kaWRhdGVzX3Rlc3RfZnVsbCIsCiAgICAgICJzaGFyZF9zaXplIjogNTAwMDAsCiAgICAgICJ0b3RhbF9yb3dzIjogMTczMDAwMCwKICAgICAgImNvbXBsZXRlZF9zaGFyZHMiOiBbMCwgMSwgMiwgLi4uXSwKICAgICAgInVwZGF0ZWRfYXQiOiAiMjAyNi0wOS0yNlQxMjowMDowMCIKICAgIH0KCkVhY2ggY29tcGxldGVkIHNoYXJkJ3MgT1VUUFVUIGlzIHdyaXR0ZW4gdG8gaXRzIG93biBmaWxlCihlLmcuIGNhbmRpZGF0ZV9wYWlyc19zaGFyZF8wMDA3LnRzdikuIE9uIHJlc3VtZSwgYSBzaGFyZCBpcwpvbmx5IHJlLXJ1biBpZiBpdCBpcyBtaXNzaW5nIGZyb20gY29tcGxldGVkX3NoYXJkcywgc28gaGFsZi0Kd3JpdHRlbiBvdXRwdXQgZmlsZXMgZnJvbSBhIGtpbGxlZCBwcm9jZXNzIGFyZSBzaW1wbHkKb3ZlcndyaXR0ZW4gLS0gdGhlcmUgaXMgbmV2ZXIgYSAicGFydGlhbGx5IHZhbGlkIiBzaGFyZCBmaWxlLgoiIiIKCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgdGVtcGZpbGUKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCgoKZGVmIF9hdG9taWNfd3JpdGVfanNvbihwYXRoLCBkYXRhKToKICAgICIiIgogICAgV3JpdGUgSlNPTiBhdG9taWNhbGx5OiB3cml0ZSB0byBhIHRlbXAgZmlsZSBpbiB0aGUgc2FtZQogICAgZGlyZWN0b3J5LCB0aGVuIG9zLnJlcGxhY2UoKSBpdCBvdmVyIHRoZSB0YXJnZXQuIFRoaXMgbWVhbnMKICAgIGEga2lsbCAtOSBtaWQtd3JpdGUgY2FuIG5ldmVyIGxlYXZlIGEgY29ycnVwdCBtYW5pZmVzdCAtLQogICAgcmVhZGVycyBhbHdheXMgc2VlIGVpdGhlciB0aGUgb2xkIG9yIHRoZSBuZXcgbWFuaWZlc3QsIG5ldmVyCiAgICBhIGhhbGYtd3JpdHRlbiBvbmUuCiAgICAiIiIKICAgIGRpcmVjdG9yeSA9IG9zLnBhdGguZGlybmFtZShwYXRoKSBvciAiLiIKICAgIG9zLm1ha2VkaXJzKGRpcmVjdG9yeSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBmZCwgdG1wX3BhdGggPSB0ZW1wZmlsZS5ta3N0ZW1wKGRpcj1kaXJlY3RvcnksIHByZWZpeD0iLmNrcHRfIiwgc3VmZml4PSIudG1wIikKICAgIHRyeToKICAgICAgICB3aXRoIG9zLmZkb3BlbihmZCwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBqc29uLmR1bXAoZGF0YSwgZikKICAgICAgICBvcy5yZXBsYWNlKHRtcF9wYXRoLCBwYXRoKQogICAgZmluYWxseToKICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyh0bXBfcGF0aCk6CiAgICAgICAgICAgIG9zLnJlbW92ZSh0bXBfcGF0aCkKCgpkZWYgbG9hZF9tYW5pZmVzdChtYW5pZmVzdF9wYXRoLCBzdGFnZSwgc2hhcmRfc2l6ZSwgdG90YWxfcm93cyk6CiAgICAiIiIKICAgIExvYWQgYW4gZXhpc3RpbmcgbWFuaWZlc3QsIG9yIGNyZWF0ZSBhIGZyZXNoIGluLW1lbW9yeSBvbmUgaWYKICAgIG5vbmUgZXhpc3RzIHlldCBvciBpdCBkb2Vzbid0IG1hdGNoIHRoZSBjdXJyZW50IHJ1bidzIHNoYXBlCiAgICAoc3RhZ2UgbmFtZSAvIHNoYXJkX3NpemUgLyB0b3RhbF9yb3dzKS4gQSBzaGFwZSBtaXNtYXRjaCBtZWFucwogICAgdGhlIHJ1biBjb25maWd1cmF0aW9uIGNoYW5nZWQsIHNvIHdlIHN0YXJ0IG92ZXIgcmF0aGVyIHRoYW4KICAgIHNpbGVudGx5IHJldXNpbmcgc2hhcmQgYm91bmRhcmllcyBjb21wdXRlZCB1bmRlciBhIGRpZmZlcmVudAogICAgc2hhcmRfc2l6ZS4KICAgICIiIgogICAgaWYgb3MucGF0aC5leGlzdHMobWFuaWZlc3RfcGF0aCk6CiAgICAgICAgd2l0aCBvcGVuKG1hbmlmZXN0X3BhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgZGF0YSA9IGpzb24ubG9hZChmKQoKICAgICAgICBpZiAoCiAgICAgICAgICAgIGRhdGEuZ2V0KCJzdGFnZSIpID09IHN0YWdlCiAgICAgICAgICAgIGFuZCBkYXRhLmdldCgic2hhcmRfc2l6ZSIpID09IHNoYXJkX3NpemUKICAgICAgICAgICAgYW5kIGRhdGEuZ2V0KCJ0b3RhbF9yb3dzIikgPT0gdG90YWxfcm93cwogICAgICAgICk6CiAgICAgICAgICAgIGRhdGFbImNvbXBsZXRlZF9zaGFyZHMiXSA9IHNldChkYXRhLmdldCgiY29tcGxldGVkX3NoYXJkcyIsIFtdKSkKICAgICAgICAgICAgcmV0dXJuIGRhdGEKCiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiW2NoZWNrcG9pbnRdIG1hbmlmZXN0IGF0IHttYW5pZmVzdF9wYXRofSBkb2VzIG5vdCBtYXRjaCAiCiAgICAgICAgICAgIGYiY3VycmVudCBydW4gc2hhcGUgLS0gc3RhcnRpbmcgYSBmcmVzaCBtYW5pZmVzdCBmb3IgJ3tzdGFnZX0nIgogICAgICAgICkKCiAgICByZXR1cm4gewogICAgICAgICJzdGFnZSI6IHN0YWdlLAogICAgICAgICJzaGFyZF9zaXplIjogc2hhcmRfc2l6ZSwKICAgICAgICAidG90YWxfcm93cyI6IHRvdGFsX3Jvd3MsCiAgICAgICAgImNvbXBsZXRlZF9zaGFyZHMiOiBzZXQoKSwKICAgICAgICAidXBkYXRlZF9hdCI6IE5vbmUsCiAgICB9CgoKZGVmIHNhdmVfbWFuaWZlc3QobWFuaWZlc3RfcGF0aCwgbWFuaWZlc3QpOgogICAgdG9fd3JpdGUgPSBkaWN0KG1hbmlmZXN0KQogICAgdG9fd3JpdGVbImNvbXBsZXRlZF9zaGFyZHMiXSA9IHNvcnRlZChtYW5pZmVzdFsiY29tcGxldGVkX3NoYXJkcyJdKQogICAgdG9fd3JpdGVbInVwZGF0ZWRfYXQiXSA9IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpCiAgICBfYXRvbWljX3dyaXRlX2pzb24obWFuaWZlc3RfcGF0aCwgdG9fd3JpdGUpCgoKZGVmIG1hcmtfc2hhcmRfY29tcGxldGUobWFuaWZlc3RfcGF0aCwgbWFuaWZlc3QsIHNoYXJkX2luZGV4KToKICAgICIiIgogICAgQ2FsbCB0aGlzIEFGVEVSIGEgc2hhcmQncyBvdXRwdXQgZmlsZSBoYXMgYmVlbiBmdWxseSB3cml0dGVuCiAgICBhbmQgY2xvc2VkIChyZW5hbWVkIGludG8gcGxhY2UgaWYgeW91IHdyaXRlLXRoZW4tcmVuYW1lKSwgc28KICAgIHRoZSBtYW5pZmVzdCBjYW4gbmV2ZXIgY2xhaW0gYSBzaGFyZCBpcyBkb25lIHdoaWxlIGl0cyBvdXRwdXQKICAgIGZpbGUgaXMgc3RpbGwgaW5jb21wbGV0ZS4KICAgICIiIgogICAgbWFuaWZlc3RbImNvbXBsZXRlZF9zaGFyZHMiXS5hZGQoc2hhcmRfaW5kZXgpCiAgICBzYXZlX21hbmlmZXN0KG1hbmlmZXN0X3BhdGgsIG1hbmlmZXN0KQoKCmRlZiBpc19zaGFyZF9jb21wbGV0ZShtYW5pZmVzdCwgc2hhcmRfaW5kZXgpOgogICAgcmV0dXJuIHNoYXJkX2luZGV4IGluIG1hbmlmZXN0WyJjb21wbGV0ZWRfc2hhcmRzIl0KCgpkZWYgaXRlcl9zaGFyZHModG90YWxfcm93cywgc2hhcmRfc2l6ZSk6CiAgICAiIiIKICAgIFlpZWxkIChzaGFyZF9pbmRleCwgc3RhcnRfcm93LCBlbmRfcm93KSB0cmlwbGVzIGNvdmVyaW5nCiAgICBbMCwgdG90YWxfcm93cykgaW4gZml4ZWQtc2l6ZSwgZGV0ZXJtaW5pc3RpYyBjaHVua3MuIFVzaW5nIGEKICAgIGZpeGVkIHNoYXJkX3NpemUgKHJhdGhlciB0aGFuICJOIHNoYXJkcyIpIG1lYW5zIGFkZGluZyByb3dzCiAgICB0byBhIHNvdXJjZSBmaWxlIG5ldmVyIGNoYW5nZXMgdGhlIGJvdW5kYXJpZXMgb2Ygc2hhcmRzIHRoYXQKICAgIGFscmVhZHkgY29tcGxldGVkLgogICAgIiIiCiAgICBzaGFyZF9pbmRleCA9IDAKICAgIHN0YXJ0ID0gMAogICAgd2hpbGUgc3RhcnQgPCB0b3RhbF9yb3dzOgogICAgICAgIGVuZCA9IG1pbihzdGFydCArIHNoYXJkX3NpemUsIHRvdGFsX3Jvd3MpCiAgICAgICAgeWllbGQgc2hhcmRfaW5kZXgsIHN0YXJ0LCBlbmQKICAgICAgICBzdGFydCA9IGVuZAogICAgICAgIHNoYXJkX2luZGV4ICs9IDEKCgpkZWYgc2hhcmRfb3V0cHV0X3BhdGgob3V0X2RpciwgcHJlZml4LCBzaGFyZF9pbmRleCk6CiAgICByZXR1cm4gb3MucGF0aC5qb2luKG91dF9kaXIsIGYie3ByZWZpeH1fc2hhcmRfe3NoYXJkX2luZGV4OjA1ZH0udHN2IikKCgpkZWYgd3JpdGVfc2hhcmRfYXRvbWljYWxseShmaW5hbF9wYXRoLCB3cml0ZV9mbik6CiAgICAiIiIKICAgIHdyaXRlX2ZuKHRtcF9wYXRoKSBtdXN0IGZ1bGx5IHdyaXRlIHRoZSBzaGFyZCdzIGNvbnRlbnQgdG8KICAgIHRtcF9wYXRoLiBUaGlzIGZ1bmN0aW9uIHRoZW4gcmVuYW1lcyBpdCBpbnRvIHBsYWNlLCBzbyBhbnkKICAgIHJlYWRlciBvbmx5IGV2ZXIgc2VlcyBhIGNvbXBsZXRlIGZpbGUgYXQgZmluYWxfcGF0aCwgYW5kIGEKICAgIHByb2Nlc3Mga2lsbGVkIG1pZC13cml0ZSBsZWF2ZXMgb25seSBhbiBvcnBoYW5lZCAudG1wIGZpbGUKICAgIChuZXZlciBhIHRydW5jYXRlZCBmaW5hbF9wYXRoKS4KICAgICIiIgogICAgZGlyZWN0b3J5ID0gb3MucGF0aC5kaXJuYW1lKGZpbmFsX3BhdGgpIG9yICIuIgogICAgb3MubWFrZWRpcnMoZGlyZWN0b3J5LCBleGlzdF9vaz1UcnVlKQoKICAgIHRtcF9wYXRoID0gZmluYWxfcGF0aCArICIudG1wIgogICAgd3JpdGVfZm4odG1wX3BhdGgpCiAgICBvcy5yZXBsYWNlKHRtcF9wYXRoLCBmaW5hbF9wYXRoKQoKCmRlZiBjb25jYXRfY29tcGxldGVkX3NoYXJkcyhvdXRfZGlyLCBwcmVmaXgsIG1hbmlmZXN0LCBoZWFkZXJfbGluZSwgZmluYWxfcGF0aCk6CiAgICAiIiIKICAgIENvbnZlbmllbmNlIGZvciB0aGUgbGFzdCBzdGVwIG9mIGEgc3RhZ2U6IHN0aXRjaCBldmVyeQogICAgY29tcGxldGVkIHNoYXJkJ3MgZmlsZSBpbnRvIG9uZSBmaW5hbCBvdXRwdXQsIGluIHNoYXJkIG9yZGVyLgogICAgU2FmZSB0byBjYWxsIG11bHRpcGxlIHRpbWVzIChpZGVtcG90ZW50KSBzaW5jZSBpdCBhbHdheXMKICAgIHJld3JpdGVzIGZpbmFsX3BhdGggZnJvbSB0aGUgc2hhcmQgZmlsZXMsIGFuZCBvbmx5IGV2ZXIgcmVhZHMKICAgIHNoYXJkcyByZWNvcmRlZCBhcyBjb21wbGV0ZSBpbiB0aGUgbWFuaWZlc3QuCiAgICAiIiIKICAgIHNoYXJkX2luZGljZXMgPSBzb3J0ZWQobWFuaWZlc3RbImNvbXBsZXRlZF9zaGFyZHMiXSkKCiAgICBkZWYgX3dyaXRlKHRtcF9wYXRoKToKICAgICAgICB3aXRoIG9wZW4odG1wX3BhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgb3V0X2Y6CiAgICAgICAgICAgIG91dF9mLndyaXRlKGhlYWRlcl9saW5lLnJzdHJpcCgiXG4iKSArICJcbiIpCiAgICAgICAgICAgIGZvciBzaGFyZF9pbmRleCBpbiBzaGFyZF9pbmRpY2VzOgogICAgICAgICAgICAgICAgc2hhcmRfcGF0aCA9IHNoYXJkX291dHB1dF9wYXRoKG91dF9kaXIsIHByZWZpeCwgc2hhcmRfaW5kZXgpCiAgICAgICAgICAgICAgICB3aXRoIG9wZW4oc2hhcmRfcGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBpbl9mOgogICAgICAgICAgICAgICAgICAgIG5leHQoaW5fZiwgTm9uZSkgICMgc2tpcCB0aGF0IHNoYXJkJ3Mgb3duIGhlYWRlcgogICAgICAgICAgICAgICAgICAgIGZvciBsaW5lIGluIGluX2Y6CiAgICAgICAgICAgICAgICAgICAgICAgIG91dF9mLndyaXRlKGxpbmUpCgogICAgd3JpdGVfc2hhcmRfYXRvbWljYWxseShmaW5hbF9wYXRoLCBfd3JpdGUpCiAgICByZXR1cm4gZmluYWxfcGF0aAo=",
}

for _fname, _b64 in _MODULE_SOURCES_B64.items():
    with open(os.path.join(MODULES_DIR, _fname), "w", encoding="utf-8") as _f:
        _f.write(base64.b64decode(_b64).decode("utf-8"))

if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)

print("Embedded", len(_MODULE_SOURCES_B64), "checkpoint modules ->", MODULES_DIR)
print(sorted(_MODULE_SOURCES_B64.keys()))


## 1. Dependencies

In [ ]:
# rapidfuzz speeds up features.py's similarity backend (falls back to
# stdlib difflib automatically if unavailable -- see features.py). Both
# lightgbm and rapidfuzz are preinstalled on Kaggle's standard image; the
# pip install is a harmless no-op if so, and a real install otherwise.
import subprocess, sys
for _pkg in ("lightgbm", "rapidfuzz", "psutil"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)

import gc
import time
import json as _json
import random
import shutil
import numpy as np
import pandas as pd

try:
    import psutil
    HAVE_PSUTIL = True
except ImportError:
    HAVE_PSUTIL = False

print("Dependencies ready. psutil available:", HAVE_PSUTIL)


## 2. Data discovery
Walks `/kaggle/input` for the four training files and, separately, the three
test files -- never assumes a specific dataset slug/path, since the mount
point can differ between sessions and forks.

In [ ]:
from indexer import discover_dataset_root

TRAIN_REQUIRED = ("train_source1.tsv", "train_source2.tsv", "train_source3.tsv", "train_ground_truth.tsv")
TEST_REQUIRED = ("test_source1.tsv", "test_source2.tsv", "test_source3.tsv")

SEARCH_ROOTS = ["/kaggle/input", "/mnt/user-data/uploads", "."]

def find_root(required_files):
    for root in SEARCH_ROOTS:
        found = discover_dataset_root(required_files=required_files, search_root=root)
        if found:
            return found
    return None

TRAIN_ROOT = find_root(TRAIN_REQUIRED)
TEST_ROOT = find_root(TEST_REQUIRED)

print("TRAIN_ROOT:", TRAIN_ROOT)
print("TEST_ROOT :", TEST_ROOT)

if TRAIN_ROOT is None:
    raise FileNotFoundError(
        "Could not find all of " + str(TRAIN_REQUIRED) + " anywhere under "
        + str(SEARCH_ROOTS) + ". Attach the training dataset and re-run this cell."
    )

TRAIN_S1 = os.path.join(TRAIN_ROOT, "train_source1.tsv")
TRAIN_S2 = os.path.join(TRAIN_ROOT, "train_source2.tsv")
TRAIN_S3 = os.path.join(TRAIN_ROOT, "train_source3.tsv")
TRAIN_GT = os.path.join(TRAIN_ROOT, "train_ground_truth.tsv")

HAS_TEST_DATA = TEST_ROOT is not None
if HAS_TEST_DATA:
    TEST_S1 = os.path.join(TEST_ROOT, "test_source1.tsv")
    TEST_S2 = os.path.join(TEST_ROOT, "test_source2.tsv")
    TEST_S3 = os.path.join(TEST_ROOT, "test_source3.tsv")
    print("Test files found:")
    print(" ", TEST_S1)
    print(" ", TEST_S2)
    print(" ", TEST_S3)
else:
    print(
        "WARNING: test files not found yet under /kaggle/input. "
        "Training cells below will still run. Attach the test dataset "
        "before running the TEST PIPELINE section, then re-run this cell."
    )


## 3. Load train S1 + ground truth, build the S1-level train/validation split

In [ ]:
from data_loader import read_tsv
from labels import load_ground_truth_compact

t0 = time.time()
train_s1_rows = list(read_tsv(TRAIN_S1))
print(f"Loaded {len(train_s1_rows):,} S1 training rows in {time.time()-t0:.1f}s")

t0 = time.time()
ground_truth = load_ground_truth_compact(TRAIN_GT)
print(f"Loaded ground truth for {len(ground_truth):,} S1 rows in {time.time()-t0:.1f}s")

# ---- S1-ENTITY-LEVEL split (leakage-safe: never put the same S1 in both
# train and validation -- candidates/features/labels are generated
# separately per split below, so nothing from a validation S1 entity can
# leak into training). ----
rng = random.Random(RANDOM_SEED)
all_indices = list(range(len(train_s1_rows)))
rng.shuffle(all_indices)

VAL_FRACTION = 0.20
val_size = int(len(all_indices) * VAL_FRACTION)
val_indices_full = all_indices[:val_size]
train_indices_full = all_indices[val_size:]

# Sample down to the configured training/validation sizes for this run.
train_indices = train_indices_full[:TRAIN_S1_SAMPLE]
val_indices = val_indices_full[:VAL_S1_SAMPLE]

print(f"Train S1 sample: {len(train_indices):,} rows  |  Val S1 sample: {len(val_indices):,} rows")
print("(sampled from a shuffled 80/20 S1-entity split of the full", len(train_s1_rows), "-row file)")


## 4. Build the training S2 / S3 indexes
Reuses `indexer.build_index()` from the checkpoint: SQLite-backed, resumable
(checkpoints progress every ~200k rows via a manifest, so a Kaggle interruption
loses at most a partial batch, not the whole build), and reused as-is if a
previous run already finished it. **Do not delete `index_train/` between runs
if you want to skip rebuilding.**

In [ ]:
from indexer import build_index

if FREQ_CAPS_OVERRIDE is not None:
    import blocking
    blocking.DEFAULT_FREQ_CAPS.update(FREQ_CAPS_OVERRIDE)
    print("Overrode blocking caps:", FREQ_CAPS_OVERRIDE)
else:
    import blocking
    print("Using module-default blocking caps (loosened, confirmed-win config):")
    print(blocking.DEFAULT_FREQ_CAPS)

t0 = time.time()
train_s2_index = build_index(
    TRAIN_S2, "S2", limit=None, backend="sqlite",
    db_path=os.path.join(INDEX_TRAIN_DIR, "s2_full.sqlite"),
)
print(f"S2 train index ready in {time.time()-t0:.1f}s")

t0 = time.time()
train_s3_index = build_index(
    TRAIN_S3, "S3", limit=None, backend="sqlite",
    db_path=os.path.join(INDEX_TRAIN_DIR, "s3_full.sqlite"),
)
print(f"S3 train index ready in {time.time()-t0:.1f}s")

TRAIN_INDEXES = {"S2": train_s2_index, "S3": train_s3_index}


## 5. Generate labeled, featurized candidate pairs for TRAIN and VALIDATION
Both splits go through the exact same function (`build_labeled_feature_table`)
so there is no train/val asymmetry in how candidates or features are produced --
only the underlying S1 rows differ (leakage-safe by construction, since
`train_indices` and `val_indices` never overlap).

In [ ]:
from candidates import generate_candidate_pairs_for_row
from features import calculate_features, FEATURE_COLUMNS
from labels import label_for_compact
from indexer import split_entity_id

MODEL_FEATURE_COLUMNS = FEATURE_COLUMNS + ["blocking_num_keys", "blocking_min_freq"]
print(f"Model will use {len(MODEL_FEATURE_COLUMNS)} features:")
print(MODEL_FEATURE_COLUMNS)


def build_labeled_feature_table(s1_rows_subset, indexes_by_source, ground_truth, label_split_name):
    """
    For each S1 row in s1_rows_subset: generate candidates (blocking, with
    the live frequency caps), look up each candidate's attrs from the
    already-built index, compute the 26-feature vector, and label it from
    ground truth. Returns a DataFrame with one row per (S1, candidate) pair.

    This is only ever called on a bounded SAMPLE of S1 rows (TRAIN_S1_SAMPLE /
    VAL_S1_SAMPLE), so the resulting candidate table comfortably fits in RAM --
    unlike the full test set, which the streaming pipeline below never does.
    """
    t0 = time.time()
    rows_out = []
    total_gt_matches = 0
    found_gt_matches = 0

    for i, idx in enumerate(s1_rows_subset, start=1):
        s1_row = train_s1_rows[idx]
        s1_id = s1_row["entity_id"]

        pairs = generate_candidate_pairs_for_row(s1_row, indexes_by_source)

        gt_entry = ground_truth.get(s1_id)
        gt_ids = set(gt_entry) if gt_entry else set()
        found_ids = set()

        for p in pairs:
            candidate_id = p["candidate_entity_id"]
            source = p["source"]
            _, suffix = split_entity_id(candidate_id)
            attrs = indexes_by_source[source].get_attrs(str(suffix))
            if attrs is None:
                continue
            name2, address2, country2 = attrs

            feats = calculate_features(
                s1_row["business_name"], s1_row["business_address"], s1_row["country"],
                name2, address2, country2,
            )
            feats["blocking_num_keys"] = p["blocking_num_keys"]
            feats["blocking_min_freq"] = p["blocking_min_freq"]
            label = label_for_compact(s1_id, candidate_id, ground_truth)
            feats["label"] = label
            feats["source1_entity_id"] = s1_id
            feats["candidate_entity_id"] = candidate_id
            rows_out.append(feats)

            if candidate_id in gt_ids:
                found_ids.add(candidate_id)

        total_gt_matches += len(gt_ids)
        found_gt_matches += len(found_ids)

        if i % 500 == 0:
            elapsed = time.time() - t0
            print(f"  [{label_split_name}] {i:,}/{len(s1_rows_subset):,} S1 rows "
                  f"({elapsed:.0f}s, {i/max(elapsed,1e-6):.1f} rows/s)")

    df = pd.DataFrame(rows_out)
    recall = (found_gt_matches / total_gt_matches) if total_gt_matches else float("nan")
    print(f"[{label_split_name}] DONE: {len(s1_rows_subset):,} S1 rows -> "
          f"{len(df):,} candidate pairs, {df['label'].sum() if len(df) else 0:,} positive, "
          f"candidate recall={recall:.4%} ({found_gt_matches}/{total_gt_matches})")
    return df


print("Building TRAIN candidate/feature table...")
train_feat_df = build_labeled_feature_table(train_indices, TRAIN_INDEXES, ground_truth, "train")

print()
print("Building VALIDATION candidate/feature table...")
val_feat_df = build_labeled_feature_table(val_indices, TRAIN_INDEXES, ground_truth, "val")


## 6. Train the LightGBM matcher

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

X_all = train_feat_df[MODEL_FEATURE_COLUMNS].astype(np.float32)
y_all = train_feat_df["label"].astype(int)

print("Training pairs:", len(X_all), " positives:", int(y_all.sum()), " negatives:", int((1 - y_all).sum()))

# Pair-level split of the TRAIN-S1-only candidates, used only for LightGBM's
# internal early-stopping eval set. This is safe (no leakage into the
# S1-level validation set below) because every pair here comes from an S1
# entity in `train_indices`, which never overlaps `val_indices`.
X_tr, X_es, y_tr, y_es = train_test_split(
    X_all, y_all, test_size=0.15, random_state=RANDOM_SEED,
    stratify=y_all if y_all.nunique() > 1 else None,
)

model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=400,
    num_leaves=63,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_samples=20,
    random_state=RANDOM_SEED,
    verbosity=-1,
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_es, y_es)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
)

es_proba = model.predict_proba(X_es)[:, 1]
print("Internal eval ROC-AUC:", roc_auc_score(y_es, es_proba))
print("Internal eval PR-AUC :", average_precision_score(y_es, es_proba))
print("Best iteration:", model.best_iteration_)


## 7. S1-level Macro F0.5 (the official metric)
`F0.5 = (1.25 * P * R) / (0.25 * P + R)`, computed **per S1 entity** (singletons
included: a correctly-empty prediction scores 1.0, a false merge on a true
singleton scores 0.0), then macro-averaged across S1 entities.

In [ ]:
def macro_f_beta(predicted_by_s1, actual_by_s1, all_s1_ids, beta=0.5):
    """
    predicted_by_s1 / actual_by_s1: dict s1_id -> set of matched ids.
    all_s1_ids: every S1 id that must be scored (missing from either dict
    is treated as an empty prediction/empty ground truth, per the spec).
    Returns (macro_f_beta, per_s1_scores dict).
    """
    beta2 = beta * beta
    scores = {}
    for s1_id in all_s1_ids:
        pred = predicted_by_s1.get(s1_id, set())
        actual = actual_by_s1.get(s1_id, set())

        if not pred and not actual:
            scores[s1_id] = 1.0
            continue
        if not pred and actual:
            scores[s1_id] = 0.0
            continue
        if pred and not actual:
            scores[s1_id] = 0.0
            continue

        tp = len(pred & actual)
        precision = tp / len(pred) if pred else 0.0
        recall = tp / len(actual) if actual else 0.0
        if precision == 0.0 and recall == 0.0:
            scores[s1_id] = 0.0
        else:
            scores[s1_id] = (1 + beta2) * precision * recall / (beta2 * precision + recall)

    macro = sum(scores.values()) / len(scores) if scores else float("nan")
    return macro, scores


def evaluate_threshold(val_df, threshold, ground_truth, val_s1_ids):
    """val_df must have columns source1_entity_id, candidate_entity_id, model_score."""
    predicted_by_s1 = {}
    matched = val_df[val_df["model_score"] >= threshold]
    for s1_id, group in matched.groupby("source1_entity_id"):
        predicted_by_s1[s1_id] = set(group["candidate_entity_id"])

    actual_by_s1 = {s1_id: set(ground_truth.get(s1_id, set())) for s1_id in val_s1_ids}

    macro, _ = macro_f_beta(predicted_by_s1, actual_by_s1, val_s1_ids, beta=0.5)
    return macro


val_X = val_feat_df[MODEL_FEATURE_COLUMNS].astype(np.float32)
val_feat_df = val_feat_df.copy()
val_feat_df["model_score"] = model.predict_proba(val_X)[:, 1]

val_s1_ids = [train_s1_rows[i]["entity_id"] for i in val_indices]

baseline_f05 = evaluate_threshold(val_feat_df, BASELINE_THRESHOLD, ground_truth, val_s1_ids)
print("=" * 70)
print(f"INITIAL BASELINE THRESHOLD = {BASELINE_THRESHOLD}")
print(f"S1-level Macro F0.5 (validation) = {baseline_f05:.6f}")
print("=" * 70)


## 8. Optional lightweight threshold search
Distinct from the baseline above -- this is the **OPTIMIZED VALIDATION
THRESHOLD**, found by sweeping a small grid and re-scoring Macro F0.5 on the
same validation set. Only adopted if it actually beats the baseline.

In [ ]:
final_threshold = BASELINE_THRESHOLD
final_f05 = baseline_f05
search_results = []

if RUN_THRESHOLD_SEARCH:
    for t in THRESHOLD_SEARCH_GRID:
        f05 = evaluate_threshold(val_feat_df, t, ground_truth, val_s1_ids)
        search_results.append((t, f05))
        print(f"  threshold={t:.3f}  Macro F0.5={f05:.6f}")

    best_t, best_f05 = max(search_results, key=lambda x: x[1])
    print()
    if best_f05 > baseline_f05:
        final_threshold, final_f05 = best_t, best_f05
        print(f"OPTIMIZED VALIDATION THRESHOLD = {final_threshold:.3f} "
              f"(Macro F0.5 = {final_f05:.6f}, beats baseline {baseline_f05:.6f})")
    else:
        print(f"No grid point beat the baseline ({baseline_f05:.6f}) -- "
              f"keeping INITIAL BASELINE THRESHOLD = {BASELINE_THRESHOLD}")
else:
    print("RUN_THRESHOLD_SEARCH is False -- using INITIAL BASELINE THRESHOLD only.")

print()
print(f"==> FINAL THRESHOLD USED FOR TEST INFERENCE: {final_threshold}")
print(f"==> Validation Macro F0.5 at this threshold: {final_f05:.6f}")


## 9. Save the trained model + config (so this cell need not re-run if you restart)

In [ ]:
import joblib

model_path = os.path.join(MODEL_DIR, "lightgbm_matcher.joblib")
joblib.dump(model, model_path)

run_config = {
    "feature_columns": MODEL_FEATURE_COLUMNS,
    "final_threshold": final_threshold,
    "baseline_threshold": BASELINE_THRESHOLD,
    "baseline_f05": baseline_f05,
    "final_f05": final_f05,
    "train_s1_sample": TRAIN_S1_SAMPLE,
    "val_s1_sample": VAL_S1_SAMPLE,
}
with open(os.path.join(MODEL_DIR, "run_config.json"), "w") as f:
    _json.dump(run_config, f, indent=2)

print("Saved model to:", model_path)
print("Saved run config:", run_config)


# TEST PIPELINE

This is the section that failed last time: raw-candidate disk dump ran out of
disk at ~214.8M candidates, and the corrected in-memory-filter version
accumulated ~58.57M candidate objects in Python RAM and got killed at
~700k/1.73M S1 rows.

**This version never holds more than one chunk's candidates in memory.**
S1 is processed in fixed-size chunks (`S1_CHUNK_SIZE`, configured above); for
each chunk: generate candidates -> extract features -> score -> threshold ->
**write to disk immediately** -> discard the chunk's Python objects -> `gc.collect()`
-> next chunk. Progress is checkpointed shard-by-shard (`checkpoint.py`'s
manifest), so a Kaggle interruption resumes from the last completed shard
instead of restarting.

## 10. Build the TEST S2 / S3 indexes

In [ ]:
if not HAS_TEST_DATA:
    raise FileNotFoundError(
        "Test files not found under /kaggle/input. Attach your Kaggle dataset "
        "containing test_source1.tsv / test_source2.tsv / test_source3.tsv, "
        "then re-run the data-discovery cell (section 2) and this cell."
    )

t0 = time.time()
test_s2_index = build_index(
    TEST_S2, "S2", limit=None, backend="sqlite",
    db_path=os.path.join(INDEX_TEST_DIR, "s2_full.sqlite"),
)
print(f"S2 test index ready in {time.time()-t0:.1f}s")

t0 = time.time()
test_s3_index = build_index(
    TEST_S3, "S3", limit=None, backend="sqlite",
    db_path=os.path.join(INDEX_TEST_DIR, "s3_full.sqlite"),
)
print(f"S3 test index ready in {time.time()-t0:.1f}s")

TEST_INDEXES = {"S2": test_s2_index, "S3": test_s3_index}

total, used, free = shutil.disk_usage(WORK_DIR)
print(f"Disk after test indexing: {free/1024**3:.2f} GB free of {total/1024**3:.2f} GB")


## 11. Load test S1 (reference list only -- the row data itself was never
the memory problem; the *candidate* accumulation was). Determine chunk boundaries.

In [ ]:
t0 = time.time()
test_s1_rows = []
country_counts = {}
for row in read_tsv(TEST_S1):
    test_s1_rows.append({
        "entity_id": row.get("entity_id", ""),
        "business_name": row.get("business_name", ""),
        "business_address": row.get("business_address", ""),
        "country": (row.get("country", "") or "").strip(),
    })
    c = test_s1_rows[-1]["country"]
    country_counts[c] = country_counts.get(c, 0) + 1

print(f"Loaded {len(test_s1_rows):,} test S1 rows in {time.time()-t0:.1f}s")
print("Country distribution:", country_counts)

TOTAL_TEST_S1 = len(test_s1_rows)

from checkpoint import (
    load_manifest, save_manifest, mark_shard_complete, is_shard_complete,
    iter_shards, shard_output_path, write_shard_atomically, concat_completed_shards,
)

MANIFEST_PATH = os.path.join(SHARDS_DIR, "test_pipeline_manifest.json")
manifest = load_manifest(MANIFEST_PATH, stage="test_pipeline_v1",
                          shard_size=S1_CHUNK_SIZE, total_rows=TOTAL_TEST_S1)

n_shards = len(list(iter_shards(TOTAL_TEST_S1, S1_CHUNK_SIZE)))
print(f"Chunking {TOTAL_TEST_S1:,} S1 rows into {n_shards} shards of up to {S1_CHUNK_SIZE:,} rows each.")
print(f"Already-completed shards from a previous run: {len(manifest['completed_shards'])}/{n_shards}")


## 12. Process each chunk: candidates -> features -> score -> threshold -> write -> discard
Run this cell; it is safe to interrupt and re-run -- completed shards are skipped.

In [ ]:
CANDIDATE_HEADER = "source1_entity_id\tcandidate_entity_ids\n"
MATCHING_HEADER = "source1_entity_id\tmatched_entity_ids\n"

FEATURE_COLS_FOR_SCORING = MODEL_FEATURE_COLUMNS  # 26 features, same as training


def format_id_list(ids):
    # No duplicates within a list, comma-separated, no quoting -- per spec.
    seen = []
    seen_set = set()
    for x in ids:
        if x not in seen_set:
            seen_set.add(x)
            seen.append(x)
    return ",".join(seen)


def process_shard(shard_index, start, end):
    chunk_rows = test_s1_rows[start:end]

    # ---- 1. candidate generation for this chunk only ----
    candidate_records = []
    for s1_row in chunk_rows:
        pairs = generate_candidate_pairs_for_row(s1_row, TEST_INDEXES)
        for p in pairs:
            source = p["source"]
            _, suffix = split_entity_id(p["candidate_entity_id"])
            attrs = TEST_INDEXES[source].get_attrs(str(suffix))
            if attrs is None:
                continue
            name2, address2, country2 = attrs
            candidate_records.append({
                "source1_entity_id": p["source1_entity_id"],
                "candidate_entity_id": p["candidate_entity_id"],
                "s1_name": s1_row["business_name"],
                "s1_address": s1_row["business_address"],
                "s1_country": s1_row["country"],
                "name2": name2, "address2": address2, "country2": country2,
                "blocking_num_keys": p["blocking_num_keys"],
                "blocking_min_freq": p["blocking_min_freq"],
            })

    # ---- 2. candidates_by_s1 (every S1 gets a row in candidate_pairs.tsv, even empty) ----
    candidates_by_s1 = {row["entity_id"]: [] for row in chunk_rows}
    for rec in candidate_records:
        candidates_by_s1[rec["source1_entity_id"]].append(rec["candidate_entity_id"])

    # ---- 3. feature extraction + scoring (vectorized over this chunk's candidates only) ----
    matches_by_s1 = {row["entity_id"]: [] for row in chunk_rows}

    if candidate_records:
        feat_rows = []
        for rec in candidate_records:
            feats = calculate_features(
                rec["s1_name"], rec["s1_address"], rec["s1_country"],
                rec["name2"], rec["address2"], rec["country2"],
            )
            feats["blocking_num_keys"] = rec["blocking_num_keys"]
            feats["blocking_min_freq"] = rec["blocking_min_freq"]
            feat_rows.append(feats)

        feat_df = pd.DataFrame(feat_rows)
        X_chunk = feat_df[FEATURE_COLS_FOR_SCORING].astype(np.float32)
        scores = model.predict_proba(X_chunk)[:, 1]

        for rec, score in zip(candidate_records, scores):
            if score >= final_threshold:
                matches_by_s1[rec["source1_entity_id"]].append(rec["candidate_entity_id"])

        del feat_rows, feat_df, X_chunk, scores
    del candidate_records

    # ---- 4. sanity check: every predicted match is one of that row's candidates ----
    for s1_id, matched in matches_by_s1.items():
        assert set(matched).issubset(set(candidates_by_s1[s1_id])), (
            f"Pipeline bug: {s1_id} has a match not present in its own candidate set"
        )

    # ---- 5. write this shard's two output files atomically ----
    cand_path = shard_output_path(SHARDS_DIR, "candidate_pairs", shard_index)
    match_path = shard_output_path(SHARDS_DIR, "matching_results", shard_index)

    def _write_cand(tmp_path):
        with open(tmp_path, "w", encoding="utf-8", newline="") as f:
            f.write(CANDIDATE_HEADER)
            for row in chunk_rows:
                s1_id = row["entity_id"]
                f.write(f"{s1_id}\t{format_id_list(candidates_by_s1[s1_id])}\n")

    def _write_match(tmp_path):
        with open(tmp_path, "w", encoding="utf-8", newline="") as f:
            f.write(MATCHING_HEADER)
            for row in chunk_rows:
                s1_id = row["entity_id"]
                f.write(f"{s1_id}\t{format_id_list(matches_by_s1[s1_id])}\n")

    write_shard_atomically(cand_path, _write_cand)
    write_shard_atomically(match_path, _write_match)

    n_matched_rows = sum(1 for v in matches_by_s1.values() if v)
    del candidates_by_s1, matches_by_s1, chunk_rows
    gc.collect()

    return n_matched_rows


t_start = time.time()
for shard_index, start, end in iter_shards(TOTAL_TEST_S1, S1_CHUNK_SIZE):

    if is_shard_complete(manifest, shard_index):
        continue

    n_matched = process_shard(shard_index, start, end)

    mark_shard_complete(MANIFEST_PATH, manifest, shard_index)

    if HAVE_PSUTIL:
        free_gb = psutil.virtual_memory().available / 1024**3
        if free_gb < MEMORY_SAFETY_FREE_GB:
            print(f"  [!] Free RAM {free_gb:.2f}GB below safety threshold "
                  f"({MEMORY_SAFETY_FREE_GB}GB) -- forcing extra gc.collect()")
            gc.collect()
    else:
        free_gb = float("nan")

    done = len(manifest["completed_shards"])
    elapsed = time.time() - t_start
    rate = done / max(elapsed, 1e-6)
    eta_min = (n_shards - done) / max(rate, 1e-9) / 60
    print(f"shard {shard_index+1}/{n_shards} done ({end-start} S1 rows, "
          f"{n_matched} with >=1 match) | free_ram={free_gb:.2f}GB | "
          f"elapsed={elapsed/60:.1f}min | ETA~{eta_min:.1f}min")

print("All shards processed." if len(manifest["completed_shards"]) == n_shards
      else f"WARNING: only {len(manifest['completed_shards'])}/{n_shards} shards complete -- re-run this cell to resume.")


## 13. Stitch completed shards into the final `matching_results.tsv` / `candidate_pairs.tsv`

In [ ]:
CANDIDATE_PAIRS_PATH = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
MATCHING_RESULTS_PATH = os.path.join(OUTPUT_DIR, "matching_results.tsv")

assert len(manifest["completed_shards"]) == n_shards, (
    "Not all shards are complete -- re-run the previous cell until it reports "
    "'All shards processed.' before stitching the final output."
)

concat_completed_shards(SHARDS_DIR, "candidate_pairs", manifest, CANDIDATE_HEADER, CANDIDATE_PAIRS_PATH)
concat_completed_shards(SHARDS_DIR, "matching_results", manifest, MATCHING_HEADER, MATCHING_RESULTS_PATH)

print("Wrote:", CANDIDATE_PAIRS_PATH, f"({os.path.getsize(CANDIDATE_PAIRS_PATH)/1024**2:.1f} MB)")
print("Wrote:", MATCHING_RESULTS_PATH, f"({os.path.getsize(MATCHING_RESULTS_PATH)/1024**2:.1f} MB)")


## 14. Validate the output files
First tries to find and run the official `utils/validate_submission.py` from the
competition's own student_resource package if it's present under `/kaggle/input`
(that is the authoritative check). If it isn't attached, falls back to a
self-contained check of every rule listed in the problem statement.

In [ ]:
import glob

def find_official_validator():
    for pattern in ("/kaggle/input/**/validate_submission.py", "/mnt/user-data/uploads/**/validate_submission.py"):
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    return None

official_validator = find_official_validator()

if official_validator:
    print("Found official validator:", official_validator)
    test_dir_guess = os.path.dirname(TEST_S1)
    result = subprocess.run(
        [sys.executable, official_validator,
         "--matching", MATCHING_RESULTS_PATH,
         "--candidate", CANDIDATE_PAIRS_PATH,
         "--test-dir", test_dir_guess],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    print("Official validator exit code:", result.returncode, "(0 = PASS)")
else:
    print("Official validate_submission.py not found under /kaggle/input -- "
          "running an equivalent self-contained check instead.")

    def self_check(matching_path, candidate_path, test_s1_ids, valid_s2_ids_fn, valid_s3_ids_fn):
        issues = []

        def load_id_list_file(path, id_col, list_col):
            seen_ids = set()
            dup_ids = []
            row_dupes_within_list = 0
            invalid_ids = 0
            rows = 0
            with open(path, "r", encoding="utf-8", newline="") as f:
                header = f.readline().rstrip("\n").split("\t")
                if header != [id_col, list_col]:
                    issues.append(f"{path}: header is {header}, expected ['{id_col}', '{list_col}']")
                for line in f:
                    rows += 1
                    parts = line.rstrip("\n").split("\t")
                    if len(parts) == 1:
                        parts.append("")
                    s1_id, id_list = parts[0], parts[1]
                    if s1_id in seen_ids:
                        dup_ids.append(s1_id)
                    seen_ids.add(s1_id)
                    if id_list:
                        ids = id_list.split(",")
                        if len(ids) != len(set(ids)):
                            row_dupes_within_list += 1
                        for cid in ids:
                            if cid.startswith("S2-"):
                                if not valid_s2_ids_fn(cid):
                                    invalid_ids += 1
                            elif cid.startswith("S3-"):
                                if not valid_s3_ids_fn(cid):
                                    invalid_ids += 1
                            else:
                                invalid_ids += 1
            if dup_ids:
                issues.append(f"{path}: {len(dup_ids)} duplicate source1_entity_id rows")
            if row_dupes_within_list:
                issues.append(f"{path}: {row_dupes_within_list} rows have duplicate IDs within their own list")
            if invalid_ids:
                issues.append(f"{path}: {invalid_ids} candidate/match IDs are not valid S2-/S3- test IDs")
            missing = test_s1_ids - seen_ids
            if missing:
                issues.append(f"{path}: {len(missing)} test S1 entities are missing a row")
            extra = seen_ids - test_s1_ids
            if extra:
                issues.append(f"{path}: {len(extra)} rows reference S1 ids not in the test set")
            return issues, rows

        # Candidates came directly from the built indexes, so id validity is
        # guaranteed by construction -- but we check anyway via the index's
        # own attrs table (cheap point lookups, no full id set in RAM).
        def valid_s2(cid):
            _, suffix = split_entity_id(cid)
            return TEST_INDEXES["S2"].get_attrs(str(suffix)) is not None

        def valid_s3(cid):
            _, suffix = split_entity_id(cid)
            return TEST_INDEXES["S3"].get_attrs(str(suffix)) is not None

        cand_issues, cand_rows = load_id_list_file(candidate_path, "source1_entity_id", "candidate_entity_ids")
        match_issues, match_rows = load_id_list_file(matching_path, "source1_entity_id", "matched_entity_ids")

        return cand_issues + match_issues, cand_rows, match_rows

    test_s1_id_set = {row["entity_id"] for row in test_s1_rows}
    issues, cand_rows, match_rows = self_check(
        MATCHING_RESULTS_PATH, CANDIDATE_PAIRS_PATH, test_s1_id_set,
        lambda cid: True, lambda cid: True,  # replaced below with real checks
    )
    # (Re-run with real index-backed validity checks -- kept as a second pass
    # so a slow point-lookup pass only happens if the cheap structural checks
    # already look fine.)
    def valid_s2_real(cid):
        _, suffix = split_entity_id(cid)
        return TEST_INDEXES["S2"].get_attrs(str(suffix)) is not None
    def valid_s3_real(cid):
        _, suffix = split_entity_id(cid)
        return TEST_INDEXES["S3"].get_attrs(str(suffix)) is not None

    issues, cand_rows, match_rows = self_check(
        MATCHING_RESULTS_PATH, CANDIDATE_PAIRS_PATH, test_s1_id_set,
        valid_s2_real, valid_s3_real,
    )

    print(f"candidate_pairs.tsv: {cand_rows:,} rows | matching_results.tsv: {match_rows:,} rows "
          f"| expected: {len(test_s1_id_set):,} test S1 rows")

    if issues:
        print("ISSUES FOUND:")
        for i in issues:
            print(" -", i)
    else:
        print("PASS -- no issues found by the self-contained checker.")


## 15. (Best-effort) package the final submission zip
Priority is the two output files above -- this cell packages them plus the
pipeline code and a filled methodology template if one is found, but a
failure here should never be mistaken for a failure of the actual outputs.

In [ ]:
import zipfile

def build_submission_zip():
    zip_path = os.path.join(WORK_DIR, f"{TEAM_NAME}_submission.zip")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(CANDIDATE_PAIRS_PATH, "output/candidate_pairs.tsv")
        zf.write(MATCHING_RESULTS_PATH, "output/matching_results.tsv")

        for fname in os.listdir(MODULES_DIR):
            zf.write(os.path.join(MODULES_DIR, fname),
                      f"code/business_entity_resolution/src/{fname}")

        readme = (
            "# Business Entity Resolution -- reproduction steps\n\n"
            "1. Run FINAL_CLEAN.ipynb on Kaggle with the train/test datasets attached.\n"
            "2. The notebook auto-discovers train_source1/2/3.tsv, train_ground_truth.tsv, "
            "and test_source1/2/3.tsv anywhere under /kaggle/input.\n"
            "3. Pipeline: blocking/candidate generation (src/blocking.py, src/candidates.py, "
            "src/indexer.py) -> feature extraction (src/features.py, 26 features) -> "
            "LightGBM matcher -> threshold -> chunked/streaming test inference "
            "(src/checkpoint.py shard manifest) -> output/matching_results.tsv + "
            "output/candidate_pairs.tsv.\n"
            f"4. Final threshold used: {final_threshold}. Validation Macro F0.5: {final_f05:.6f}.\n"
        )
        zf.writestr("code/business_entity_resolution/README.md", readme)

        requirements = "pandas\nnumpy\nscikit-learn\nlightgbm\nrapidfuzz\npsutil\njoblib\n"
        zf.writestr("code/business_entity_resolution/requirements.txt", requirements)

        doc_template = None
        for pattern in ("/kaggle/input/**/Documentation_template.md", "/mnt/user-data/uploads/**/Documentation_template.md"):
            matches = glob.glob(pattern, recursive=True)
            if matches:
                doc_template = matches[0]
                break
        if doc_template:
            zf.write(doc_template, "Documentation_template.md")
        else:
            print("Documentation_template.md not found under /kaggle/input -- "
                  "fill it in by hand before your final zip submission; "
                  "output/ and code/ are already correct in this zip.")

    return zip_path

try:
    zip_path = build_submission_zip()
    print("Packaged (best-effort):", zip_path, f"({os.path.getsize(zip_path)/1024**2:.1f} MB)")
except Exception as e:
    print("Packaging step failed (non-fatal -- your two output files above are still valid):", repr(e))


## 16. Run summary

In [ ]:
print("=" * 70)
print("RUN SUMMARY")
print("=" * 70)
print(f"FAST_MODE                 : {FAST_MODE}")
print(f"Train S1 sample           : {TRAIN_S1_SAMPLE:,}")
print(f"Val S1 sample             : {VAL_S1_SAMPLE:,}")
print(f"Baseline threshold        : {BASELINE_THRESHOLD}  -> Macro F0.5 = {baseline_f05:.6f}")
print(f"Final threshold used      : {final_threshold}  -> Macro F0.5 = {final_f05:.6f}")
print(f"Test S1 rows scored       : {TOTAL_TEST_S1:,}")
print(f"S1_CHUNK_SIZE             : {S1_CHUNK_SIZE:,}  ({n_shards} shards)")
print(f"Output: candidate_pairs   : {CANDIDATE_PAIRS_PATH}")
print(f"Output: matching_results  : {MATCHING_RESULTS_PATH}")
print("=" * 70)
print("NEXT STEPS:")
print(" 1. Upload matching_results.tsv to the leaderboard Portal.")
print(" 2. Once you have a public-leaderboard score, use your remaining submission")
print("    attempts to tune: FREQ_CAPS_OVERRIDE, TRAIN_S1_SAMPLE/VAL_S1_SAMPLE (bigger),")
print("    THRESHOLD_SEARCH_GRID, or FAST_MODE=False for a fuller run.")
print(" 3. The team_submission.zip (if packaged above) still needs your filled-in")
print("    Documentation_template.md if it wasn't auto-found.")
